In [3]:
import os, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import scipy.io
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ    = 64
BATCH  = 128
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

Device: cpu


In [4]:
CMAPS_DIR = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps'
COLS = list(range(26))

def load_cmapss(fd):
    """Load C-MAPSS FDxxx. Returns X (N,SEQ,14), y_rul (N,), y_state (N,) in raw cycles."""
    train_df = pd.read_csv(f'{CMAPS_DIR}/train_FD{fd:03d}.txt', sep=r'\s+', header=None, names=range(26))
    test_df  = pd.read_csv(f'{CMAPS_DIR}/test_FD{fd:03d}.txt',  sep=r'\s+', header=None, names=range(26))
    rul_df   = pd.read_csv(f'{CMAPS_DIR}/RUL_FD{fd:03d}.txt',   header=None, names=['RUL'])

    # Select 14 informative sensors (standard in literature)
    sensors = [6,7,8,11,12,13,15,16,17,18,19,21,24,25]

    def add_rul(df):
        df = df.copy()
        max_cycles = df.groupby(0)[1].max().rename('max_cycle')
        df = df.join(max_cycles, on=0)
        df['RUL'] = (df['max_cycle'] - df[1]).clip(upper=125)
        df['state'] = pd.cut(df['RUL'], bins=[-1,25,50,200], labels=[0,1,2]).astype(int)
        return df

    train_df = add_rul(train_df)

    # Normalize features using train stats
    feat_mean = train_df[sensors].mean()
    feat_std  = train_df[sensors].std().replace(0, 1)

    def make_windows(df, rul_arr=None):
        engines = df[0].unique()
        Xs, yr, ys = [], [], []
        for i, eng in enumerate(engines):
            sub = df[df[0]==eng].reset_index(drop=True)
            feats = ((sub[sensors] - feat_mean) / feat_std).values.astype(np.float32)
            n = len(feats)
            if n < SEQ:
                pad = np.zeros((SEQ - n, 14), dtype=np.float32)
                feats = np.vstack([pad, feats])
                n = SEQ
            for t in range(n - SEQ + 1):
                window = feats[t:t+SEQ]
                if rul_arr is not None:
                    # test: one window per engine, use provided RUL
                    break
                else:
                    rul_val = sub.loc[t + SEQ - 1, 'RUL']
                    state_val = sub.loc[t + SEQ - 1, 'state']
                Xs.append(window)
                yr.append(float(rul_val))
                ys.append(int(state_val))
            if rul_arr is not None:
                # last window of test engine + provided RUL
                window = feats[n-SEQ:n]
                Xs.append(window)
                yr.append(float(rul_arr[i]))
                ys.append(int(min(2, max(0, 2 - (rul_arr[i]>50) - (rul_arr[i]>25)))))
        return (torch.tensor(np.array(Xs)),
                torch.tensor(yr, dtype=torch.float32),
                torch.tensor(ys, dtype=torch.long))

    Xtr, ytr_r, ytr_s = make_windows(train_df)

    # Test set
    test_df = test_df.copy()
    test_feats = ((test_df[sensors] - feat_mean) / feat_std)
    test_df_norm = test_df.copy()
    test_df_norm[sensors] = test_feats
    test_df_norm[0] = test_df[0]
    test_df_norm[1] = test_df[1]
    rul_test = rul_df['RUL'].values.astype(np.float32)
    Xte, yte_r, yte_s = make_windows(test_df_norm, rul_arr=rul_test)

    # Train/val split (80/20 by engine)
    n_engines = train_df[0].nunique()
    val_engines = set(train_df[0].unique()[int(n_engines*0.8):])
    val_mask = train_df.apply(lambda r: r[0] in val_engines, axis=1).values

    # Rebuild split properly
    def make_windows_split(df, split):
        engines = df[0].unique()
        n_val_start = int(len(engines) * 0.8)
        sel_engines = engines[n_val_start:] if split == 'val' else engines[:n_val_start]
        Xs, yr, ys = [], [], []
        for eng in sel_engines:
            sub = df[df[0]==eng].reset_index(drop=True)
            feats = ((sub[sensors] - feat_mean) / feat_std).values.astype(np.float32)
            n = len(feats)
            if n < SEQ:
                pad = np.zeros((SEQ - n, 14), dtype=np.float32)
                feats = np.vstack([pad, feats])
                n = SEQ
            for t in range(n - SEQ + 1):
                Xs.append(feats[t:t+SEQ])
                yr.append(float(sub.loc[t + SEQ - 1, 'RUL']))
                ys.append(int(sub.loc[t + SEQ - 1, 'state']))
        return (torch.tensor(np.array(Xs)),
                torch.tensor(yr, dtype=torch.float32),
                torch.tensor(ys, dtype=torch.long))

    Xtr2, ytr2_r, ytr2_s = make_windows_split(train_df, 'train')
    Xvl, yvl_r, yvl_s    = make_windows_split(train_df, 'val')

    def make_loader(X, yr, ys, shuffle=False):
        return DataLoader(TensorDataset(X, yr, ys), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    tr = make_loader(Xtr2, ytr2_r, ytr2_s, shuffle=True)
    va = make_loader(Xvl,  yvl_r,  yvl_s)
    te = make_loader(Xte,  yte_r,  yte_s)
    print(f"FD{fd:03d}  tr:{len(Xtr2):>6}  val:{len(Xvl):>5}  te:{len(Xte):>4}")
    return tr, va, te

cmapss = {}
for fd in [1, 2, 3, 4]:
    cmapss[fd] = load_cmapss(fd)

FD001  tr: 11098  val: 3233  te: 100
FD002  tr: 29999  val: 7380  te: 259
FD003  tr: 14739  val: 3681  te: 100
FD004  tr: 36535  val: 9027  te: 248


In [7]:
BAT_DIR  = '/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset'
BAT_META = BAT_DIR + '/metadata.csv'

def load_battery():
    meta = pd.read_csv(BAT_META)
    # Debug: see actual columns
    print("Meta columns:", meta.columns.tolist())
    print("Types:", meta['type'].unique() if 'type' in meta.columns else meta.dtypes)
    
    # Filter discharge cycles — handle both 'type' and 'Type' column names
    type_col = 'type' if 'type' in meta.columns else 'Type'
    discharge = meta[meta[type_col].str.lower() == 'discharge'].copy()
    
    # Find battery ID column
    id_col = [c for c in meta.columns if 'battery' in c.lower() or 'id' in c.lower()][0]
    batteries = discharge[id_col].unique()
    
    # Find sort column (cycle number / index)
    sort_col = None
    for candidate in ['cycle_count', 'cycle', 'Cycle', 'cycle_id', 'cycle_number',
                       'start_time', 'date', 'ambient_temperature']:
        if candidate in discharge.columns:
            sort_col = candidate
            break
    print(f"Using id_col='{id_col}', sort_col='{sort_col}'")

    all_seqs = []
    for bid in batteries:
        rows = discharge[discharge[id_col] == bid]
        if sort_col:
            rows = rows.sort_values(sort_col)
        rows = rows.reset_index(drop=True)

        cap_vals = []
        for _, r in rows.iterrows():
            fname = r['filename'] if 'filename' in r.index else r['file_name']
            fpath = BAT_DIR + '/data/' + str(fname)
            if not os.path.exists(fpath):
                continue
            df = pd.read_csv(fpath)
            # Find capacity column (case-insensitive)
            cap_col = [c for c in df.columns if 'capacity' in c.lower() or 'cap' in c.lower()]
            if not cap_col:
                # Try last column as proxy
                cap = df.iloc[-1, -1]
            else:
                cap = df[cap_col[0]].dropna().iloc[-1]
            try:
                cap_vals.append(float(cap))
            except:
                continue

        n = len(cap_vals)
        if n < SEQ + 5:
            continue

        cap_arr = np.array(cap_vals, dtype=np.float32)
        max_cap = cap_arr[0] if cap_arr[0] > 0 else cap_arr.max()
        soh     = np.clip(cap_arr / max_cap, 0.0, 1.0)

        delta      = np.diff(soh, prepend=soh[0])
        cycle_norm = np.arange(n, dtype=np.float32) / n
        feats      = np.stack([soh, delta, cycle_norm, cap_arr / (max_cap + 1e-8)], axis=1)

        # RUL = cycles to EOL (SoH < 0.8), clipped at 200
        eol_indices = np.where(soh < 0.8)[0]
        eol_idx     = int(eol_indices[0]) if len(eol_indices) > 0 else n
        rul         = np.clip((eol_idx - np.arange(n)).astype(np.float32), 0, 200)
        state       = (rul <= 50).astype(np.int64) + (rul <= 20).astype(np.int64)

        for t in range(n - SEQ + 1):
            all_seqs.append((feats[t:t+SEQ], rul[t+SEQ-1], state[t+SEQ-1]))

    if len(all_seqs) == 0:
        raise RuntimeError("No battery sequences loaded — check metadata and data paths.")

    np.random.shuffle(all_seqs)
    n_total = len(all_seqs)
    n_tr    = int(n_total * 0.6)
    n_vl    = int(n_total * 0.2)

    def to_loader(seqs, shuffle=False):
        X  = torch.tensor(np.array([s[0] for s in seqs]))
        yr = torch.tensor([s[1] for s in seqs], dtype=torch.float32)
        ys = torch.tensor([s[2] for s in seqs], dtype=torch.long)
        return DataLoader(TensorDataset(X, yr, ys), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    tr = to_loader(all_seqs[:n_tr],          shuffle=True)
    va = to_loader(all_seqs[n_tr:n_tr+n_vl])
    te = to_loader(all_seqs[n_tr+n_vl:])
    n_feat = 4
    print(f"Battery  tr:{n_tr:>6}  val:{n_vl:>5}  te:{n_total-n_tr-n_vl:>4}  seqs_total:{n_total}  feat:{n_feat}")
    return tr, va, te, n_feat

bat_tr, bat_va, bat_te, BAT_FEAT = load_battery()

Meta columns: ['type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct']
Types: ['discharge' 'impedance' 'charge']
Using id_col='battery_id', sort_col='start_time'
Battery  tr:   652  val:  217  te: 218  seqs_total:1087  feat:4


In [8]:
import os, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import scipy.io
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ    = 64
BATCH  = 128
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

Device: cpu


In [9]:
CMAPS_DIR = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps'
SENSORS   = [6,7,8,11,12,13,15,16,17,18,19,21,24,25]  # 14 informative sensors

def load_cmapss(fd):
    train_df = pd.read_csv(f'{CMAPS_DIR}/train_FD{fd:03d}.txt', sep=r'\s+', header=None)
    test_df  = pd.read_csv(f'{CMAPS_DIR}/test_FD{fd:03d}.txt',  sep=r'\s+', header=None)
    rul_df   = pd.read_csv(f'{CMAPS_DIR}/RUL_FD{fd:03d}.txt',   header=None, names=['RUL'])

    # Add piecewise-linear RUL (clipped at 125)
    max_cycles = train_df.groupby(0)[1].max().rename('max_cycle')
    train_df   = train_df.join(max_cycles, on=0)
    train_df['RUL']   = (train_df['max_cycle'] - train_df[1]).clip(upper=125)
    train_df['state'] = pd.cut(train_df['RUL'], bins=[-1,25,50,200],
                                labels=[0,1,2]).astype(int)

    feat_mean = train_df[SENSORS].mean()
    feat_std  = train_df[SENSORS].std().replace(0, 1)

    def make_windows_for_engines(df, engine_list):
        Xs, yr, ys = [], [], []
        for eng in engine_list:
            sub   = df[df[0]==eng].reset_index(drop=True)
            feats = ((sub[SENSORS] - feat_mean) / feat_std).values.astype(np.float32)
            n     = len(feats)
            if n < SEQ:
                pad   = np.zeros((SEQ - n, 14), dtype=np.float32)
                feats = np.vstack([pad, feats])
                n     = SEQ
            for t in range(n - SEQ + 1):
                Xs.append(feats[t:t+SEQ])
                yr.append(float(sub.loc[min(t+SEQ-1, len(sub)-1), 'RUL']))
                ys.append(int(sub.loc[min(t+SEQ-1, len(sub)-1), 'state']))
        return (torch.tensor(np.array(Xs)),
                torch.tensor(yr, dtype=torch.float32),
                torch.tensor(ys, dtype=torch.long))

    engines   = train_df[0].unique()
    n_tr_eng  = int(len(engines) * 0.8)
    tr_eng    = engines[:n_tr_eng]
    vl_eng    = engines[n_tr_eng:]

    Xtr, ytr_r, ytr_s = make_windows_for_engines(train_df, tr_eng)
    Xvl, yvl_r, yvl_s = make_windows_for_engines(train_df, vl_eng)

    # Test: one window per engine using last SEQ cycles + provided RUL
    test_engines = test_df[0].unique()
    rul_test     = rul_df['RUL'].values.astype(np.float32)
    Xte_list, yte_r_list, yte_s_list = [], [], []
    for i, eng in enumerate(test_engines):
        sub   = test_df[test_df[0]==eng].reset_index(drop=True)
        feats = ((sub[SENSORS] - feat_mean) / feat_std).values.astype(np.float32)
        n     = len(feats)
        if n < SEQ:
            pad   = np.zeros((SEQ - n, 14), dtype=np.float32)
            feats = np.vstack([pad, feats])
        Xte_list.append(feats[-SEQ:])
        rul_val = rul_test[i]
        yte_r_list.append(float(rul_val))
        state   = 0 if rul_val > 50 else (1 if rul_val > 25 else 2)
        yte_s_list.append(state)

    Xte  = torch.tensor(np.array(Xte_list))
    yte_r = torch.tensor(yte_r_list, dtype=torch.float32)
    yte_s = torch.tensor(yte_s_list, dtype=torch.long)

    def make_loader(X, yr, ys, shuffle=False):
        return DataLoader(TensorDataset(X, yr, ys), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    tr = make_loader(Xtr, ytr_r, ytr_s, shuffle=True)
    va = make_loader(Xvl, yvl_r, yvl_s)
    te = make_loader(Xte, yte_r,  yte_s)
    print(f"FD{fd:03d}  tr:{len(Xtr):>6}  val:{len(Xvl):>5}  te:{len(Xte):>4}")
    return tr, va, te

cmapss = {}
for fd in [1,2,3,4]:
    cmapss[fd] = load_cmapss(fd)

FD001  tr: 11098  val: 3233  te: 100
FD002  tr: 29999  val: 7380  te: 259
FD003  tr: 14739  val: 3681  te: 100
FD004  tr: 36535  val: 9027  te: 248


In [10]:
BAT_DIR  = '/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset'
BAT_META = BAT_DIR + '/metadata.csv'

def load_battery():
    meta      = pd.read_csv(BAT_META)
    discharge = meta[meta['type'] == 'discharge'].copy()
    batteries = discharge['battery_id'].unique()

    all_seqs = []
    for bid in batteries:
        rows     = discharge[discharge['battery_id'] == bid].sort_values('start_time').reset_index(drop=True)
        cap_vals = []
        for _, r in rows.iterrows():
            fpath = BAT_DIR + '/data/' + str(r['filename'])
            if not os.path.exists(fpath):
                continue
            # Use Capacity column directly from metadata row (pre-computed)
            cap = r['Capacity']
            try:
                cap_vals.append(float(cap))
            except:
                continue

        n = len(cap_vals)
        if n < SEQ + 5:
            continue

        cap_arr  = np.array(cap_vals, dtype=np.float32)
        max_cap  = cap_arr[0] if cap_arr[0] > 0 else cap_arr.max()
        soh      = np.clip(cap_arr / max_cap, 0.0, 1.0)
        delta    = np.diff(soh, prepend=soh[0])
        cyc_norm = np.arange(n, dtype=np.float32) / n
        feats    = np.stack([soh, delta, cyc_norm, cap_arr / (max_cap + 1e-8)], axis=1)

        eol_idx  = int(np.where(soh < 0.8)[0][0]) if np.any(soh < 0.8) else n
        rul      = np.clip((eol_idx - np.arange(n)).astype(np.float32), 0, 200)
        state    = (rul <= 50).astype(np.int64) + (rul <= 20).astype(np.int64)

        for t in range(n - SEQ + 1):
            all_seqs.append((feats[t:t+SEQ], rul[t+SEQ-1], state[t+SEQ-1]))

    np.random.shuffle(all_seqs)
    n_total = len(all_seqs)
    n_tr    = int(n_total * 0.6)
    n_vl    = int(n_total * 0.2)

    def to_loader(seqs, shuffle=False):
        X  = torch.tensor(np.array([s[0] for s in seqs]))
        yr = torch.tensor([s[1] for s in seqs], dtype=torch.float32)
        ys = torch.tensor([s[2] for s in seqs], dtype=torch.long)
        return DataLoader(TensorDataset(X, yr, ys), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    tr = to_loader(all_seqs[:n_tr], shuffle=True)
    va = to_loader(all_seqs[n_tr:n_tr+n_vl])
    te = to_loader(all_seqs[n_tr+n_vl:])
    print(f"Battery  tr:{n_tr:>6}  val:{n_vl:>5}  te:{n_total-n_tr-n_vl:>4}  total:{n_total}")
    return tr, va, te, 4

bat_tr, bat_va, bat_te, BAT_FEAT = load_battery()

Battery  tr:   652  val:  217  te: 218  total:1087


In [ ]:
CWRU_DIR      = '/kaggle/input/datasets/sufian79/cwru-mat-full-dataset'
N_CWRU_CLASSES = 10
CWRU_SEQ      = 1024

CWRU_FILES = {
    '97.mat':0,'98.mat':0,'99.mat':0,'100.mat':0,
    '105.mat':1,'106.mat':1,'107.mat':1,'108.mat':1,
    '169.mat':2,'170.mat':2,'171.mat':2,'172.mat':2,
    '209.mat':3,'210.mat':3,'211.mat':3,'212.mat':3,
    '118.mat':4,'119.mat':4,'120.mat':4,'121.mat':4,
    '185.mat':5,'186.mat':5,'187.mat':5,'188.mat':5,
    '222.mat':6,'223.mat':6,'224.mat':6,'225.mat':6,
    '130.mat':7,'131.mat':7,'132.mat':7,'133.mat':7,
    '197.mat':8,'198.mat':8,'199.mat':8,'200.mat':8,
    '234.mat':9,'235.mat':9,'236.mat':9,'237.mat':9,
}

def load_cwru():
    all_X, all_y = [], []
    for fname, cls_id in CWRU_FILES.items():
        fpath = os.path.join(CWRU_DIR, fname)
        if not os.path.exists(fpath):
            continue
        mat = scipy.io.loadmat(fpath)
        key = [k for k in mat.keys() if 'DE_time' in k]
        if not key:
            continue
        sig  = mat[key[0]].flatten().astype(np.float32)
        sig  = (sig - sig.mean()) / (sig.std() + 1e-8)
        step = CWRU_SEQ // 2
        for start in range(0, len(sig) - CWRU_SEQ + 1, step):
            all_X.append(sig[start:start+CWRU_SEQ].reshape(CWRU_SEQ, 1))
            all_y.append(cls_id)

    idx   = np.random.RandomState(SEED).permutation(len(all_X))
    all_X = np.array(all_X)[idx]
    all_y = np.array(all_y)[idx]

    n_total = len(all_X)
    n_tr    = int(n_total * 0.6)
    n_vl    = int(n_total * 0.2)

    def to_loader(X, y, shuffle=False):
        Xt = torch.tensor(X)
        yt = torch.tensor(y, dtype=torch.long)
        yr = torch.zeros(len(y), dtype=torch.float32)
        return DataLoader(TensorDataset(Xt, yr, yt), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    tr = to_loader(all_X[:n_tr],          all_y[:n_tr],          shuffle=True)
    va = to_loader(all_X[n_tr:n_tr+n_vl], all_y[n_tr:n_tr+n_vl])
    te = to_loader(all_X[n_tr+n_vl:],     all_y[n_tr+n_vl:])
    print(f"CWRU     tr:{n_tr:>6}  val:{n_vl:>5}  te:{n_total-n_tr-n_vl:>4}  classes:{N_CWRU_CLASSES}")
    return tr, va, te

cwru_tr, cwru_va, cwru_te = load_cwru()

In [11]:
CWRU_DIR      = '/kaggle/input/datasets/sufian79/cwru-mat-full-dataset'
N_CWRU_CLASSES = 10
CWRU_SEQ      = 1024

CWRU_FILES = {
    '97.mat':0,'98.mat':0,'99.mat':0,'100.mat':0,
    '105.mat':1,'106.mat':1,'107.mat':1,'108.mat':1,
    '169.mat':2,'170.mat':2,'171.mat':2,'172.mat':2,
    '209.mat':3,'210.mat':3,'211.mat':3,'212.mat':3,
    '118.mat':4,'119.mat':4,'120.mat':4,'121.mat':4,
    '185.mat':5,'186.mat':5,'187.mat':5,'188.mat':5,
    '222.mat':6,'223.mat':6,'224.mat':6,'225.mat':6,
    '130.mat':7,'131.mat':7,'132.mat':7,'133.mat':7,
    '197.mat':8,'198.mat':8,'199.mat':8,'200.mat':8,
    '234.mat':9,'235.mat':9,'236.mat':9,'237.mat':9,
}

def load_cwru():
    all_X, all_y = [], []
    for fname, cls_id in CWRU_FILES.items():
        fpath = os.path.join(CWRU_DIR, fname)
        if not os.path.exists(fpath):
            continue
        mat = scipy.io.loadmat(fpath)
        key = [k for k in mat.keys() if 'DE_time' in k]
        if not key:
            continue
        sig  = mat[key[0]].flatten().astype(np.float32)
        sig  = (sig - sig.mean()) / (sig.std() + 1e-8)
        step = CWRU_SEQ // 2
        for start in range(0, len(sig) - CWRU_SEQ + 1, step):
            all_X.append(sig[start:start+CWRU_SEQ].reshape(CWRU_SEQ, 1))
            all_y.append(cls_id)

    idx   = np.random.RandomState(SEED).permutation(len(all_X))
    all_X = np.array(all_X)[idx]
    all_y = np.array(all_y)[idx]

    n_total = len(all_X)
    n_tr    = int(n_total * 0.6)
    n_vl    = int(n_total * 0.2)

    def to_loader(X, y, shuffle=False):
        Xt = torch.tensor(X)
        yt = torch.tensor(y, dtype=torch.long)
        yr = torch.zeros(len(y), dtype=torch.float32)
        return DataLoader(TensorDataset(Xt, yr, yt), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    tr = to_loader(all_X[:n_tr],          all_y[:n_tr],          shuffle=True)
    va = to_loader(all_X[n_tr:n_tr+n_vl], all_y[n_tr:n_tr+n_vl])
    te = to_loader(all_X[n_tr+n_vl:],     all_y[n_tr+n_vl:])
    print(f"CWRU     tr:{n_tr:>6}  val:{n_vl:>5}  te:{n_total-n_tr-n_vl:>4}  classes:{N_CWRU_CLASSES}")
    return tr, va, te

cwru_tr, cwru_va, cwru_te = load_cwru()

CWRU     tr:  7099  val: 2366  te:2367  classes:10


In [12]:
class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()
        self.conv = nn.Conv1d(d, d, kernel_size=3, padding=dilation,
                              dilation=dilation, padding_mode='zeros')
        self.bn   = nn.BatchNorm1d(d)
        self.act  = nn.GELU()
    def forward(self, x):
        return x + self.act(self.bn(self.conv(x)))

class CrossSensorGate(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        g = self.gate(x.mean(dim=1, keepdim=True))
        return x * g

class TinyPrognostics(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3):
        super().__init__()
        self.embed = nn.Linear(n_sensors, d)
        self.tcn   = nn.Sequential(
            DilatedBlock(d, 1),
            DilatedBlock(d, 2),
            DilatedBlock(d, 4),
        )
        self.gate  = CrossSensorGate(d)
        self.skip  = nn.Linear(n_sensors, d)
        self.gru   = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head   = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32,1))
        self.state_head = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32,n_classes))

    def forward(self, x):
        h    = self.embed(x).permute(0,2,1)
        h    = self.tcn(h).permute(0,2,1)
        h    = self.gate(h) + self.skip(x)
        _, hT = self.gru(h)
        hT   = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

m      = TinyPrognostics()
params = sum(p.numel() for p in m.parameters())
kb     = sum(p.numel()*p.element_size() for p in m.parameters()) / 1024
print(f"TinyPrognostics: {params:,} params  {kb:.1f} KB")

TinyPrognostics: 12,052 params  47.1 KB


In [16]:
def nasa_score(pred, true):
    d = pred - true
    s = np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)
    total = s.sum()
    return float(total) if np.isfinite(total) else float('nan')

def train_epoch(model, loader, optimizer, is_cwru=False):
    model.train()
    mse_fn = nn.MSELoss()
    ce_fn  = nn.CrossEntropyLoss()
    for X, yr, ys in loader:
        X, yr, ys = X.to(DEVICE), yr.to(DEVICE), ys.to(DEVICE)
        optimizer.zero_grad()
        pred_r, pred_s = model(X)
        loss = ce_fn(pred_s, ys) if is_cwru else \
               mse_fn(pred_r, yr) + 0.15 * ce_fn(pred_s, ys)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

def evaluate(model, loader, is_cwru=False):
    model.eval()
    preds_r, trues_r, preds_s, trues_s = [], [], [], []
    with torch.no_grad():
        for X, yr, ys in loader:
            X = X.to(DEVICE)
            pred_r, pred_s = model(X)
            preds_r.extend(pred_r.cpu().numpy())
            trues_r.extend(yr.numpy())
            preds_s.extend(pred_s.argmax(1).cpu().numpy())
            trues_s.extend(ys.numpy())
    pr = np.array(preds_r); tr = np.array(trues_r)
    ps = np.array(preds_s); ts = np.array(trues_s)
    rmse  = float(np.sqrt(np.mean((pr - tr)**2)))
    mae   = float(np.mean(np.abs(pr - tr)))
    acc   = float((ps == ts).mean() * 100)
    score = nasa_score(pr, tr)
    return rmse, mae, acc, score

def train_model(model, tr, va, te,
                epochs=100, lr=1e-3, is_cwru=False,
                patience=20, min_lr=1e-5):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=min_lr)
    best_val, best_state, wait = float('inf'), None, 0

    for epoch in range(1, epochs+1):
        train_epoch(model, tr, optimizer, is_cwru=is_cwru)
        val_rmse, _, val_acc, _ = evaluate(model, va, is_cwru=is_cwru)
        val_metric = -val_acc if is_cwru else val_rmse
        scheduler.step()
        if val_metric < best_val:
            best_val  = val_metric
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return evaluate(model, te, is_cwru=is_cwru)

In [17]:
results = {}
print(f"{'Dataset':10} {'RMSE':>8} {'MAE':>8} {'Acc%':>8} {'NASAScore':>12}")
print("-"*52)

for fd in [1,2,3,4]:
    torch.manual_seed(SEED)
    model = TinyPrognostics(n_sensors=14, d=24, n_classes=3).to(DEVICE)
    tr, va, te = cmapss[fd]
    rmse, mae, acc, score = train_model(model, tr, va, te, epochs=100, patience=20)
    results[f'FD{fd:03d}'] = dict(rmse=rmse, mae=mae, acc=acc, score=score)
    torch.save(model.state_dict(), f'/kaggle/working/tiny_fd{fd:03d}.pt')
    print(f"FD{fd:03d}      {rmse:8.4f} {mae:8.4f} {acc:8.2f} {score:12.2f}")

Dataset        RMSE      MAE     Acc%    NASAScore
----------------------------------------------------
FD001       16.5145  12.5240     8.00       485.99
FD002       26.9720  19.0001     6.18      7069.07
FD003       13.7683   9.7468     9.00       440.40
FD004       29.3507  21.1074     6.45      9037.30


In [18]:
torch.manual_seed(SEED)
bat_model = TinyPrognostics(n_sensors=BAT_FEAT, d=24, n_classes=3).to(DEVICE)
rmse, mae, acc, score = train_model(bat_model, bat_tr, bat_va, bat_te, epochs=100, patience=20)
results['battery'] = dict(rmse=rmse, mae=mae, acc=acc, score=score)
torch.save(bat_model.state_dict(), '/kaggle/working/tiny_battery.pt')
print(f"Battery   {rmse:8.4f} {mae:8.4f} {acc:8.2f} {score:12.2f}")

Battery     1.9995   0.4131    98.17        11.18


In [19]:
torch.manual_seed(SEED)
cwru_model = TinyPrognostics(n_sensors=1, d=24, n_classes=N_CWRU_CLASSES).to(DEVICE)
rmse, mae, acc, score = train_model(cwru_model, cwru_tr, cwru_va, cwru_te,
                                     epochs=80, is_cwru=True, patience=15)
results['cwru'] = dict(rmse=rmse, mae=mae, acc=acc, score=score)
torch.save(cwru_model.state_dict(), '/kaggle/working/tiny_cwru.pt')
print(f"CWRU Acc: {acc:.2f}%  ({N_CWRU_CLASSES}-class fault classification)")

CWRU Acc: 99.79%  (10-class fault classification)


In [20]:
# Ridge
print("RIDGE BASELINE")
for fd in [1,2,3,4]:
    tr, va, te = cmapss[fd]
    Xtr_np = torch.cat([b[0] for b in tr]).numpy().reshape(-1, SEQ*14)
    ytr_np = torch.cat([b[1] for b in tr]).numpy()
    Xte_np = torch.cat([b[0] for b in te]).numpy().reshape(-1, SEQ*14)
    yte_np = torch.cat([b[1] for b in te]).numpy()
    ridge  = Ridge(alpha=1.0).fit(Xtr_np, ytr_np)
    rmse   = float(np.sqrt(mean_squared_error(yte_np, ridge.predict(Xte_np))))
    results[f'FD{fd:03d}']['ridge_rmse'] = rmse
    print(f"  FD{fd:03d} Ridge RMSE: {rmse:.4f}")

# LSTM & CNN baselines
class LSTMBaseline(nn.Module):
    def __init__(self, n_sensors=14, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, hidden, 2, batch_first=True, dropout=0.2)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(h[-1]).squeeze(-1), torch.zeros(x.size(0),3).to(x.device)

class CNNBaseline(nn.Module):
    def __init__(self, n_sensors=14, filters=32):
        super().__init__()
        self.net  = nn.Sequential(
            nn.Conv1d(n_sensors, filters,   3, padding=1), nn.ReLU(), nn.BatchNorm1d(filters),
            nn.Conv1d(filters,   filters*2, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(filters*2),
            nn.AdaptiveAvgPool1d(1))
        self.head = nn.Linear(filters*2, 1)
    def forward(self, x):
        h = self.net(x.permute(0,2,1)).squeeze(-1)
        return self.head(h).squeeze(-1), torch.zeros(x.size(0),3).to(x.device)

print("\nNEURAL BASELINES")
print(f"{'Model':16} {'FD001':>8} {'FD002':>8} {'FD003':>8} {'FD004':>8}  {'Size':>8}")
for name, Cls, kw in [('LSTM-64', LSTMBaseline, {'hidden':64}),
                       ('CNN-32',  CNNBaseline,  {'filters':32})]:
    row = []
    for fd in [1,2,3,4]:
        torch.manual_seed(SEED)
        m = Cls(**kw).to(DEVICE)
        tr, va, te = cmapss[fd]
        rmse, mae, acc, score = train_model(m, tr, va, te, epochs=80, patience=15)
        results[f'FD{fd:03d}'][f'{name}_rmse'] = rmse
        row.append(rmse)
    kb = sum(p.numel()*p.element_size() for p in Cls(**kw).parameters())/1024
    print(f"{name:16} {row[0]:8.4f} {row[1]:8.4f} {row[2]:8.4f} {row[3]:8.4f}  {kb:6.1f}KB")

RIDGE BASELINE
  FD001 Ridge RMSE: 42.2707
  FD002 Ridge RMSE: 54.7347
  FD003 Ridge RMSE: 43.3707
  FD004 Ridge RMSE: 54.7602

NEURAL BASELINES
Model               FD001    FD002    FD003    FD004      Size
LSTM-64           16.5333  27.8831  16.8035  29.9208   210.3KB
CNN-32            27.4334  41.2675  26.3415  59.0341    30.6KB


In [21]:
class TinyAblation(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3,
                 no_gate=False, no_skip=False, no_dilation=False):
        super().__init__()
        self.no_gate  = no_gate
        self.no_skip  = no_skip
        self.embed    = nn.Linear(n_sensors, d)
        dilations     = [1,1,1] if no_dilation else [1,2,4]
        self.tcn      = nn.Sequential(*[DilatedBlock(d, dil) for dil in dilations])
        if not no_gate: self.gate = CrossSensorGate(d)
        if not no_skip: self.skip = nn.Linear(n_sensors, d)
        self.gru      = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head   = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Linear(32,1))
        self.state_head = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Linear(32,n_classes))

    def forward(self, x):
        h = self.embed(x).permute(0,2,1)
        h = self.tcn(h).permute(0,2,1)
        if not self.no_gate: h = self.gate(h)
        if not self.no_skip: h = h + self.skip(x)
        _, hT = self.gru(h)
        hT = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

ablation_configs = {
    'Full Model':  dict(no_gate=False, no_skip=False, no_dilation=False),
    'No Gate':     dict(no_gate=True,  no_skip=False, no_dilation=False),
    'No Skip':     dict(no_gate=False, no_skip=True,  no_dilation=False),
    'No Dilation': dict(no_gate=False, no_skip=False, no_dilation=True),
}

ablation_results = {}
print(f"{'Config':16} {'FD001':>8} {'FD003':>8}")
print("-"*36)
for name, cfg in ablation_configs.items():
    row = {}
    for fd in [1, 3]:
        torch.manual_seed(SEED)
        m = TinyAblation(n_sensors=14, **cfg).to(DEVICE)
        tr, va, te = cmapss[fd]
        rmse, *_ = train_model(m, tr, va, te, epochs=80, patience=15)
        row[fd] = rmse
    ablation_results[name] = row
    print(f"{name:16} {row[1]:8.4f} {row[3]:8.4f}")

Config              FD001    FD003
------------------------------------
Full Model        15.8561  14.3871
No Gate           17.7880  14.8802
No Skip           14.5267  14.7692
No Dilation       14.6343  14.5195


In [25]:
def transfer_finetune(source_pt, target_loaders, source_sensors=14, target_sensors=14,
                       data_frac=1.0, epochs=50, lr=5e-4, patience=12):
    tr, va, te = target_loaders

    # Load source model (14 sensors = FD001 architecture)
    source_model = TinyPrognostics(n_sensors=source_sensors, d=24, n_classes=3).to(DEVICE)
    source_model.load_state_dict(torch.load(source_pt, map_location=DEVICE))

    # Build target model with correct input size
    model = TinyPrognostics(n_sensors=target_sensors, d=24, n_classes=3).to(DEVICE)

    # Copy all weights EXCEPT embed and skip (sensor-dependent layers)
    src_sd = source_model.state_dict()
    tgt_sd = model.state_dict()
    for k in tgt_sd:
        if k in src_sd and src_sd[k].shape == tgt_sd[k].shape:
            tgt_sd[k] = src_sd[k].clone()
        # embed.weight, skip.weight stay randomly initialized if shapes differ
    model.load_state_dict(tgt_sd)

    # Freeze TCN + GRU (shared temporal layers), fine-tune embed + skip + heads
    for p in model.tcn.parameters(): p.requires_grad = False
    for p in model.gru.parameters(): p.requires_grad = False

    if data_frac < 1.0:
        full_ds = tr.dataset
        n_keep  = max(64, int(len(full_ds) * data_frac))
        idx     = torch.randperm(len(full_ds))[:n_keep]
        tr      = DataLoader(Subset(full_ds, idx), batch_size=BATCH,
                             shuffle=True, num_workers=2, pin_memory=True)

    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    best_val, best_state, wait = float('inf'), None, 0

    for epoch in range(1, epochs+1):
        train_epoch(model, tr, optimizer)
        val_rmse, *_ = evaluate(model, va)
        scheduler.step()
        if val_rmse < best_val:
            best_val   = val_rmse
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience: break

    model.load_state_dict(best_state)
    return evaluate(model, te)

fracs = [0.10, 0.25, 0.50, 1.00]
transfer_results = {}

# FD001 (14 sensors) → Battery (4 sensors)
scratch_bat = results['battery']['rmse']
transfer_results['Battery'] = {}
print(f"FD001 → Battery  (scratch RMSE={scratch_bat:.4f})")
print(f"{'Frac':>6}  {'FT RMSE':>10}  {'Δ vs Scratch':>14}")
for frac in fracs:
    rmse, *_ = transfer_finetune(
        '/kaggle/working/tiny_fd001.pt',
        (bat_tr, bat_va, bat_te),
        source_sensors=14,
        target_sensors=BAT_FEAT,
        data_frac=frac)
    transfer_results['Battery'][frac] = rmse
    print(f"{frac*100:5.0f}%  {rmse:10.4f}  {scratch_bat-rmse:+14.4f}")

# FD001 (14 sensors) → FD003 (14 sensors) — same architecture, direct load
scratch_fd3 = results['FD003']['rmse']
transfer_results['FD003'] = {}
print(f"\nFD001 → FD003  (scratch RMSE={scratch_fd3:.4f})")
print(f"{'Frac':>6}  {'FT RMSE':>10}  {'Δ vs Scratch':>14}")
for frac in fracs:
    rmse, *_ = transfer_finetune(
        '/kaggle/working/tiny_fd001.pt',
        cmapss[3],
        source_sensors=14,
        target_sensors=14,
        data_frac=frac)
    transfer_results['FD003'][frac] = rmse
    print(f"{frac*100:5.0f}%  {rmse:10.4f}  {scratch_fd3-rmse:+14.4f}")

FD001 → Battery  (scratch RMSE=1.9995)
  Frac     FT RMSE    Δ vs Scratch
   10%      4.2884         -2.2889
   25%      3.9164         -1.9169
   50%      3.9373         -1.9379
  100%      3.7996         -1.8001

FD001 → FD003  (scratch RMSE=13.7683)
  Frac     FT RMSE    Δ vs Scratch
   10%     16.3082         -2.5399
   25%     14.8410         -1.0727
   50%     13.8171         -0.0488
  100%     14.1706         -0.4023


In [27]:
print("\n" + "="*75)
print("TINYPROGNOSTICS v6 — MASTER RESULTS")
print("="*75)

tiny_kb = sum(p.numel()*p.element_size() for p in TinyPrognostics().parameters())/1024
lstm_kb = sum(p.numel()*p.element_size() for p in LSTMBaseline().parameters())/1024
cnn_kb  = sum(p.numel()*p.element_size() for p in CNNBaseline().parameters())/1024
print(f"Model sizes:  Tiny={tiny_kb:.1f}KB  LSTM={lstm_kb:.1f}KB  CNN={cnn_kb:.1f}KB\n")

print(f"{'Dataset':10} {'Tiny RMSE':>10} {'LSTM RMSE':>10} {'CNN RMSE':>10} {'Ridge RMSE':>11}")
print("-"*55)
for fd in [1,2,3,4]:
    k = f'FD{fd:03d}'
    r = results[k]
    print(f"{k:10} {r['rmse']:10.4f} "
          f"{r.get('LSTM-64_rmse', float('nan')):10.4f} "
          f"{r.get('CNN-32_rmse',  float('nan')):10.4f} "
          f"{r.get('ridge_rmse',   float('nan')):11.4f}")

r = results['battery']
print(f"{'Battery':10} {r['rmse']:10.4f}  {'—':>10}  {'—':>10}  {'—':>11}")
r = results['cwru']
print(f"{'CWRU':10}  {'Acc%':>10}  {r['acc']:10.2f}%")

print("\nABLATION (RMSE on FD001 and FD003)")
print(f"{'Config':16} {'FD001':>8} {'FD003':>8}  {'ΔRMSE FD001':>12}  {'ΔRMSE FD003':>12}")
base_fd1 = ablation_results['Full Model'][1]
base_fd3 = ablation_results['Full Model'][3]
for name, row in ablation_results.items():
    d1 = row[1] - base_fd1
    d3 = row[3] - base_fd3
    print(f"{name:16} {row[1]:8.4f} {row[3]:8.4f}  {d1:+12.4f}  {d3:+12.4f}")

print("\nNASA PROGNOSTIC SCORES (lower is better)")
for fd in [1,2,3,4]:
    k = f'FD{fd:03d}'
    print(f"  {k}: {results[k]['score']:.2f}")

print("\nTRANSFER LEARNING — FD001 pretrain → target")
print(f"{'Target':10} {'10%':>8} {'25%':>8} {'50%':>8} {'100%':>8}  {'Scratch':>8}")
for tgt, frac_dict in transfer_results.items():
    scratch = results['battery']['rmse'] if tgt=='Battery' else results['FD003']['rmse']
    vals = [frac_dict.get(f, float('nan')) for f in fracs]
    print(f"{tgt:10} {vals[0]:8.4f} {vals[1]:8.4f} {vals[2]:8.4f} {vals[3]:8.4f}  {scratch:8.4f}")

print("="*75)


TINYPROGNOSTICS v6 — MASTER RESULTS
Model sizes:  Tiny=47.1KB  LSTM=210.3KB  CNN=30.6KB

Dataset     Tiny RMSE  LSTM RMSE   CNN RMSE  Ridge RMSE
-------------------------------------------------------
FD001         16.5145    16.5333    27.4334     42.2707
FD002         26.9720    27.8831    41.2675     54.7347
FD003         13.7683    16.8035    26.3415     43.3707
FD004         29.3507    29.9208    59.0341     54.7602
Battery        1.9995           —           —            —
CWRU              Acc%       99.79%

ABLATION (RMSE on FD001 and FD003)
Config              FD001    FD003   ΔRMSE FD001   ΔRMSE FD003
Full Model        15.8561  14.3871       +0.0000       +0.0000
No Gate           17.7880  14.8802       +1.9320       +0.4931
No Skip           14.5267  14.7692       -1.3294       +0.3822
No Dilation       14.6343  14.5195       -1.2217       +0.1325

NASA PROGNOSTIC SCORES (lower is better)
  FD001: 485.99
  FD002: 7069.07
  FD003: 440.40
  FD004: 9037.30

TRANSFER LEARNING —

In [1]:
# ============================================================
# CELL 1 — Imports & Config
# ============================================================
import os, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import scipy.io
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ    = 64
BATCH  = 128
SEED   = 42
EPOCHS = 120        # unified for ALL experiments (main + ablation + baselines)
PATIENCE = 25       # unified patience
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

Device: cpu


In [2]:
# ============================================================
# CELL 2 — C-MAPSS Data Pipeline
# ============================================================
CMAPSS_DIR = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps'

# 14 standard informative sensors (Saxena et al.)
SENSORS = [6, 7, 8, 11, 12, 13, 15, 16, 17, 18, 19, 21, 24, 25]
RUL_CAP = 125

def load_cmapss(fd):
    train_df = pd.read_csv(f'{CMAPSS_DIR}/train_FD{fd:03d}.txt',
                           sep=r'\s+', header=None)
    test_df  = pd.read_csv(f'{CMAPSS_DIR}/test_FD{fd:03d}.txt',
                           sep=r'\s+', header=None)
    rul_df   = pd.read_csv(f'{CMAPSS_DIR}/RUL_FD{fd:03d}.txt',
                           header=None, names=['RUL'])

    # Piecewise-linear RUL, clipped at 125
    max_cyc = train_df.groupby(0)[1].max().rename('max_cycle')
    train_df = train_df.join(max_cyc, on=0)
    train_df['RUL'] = (train_df['max_cycle'] - train_df[1]).clip(upper=RUL_CAP)

    # Normalize using train statistics only
    feat_mean = train_df[SENSORS].mean()
    feat_std  = train_df[SENSORS].std().replace(0, 1)

    def windows_for_engines(df, engine_list):
        Xs, yr = [], []
        for eng in engine_list:
            sub   = df[df[0] == eng].reset_index(drop=True)
            feats = ((sub[SENSORS] - feat_mean) / feat_std).values.astype(np.float32)
            n = len(feats)
            if n < SEQ:
                pad   = np.zeros((SEQ - n, len(SENSORS)), dtype=np.float32)
                feats = np.vstack([pad, feats])
                n     = SEQ
            for t in range(n - SEQ + 1):
                Xs.append(feats[t:t + SEQ])
                yr.append(float(sub.loc[min(t + SEQ - 1, len(sub) - 1), 'RUL']))
        return (torch.tensor(np.array(Xs)),
                torch.tensor(yr, dtype=torch.float32))

    engines    = train_df[0].unique()
    n_tr_eng   = int(len(engines) * 0.8)
    Xtr, ytr   = windows_for_engines(train_df, engines[:n_tr_eng])
    Xvl, yvl   = windows_for_engines(train_df, engines[n_tr_eng:])

    # Test: last SEQ-step window per engine + provided RUL
    test_engines = test_df[0].unique()
    rul_test     = rul_df['RUL'].values.astype(np.float32)
    Xte_list, yte_list = [], []
    for i, eng in enumerate(test_engines):
        sub   = test_df[test_df[0] == eng].reset_index(drop=True)
        feats = ((sub[SENSORS] - feat_mean) / feat_std).values.astype(np.float32)
        n = len(feats)
        if n < SEQ:
            pad   = np.zeros((SEQ - n, len(SENSORS)), dtype=np.float32)
            feats = np.vstack([pad, feats])
        Xte_list.append(feats[-SEQ:])
        yte_list.append(float(rul_test[i]))

    Xte = torch.tensor(np.array(Xte_list))
    yte = torch.tensor(yte_list, dtype=torch.float32)

    def make_loader(X, y, shuffle=False):
        return DataLoader(TensorDataset(X, y), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    tr = make_loader(Xtr, ytr, shuffle=True)
    va = make_loader(Xvl, yvl)
    te = make_loader(Xte, yte)
    print(f"FD{fd:03d}  tr:{len(Xtr):>6}  val:{len(Xvl):>5}  te:{len(Xte):>4}")
    return tr, va, te

cmapss = {}
for fd in [1, 2, 3, 4]:
    cmapss[fd] = load_cmapss(fd)

FD001  tr: 11098  val: 3233  te: 100
FD002  tr: 29999  val: 7380  te: 259
FD003  tr: 14739  val: 3681  te: 100
FD004  tr: 36535  val: 9027  te: 248


In [3]:
# ============================================================
# CELL 3 — Battery Data Pipeline
# ============================================================
BAT_DIR  = '/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset'
BAT_META = BAT_DIR + '/metadata.csv'

def load_battery():
    meta      = pd.read_csv(BAT_META)
    discharge = meta[meta['type'] == 'discharge'].copy()
    batteries = discharge['battery_id'].unique()

    all_seqs = []
    for bid in batteries:
        rows = discharge[discharge['battery_id'] == bid]\
               .sort_values('start_time').reset_index(drop=True)
        cap_vals = []
        for _, r in rows.iterrows():
            try:
                cap_vals.append(float(r['Capacity']))
            except:
                continue

        n = len(cap_vals)
        if n < SEQ + 5:
            continue

        cap_arr = np.array(cap_vals, dtype=np.float32)
        max_cap = cap_arr[0] if cap_arr[0] > 0 else cap_arr.max()
        soh     = np.clip(cap_arr / max_cap, 0.0, 1.0)
        delta   = np.diff(soh, prepend=soh[0])
        cyc_n   = np.arange(n, dtype=np.float32) / n
        feats   = np.stack([soh, delta, cyc_n, cap_arr / (max_cap + 1e-8)], axis=1)

        # RUL = cycles to SoH < 0.8, clipped at 200
        eol_idx = int(np.where(soh < 0.8)[0][0]) if np.any(soh < 0.8) else n
        rul     = np.clip((eol_idx - np.arange(n)).astype(np.float32), 0, 200)

        for t in range(n - SEQ + 1):
            all_seqs.append((feats[t:t + SEQ], rul[t + SEQ - 1]))

    np.random.RandomState(SEED).shuffle(all_seqs)
    n_total = len(all_seqs)
    n_tr    = int(n_total * 0.6)
    n_vl    = int(n_total * 0.2)

    def to_loader(seqs, shuffle=False):
        X  = torch.tensor(np.array([s[0] for s in seqs]))
        yr = torch.tensor([s[1] for s in seqs], dtype=torch.float32)
        return DataLoader(TensorDataset(X, yr), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    tr = to_loader(all_seqs[:n_tr], shuffle=True)
    va = to_loader(all_seqs[n_tr:n_tr + n_vl])
    te = to_loader(all_seqs[n_tr + n_vl:])
    print(f"Battery  tr:{n_tr:>6}  val:{n_vl:>5}  te:{n_total-n_tr-n_vl:>4}  total:{n_total}")
    return tr, va, te

bat_tr, bat_va, bat_te = load_battery()
BAT_FEAT = 4

Battery  tr:   652  val:  217  te: 218  total:1087


In [4]:
# ============================================================
# CELL 4 — CWRU Data Pipeline
# ============================================================
CWRU_DIR     = '/kaggle/input/datasets/sufian79/cwru-mat-full-dataset'
N_CWRU_CLS   = 10
CWRU_SEQ     = 1024

# 10-class standard split: normal + 3 fault types × 3 severities
CWRU_FILES = {
    '97.mat':0, '98.mat':0, '99.mat':0, '100.mat':0,        # Normal
    '105.mat':1,'106.mat':1,'107.mat':1,'108.mat':1,         # Ball 0.007
    '169.mat':2,'170.mat':2,'171.mat':2,'172.mat':2,         # Ball 0.014
    '209.mat':3,'210.mat':3,'211.mat':3,'212.mat':3,         # Ball 0.021
    '118.mat':4,'119.mat':4,'120.mat':4,'121.mat':4,         # IR 0.007
    '185.mat':5,'186.mat':5,'187.mat':5,'188.mat':5,         # IR 0.014
    '222.mat':6,'223.mat':6,'224.mat':6,'225.mat':6,         # IR 0.021
    '130.mat':7,'131.mat':7,'132.mat':7,'133.mat':7,         # OR 0.007
    '197.mat':8,'198.mat':8,'199.mat':8,'200.mat':8,         # OR 0.014
    '234.mat':9,'235.mat':9,'236.mat':9,'237.mat':9,         # OR 0.021
}

def load_cwru():
    all_X, all_y = [], []
    for fname, cls_id in CWRU_FILES.items():
        fpath = os.path.join(CWRU_DIR, fname)
        if not os.path.exists(fpath):
            print(f"  Missing: {fname}")
            continue
        mat  = scipy.io.loadmat(fpath)
        keys = [k for k in mat.keys() if 'DE_time' in k]
        if not keys:
            continue
        sig  = mat[keys[0]].flatten().astype(np.float32)
        sig  = (sig - sig.mean()) / (sig.std() + 1e-8)
        step = CWRU_SEQ // 2                             # 50% overlap
        for start in range(0, len(sig) - CWRU_SEQ + 1, step):
            all_X.append(sig[start:start + CWRU_SEQ].reshape(CWRU_SEQ, 1))
            all_y.append(cls_id)

    idx   = np.random.RandomState(SEED).permutation(len(all_X))
    all_X = np.array(all_X)[idx]
    all_y = np.array(all_y)[idx]

    n_total = len(all_X)
    n_tr    = int(n_total * 0.6)
    n_vl    = int(n_total * 0.2)

    def to_loader(X, y, shuffle=False):
        return DataLoader(TensorDataset(torch.tensor(X),
                                        torch.tensor(y, dtype=torch.long)),
                          batch_size=BATCH, shuffle=shuffle,
                          num_workers=2, pin_memory=True)

    tr = to_loader(all_X[:n_tr],          all_y[:n_tr],          shuffle=True)
    va = to_loader(all_X[n_tr:n_tr+n_vl], all_y[n_tr:n_tr+n_vl])
    te = to_loader(all_X[n_tr+n_vl:],     all_y[n_tr+n_vl:])
    print(f"CWRU     tr:{n_tr:>6}  val:{n_vl:>5}  te:{n_total-n_tr-n_vl:>4}  classes:{N_CWRU_CLS}")
    return tr, va, te

cwru_tr, cwru_va, cwru_te = load_cwru()

CWRU     tr:  7099  val: 2366  te:2367  classes:10


In [5]:
# ============================================================
# CELL 5 — Architecture
# ============================================================
class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()
        self.conv = nn.Conv1d(d, d, kernel_size=3, padding=dilation,
                              dilation=dilation, padding_mode='zeros')
        self.bn   = nn.BatchNorm1d(d)
        self.act  = nn.GELU()
    def forward(self, x):
        return x + self.act(self.bn(self.conv(x)))

class CrossSensorGate(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):                       # x: (B, T, d)
        g = self.gate(x.mean(dim=1, keepdim=True))  # (B, 1, d)
        return x * g

class TinyPrognostics(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3):
        super().__init__()
        self.embed = nn.Linear(n_sensors, d)
        self.tcn   = nn.Sequential(
            DilatedBlock(d, 1),
            DilatedBlock(d, 2),
            DilatedBlock(d, 4),
        )
        self.gate  = CrossSensorGate(d)
        self.skip  = nn.Linear(n_sensors, d)
        self.gru   = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head   = nn.Sequential(
            nn.Linear(d, 32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32, 1))
        self.state_head = nn.Sequential(
            nn.Linear(d, 32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32, n_classes))

    def forward(self, x):                       # x: (B, T, C)
        h = self.embed(x).permute(0, 2, 1)     # (B, d, T)
        h = self.tcn(h).permute(0, 2, 1)       # (B, T, d)
        h = self.gate(h) + self.skip(x)        # gated + raw skip
        _, hT = self.gru(h)
        hT = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

# Verify size
m      = TinyPrognostics()
params = sum(p.numel() for p in m.parameters())
kb     = sum(p.numel() * p.element_size() for p in m.parameters()) / 1024
print(f"TinyPrognostics: {params:,} params  {kb:.1f} KB")

TinyPrognostics: 12,052 params  47.1 KB


In [6]:
# ============================================================
# CELL 6 — Training Utilities
# ============================================================
def nasa_score(pred: np.ndarray, true: np.ndarray) -> float:
    """Asymmetric NASA prognostic score. Lower is better."""
    d = pred - true
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return float(s.sum()) if np.isfinite(s.sum()) else float('nan')

# ── Regression trainer (C-MAPSS / Battery) ──────────────────
def train_epoch_reg(model, loader, optimizer):
    model.train()
    mse = nn.MSELoss()
    for X, yr in loader:
        X, yr = X.to(DEVICE), yr.to(DEVICE)
        optimizer.zero_grad()
        pred_r, _ = model(X)
        mse(pred_r, yr).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

def evaluate_reg(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, yr in loader:
            pred_r, _ = model(X.to(DEVICE))
            preds.extend(pred_r.cpu().numpy())
            trues.extend(yr.numpy())
    pr, tr = np.array(preds), np.array(trues)
    return (float(np.sqrt(np.mean((pr - tr) ** 2))),
            float(np.mean(np.abs(pr - tr))),
            nasa_score(pr, tr))

def train_reg(model, tr, va, te,
              epochs=EPOCHS, lr=1e-3, patience=PATIENCE):
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    best_val, best_sd, wait = float('inf'), None, 0
    for _ in range(epochs):
        train_epoch_reg(model, tr, opt)
        val_rmse, _, _ = evaluate_reg(model, va)
        sched.step()
        if val_rmse < best_val:
            best_val = val_rmse
            best_sd  = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_sd)
    return evaluate_reg(model, te)

# ── Classification trainer (CWRU) ───────────────────────────
def train_epoch_cls(model, loader, optimizer):
    model.train()
    ce = nn.CrossEntropyLoss()
    for X, ys in loader:
        X, ys = X.to(DEVICE), ys.to(DEVICE)
        optimizer.zero_grad()
        _, pred_s = model(X)
        ce(pred_s, ys).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

def evaluate_cls(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, ys in loader:
            _, pred_s = model(X.to(DEVICE))
            preds.extend(pred_s.argmax(1).cpu().numpy())
            trues.extend(ys.numpy())
    return float((np.array(preds) == np.array(trues)).mean() * 100)

def train_cls(model, tr, va, te,
              epochs=EPOCHS, lr=1e-3, patience=PATIENCE):
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    best_val, best_sd, wait = -1.0, None, 0
    for _ in range(epochs):
        train_epoch_cls(model, tr, opt)
        val_acc = evaluate_cls(model, va)
        sched.step()
        if val_acc > best_val:
            best_val = val_acc
            best_sd  = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_sd)
    return evaluate_cls(model, te)

In [7]:
# ============================================================
# CELL 7 — Main Results: TinyPrognostics on C-MAPSS
# ============================================================
results = {}
print(f"{'Dataset':8} {'RMSE':>8} {'MAE':>8} {'NASAScore':>12}")
print("-" * 42)

for fd in [1, 2, 3, 4]:
    torch.manual_seed(SEED)
    model = TinyPrognostics(n_sensors=14, d=24, n_classes=3).to(DEVICE)
    tr, va, te = cmapss[fd]
    rmse, mae, score = train_reg(model, tr, va, te)
    results[f'FD{fd:03d}'] = dict(rmse=rmse, mae=mae, score=score)
    torch.save(model.state_dict(), f'/kaggle/working/tiny_fd{fd:03d}.pt')
    print(f"FD{fd:03d}   {rmse:8.4f} {mae:8.4f} {score:12.2f}")

Dataset      RMSE      MAE    NASAScore
------------------------------------------
FD001    15.1547  11.4861       359.19
FD002    27.1246  19.2530      6590.53
FD003    14.0647   9.9892       571.30
FD004    28.5491  20.6789      7296.92


In [8]:
# ============================================================
# CELL 8 — Battery Results
# ============================================================
torch.manual_seed(SEED)
bat_model = TinyPrognostics(n_sensors=BAT_FEAT, d=24, n_classes=3).to(DEVICE)
rmse, mae, score = train_reg(bat_model, bat_tr, bat_va, bat_te)
results['battery'] = dict(rmse=rmse, mae=mae, score=score)
torch.save(bat_model.state_dict(), '/kaggle/working/tiny_battery.pt')
print(f"Battery  RMSE:{rmse:.4f}  MAE:{mae:.4f}  Score:{score:.2f}")

Battery  RMSE:1.2281  MAE:0.2456  Score:5.65


In [9]:
# ============================================================
# CELL 9 — CWRU Results
# ============================================================
torch.manual_seed(SEED)
cwru_model = TinyPrognostics(n_sensors=1, d=24, n_classes=N_CWRU_CLS).to(DEVICE)
acc = train_cls(cwru_model, cwru_tr, cwru_va, cwru_te)
results['cwru'] = dict(acc=acc)
torch.save(cwru_model.state_dict(), '/kaggle/working/tiny_cwru.pt')
print(f"CWRU  Accuracy: {acc:.2f}%  ({N_CWRU_CLS}-class fault classification)")

CWRU  Accuracy: 99.83%  (10-class fault classification)


In [10]:
# ============================================================
# CELL 10 — Baselines (Ridge, LSTM-64, CNN-32)
# ============================================================
# Ridge
print("RIDGE BASELINE")
for fd in [1, 2, 3, 4]:
    tr, va, te = cmapss[fd]
    Xtr_np = torch.cat([b[0] for b in tr]).numpy().reshape(-1, SEQ * 14)
    ytr_np = torch.cat([b[1] for b in tr]).numpy()
    Xte_np = torch.cat([b[0] for b in te]).numpy().reshape(-1, SEQ * 14)
    yte_np = torch.cat([b[1] for b in te]).numpy()
    ridge  = Ridge(alpha=1.0).fit(Xtr_np, ytr_np)
    rmse   = float(np.sqrt(mean_squared_error(yte_np, ridge.predict(Xte_np))))
    results[f'FD{fd:03d}']['ridge_rmse'] = rmse
    print(f"  FD{fd:03d}  Ridge RMSE: {rmse:.4f}")

# Neural baselines — SAME epochs/patience as main (EPOCHS, PATIENCE)
class LSTMBaseline(nn.Module):
    def __init__(self, n_sensors=14, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, hidden, 2, batch_first=True, dropout=0.2)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(h[-1]).squeeze(-1), None

class CNNBaseline(nn.Module):
    def __init__(self, n_sensors=14, filters=32):
        super().__init__()
        self.net  = nn.Sequential(
            nn.Conv1d(n_sensors, filters,   3, padding=1), nn.ReLU(), nn.BatchNorm1d(filters),
            nn.Conv1d(filters, filters * 2, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(filters * 2),
            nn.AdaptiveAvgPool1d(1))
        self.head = nn.Linear(filters * 2, 1)
    def forward(self, x):
        h = self.net(x.permute(0, 2, 1)).squeeze(-1)
        return self.head(h).squeeze(-1), None

# Adapter: baselines return (pred, None); wrap evaluate_reg
def evaluate_reg_baseline(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, yr in loader:
            pred_r, _ = model(X.to(DEVICE))
            preds.extend(pred_r.cpu().numpy())
            trues.extend(yr.numpy())
    pr, tr = np.array(preds), np.array(trues)
    return float(np.sqrt(np.mean((pr - tr) ** 2))), float(np.mean(np.abs(pr - tr))), nasa_score(pr, tr)

def train_reg_baseline(model, tr, va, te):
    opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    mse   = nn.MSELoss()
    best_val, best_sd, wait = float('inf'), None, 0
    for _ in range(EPOCHS):
        model.train()
        for X, yr in tr:
            X, yr = X.to(DEVICE), yr.to(DEVICE)
            opt.zero_grad()
            pred, _ = model(X)
            mse(pred, yr).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        val_rmse, _, _ = evaluate_reg_baseline(model, va)
        if val_rmse < best_val:
            best_val = val_rmse
            best_sd  = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_sd)
    return evaluate_reg_baseline(model, te)

print(f"\n{'Model':16} {'FD001':>8} {'FD002':>8} {'FD003':>8} {'FD004':>8} {'Size':>8}")
for name, Cls, kw in [('LSTM-64', LSTMBaseline, {'hidden': 64}),
                      ('CNN-32',  CNNBaseline,  {'filters': 32})]:
    row = []
    for fd in [1, 2, 3, 4]:
        torch.manual_seed(SEED)
        m  = Cls(**kw).to(DEVICE)
        tr, va, te = cmapss[fd]
        rmse, _, _ = train_reg_baseline(m, tr, va, te)
        results[f'FD{fd:03d}'][f'{name}_rmse'] = rmse
        row.append(rmse)
    kb = sum(p.numel() * p.element_size() for p in Cls(**kw).parameters()) / 1024
    print(f"{name:16} {row[0]:8.4f} {row[1]:8.4f} {row[2]:8.4f} {row[3]:8.4f} {kb:6.1f}KB")

RIDGE BASELINE
  FD001  Ridge RMSE: 43.2855
  FD002  Ridge RMSE: 55.2645
  FD003  Ridge RMSE: 42.4939
  FD004  Ridge RMSE: 55.4057

Model               FD001    FD002    FD003    FD004     Size
LSTM-64           16.3336  28.6739  15.4139  29.3802  210.3KB
CNN-32            27.2845  41.2742  25.9456  36.4140   30.6KB


In [11]:
# ============================================================
# CELL 11 — Ablation Study (IDENTICAL config to main: EPOCHS, PATIENCE)
# ============================================================
class TinyAblation(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3,
                 no_gate=False, no_skip=False, no_dilation=False):
        super().__init__()
        self.no_gate = no_gate
        self.no_skip = no_skip
        self.embed   = nn.Linear(n_sensors, d)
        dilations    = [1, 1, 1] if no_dilation else [1, 2, 4]
        self.tcn     = nn.Sequential(*[DilatedBlock(d, dil) for dil in dilations])
        if not no_gate: self.gate = CrossSensorGate(d)
        if not no_skip: self.skip = nn.Linear(n_sensors, d)
        self.gru     = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head   = nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Linear(32, 1))
        self.state_head = nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Linear(32, n_classes))

    def forward(self, x):
        h = self.embed(x).permute(0, 2, 1)
        h = self.tcn(h).permute(0, 2, 1)
        if not self.no_gate: h = self.gate(h)
        if not self.no_skip: h = h + self.skip(x)
        _, hT = self.gru(h)
        hT = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

# Wrap for regression (same train_reg but model has two outputs)
def train_reg_ablation(model, tr, va, te):
    opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    mse   = nn.MSELoss()
    best_val, best_sd, wait = float('inf'), None, 0
    for _ in range(EPOCHS):
        model.train()
        for X, yr in tr:
            X, yr = X.to(DEVICE), yr.to(DEVICE)
            opt.zero_grad()
            pred_r, _ = model(X)
            mse(pred_r, yr).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for X, yr in va:
                pred_r, _ = model(X.to(DEVICE))
                preds.extend(pred_r.cpu().numpy())
                trues.extend(yr.numpy())
        val_rmse = float(np.sqrt(np.mean((np.array(preds) - np.array(trues)) ** 2)))
        if val_rmse < best_val:
            best_val = val_rmse
            best_sd  = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(best_sd)
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, yr in te:
            pred_r, _ = model(X.to(DEVICE))
            preds.extend(pred_r.cpu().numpy())
            trues.extend(yr.numpy())
    pr, tr_arr = np.array(preds), np.array(trues)
    return float(np.sqrt(np.mean((pr - tr_arr) ** 2)))

ablation_cfgs = {
    'Full':       dict(no_gate=False, no_skip=False, no_dilation=False),
    'w/o Gate':   dict(no_gate=True,  no_skip=False, no_dilation=False),
    'w/o Skip':   dict(no_gate=False, no_skip=True,  no_dilation=False),
    'w/o Dil.':   dict(no_gate=False, no_skip=False, no_dilation=True),
}

abl = {}
print(f"{'Config':12} {'FD001':>8} {'FD003':>8}  {'ΔRMSE FD001':>12} {'ΔRMSE FD003':>12}")
print("-" * 58)
base_fd1 = base_fd3 = None
for name, cfg in ablation_cfgs.items():
    row = {}
    for fd in [1, 3]:
        torch.manual_seed(SEED)
        m = TinyAblation(n_sensors=14, **cfg).to(DEVICE)
        row[fd] = train_reg_ablation(m, *cmapss[fd])
    abl[name] = row
    if base_fd1 is None:
        base_fd1, base_fd3 = row[1], row[3]
    d1 = row[1] - base_fd1
    d3 = row[3] - base_fd3
    print(f"{name:12} {row[1]:8.4f} {row[3]:8.4f}  {d1:+12.4f} {d3:+12.4f}")

Config          FD001    FD003   ΔRMSE FD001  ΔRMSE FD003
----------------------------------------------------------
Full          16.2166  14.4355       +0.0000      +0.0000
w/o Gate      18.6918  14.2138       +2.4752      -0.2216
w/o Skip      14.5053  14.6780       -1.7113      +0.2425
w/o Dil.      14.8423  14.0499       -1.3743      -0.3855


In [12]:
# ============================================================
# CELL 12 — Transfer Learning
# ============================================================
fracs = [0.10, 0.25, 0.50, 1.00]
transfer_results = {}

def run_transfer(source_pt, target_loaders, src_sensors, tgt_sensors, frac):
    tr_full, va, te = target_loaders
    src_model = TinyPrognostics(n_sensors=src_sensors, d=24, n_classes=3).to(DEVICE)
    src_model.load_state_dict(torch.load(source_pt, map_location=DEVICE))

    tgt_model = TinyPrognostics(n_sensors=tgt_sensors, d=24, n_classes=3).to(DEVICE)
    src_sd, tgt_sd = src_model.state_dict(), tgt_model.state_dict()
    for k in tgt_sd:
        if k in src_sd and src_sd[k].shape == tgt_sd[k].shape:
            tgt_sd[k] = src_sd[k].clone()
    tgt_model.load_state_dict(tgt_sd)

    # Freeze shared temporal backbone
    for p in tgt_model.tcn.parameters(): p.requires_grad = False
    for p in tgt_model.gru.parameters(): p.requires_grad = False

    if frac < 1.0:
        ds    = tr_full.dataset
        n_keep = max(64, int(len(ds) * frac))
        idx   = torch.randperm(len(ds), generator=torch.Generator().manual_seed(SEED))[:n_keep]
        tr    = DataLoader(Subset(ds, idx), batch_size=BATCH, shuffle=True,
                           num_workers=2, pin_memory=True)
    else:
        tr = tr_full

    opt   = optim.AdamW(filter(lambda p: p.requires_grad, tgt_model.parameters()),
                        lr=5e-4, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60, eta_min=1e-5)
    mse   = nn.MSELoss()
    best_val, best_sd, wait = float('inf'), None, 0

    for _ in range(60):
        tgt_model.train()
        for X, yr in tr:
            X, yr = X.to(DEVICE), yr.to(DEVICE)
            opt.zero_grad()
            pred_r, _ = tgt_model(X)
            mse(pred_r, yr).backward()
            nn.utils.clip_grad_norm_(tgt_model.parameters(), 1.0)
            opt.step()
        sched.step()
        tgt_model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for X, yr in va:
                pred_r, _ = tgt_model(X.to(DEVICE))
                preds.extend(pred_r.cpu().numpy())
                trues.extend(yr.numpy())
        val_rmse = float(np.sqrt(np.mean((np.array(preds) - np.array(trues)) ** 2)))
        if val_rmse < best_val:
            best_val = val_rmse
            best_sd  = {k: v.clone() for k, v in tgt_model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= 15: break

    tgt_model.load_state_dict(best_sd)
    rmse, mae, score = evaluate_reg(tgt_model, te)
    return rmse

# FD001 → FD003
print(f"FD001 → FD003  (scratch={results['FD003']['rmse']:.4f})")
print(f"{'Frac':>6} {'RMSE':>8}")
transfer_results['FD003'] = {}
for frac in fracs:
    rmse = run_transfer('/kaggle/working/tiny_fd001.pt', cmapss[3],
                        src_sensors=14, tgt_sensors=14, frac=frac)
    transfer_results['FD003'][frac] = rmse
    print(f"{frac*100:5.0f}%  {rmse:8.4f}")

# FD001 → Battery
print(f"\nFD001 → Battery  (scratch={results['battery']['rmse']:.4f})")
print(f"{'Frac':>6} {'RMSE':>8}")
transfer_results['Battery'] = {}
for frac in fracs:
    rmse = run_transfer('/kaggle/working/tiny_fd001.pt',
                        (bat_tr, bat_va, bat_te),
                        src_sensors=14, tgt_sensors=BAT_FEAT, frac=frac)
    transfer_results['Battery'][frac] = rmse
    print(f"{frac*100:5.0f}%  {rmse:8.4f}")

FD001 → FD003  (scratch=14.0647)
  Frac     RMSE
   10%   15.4123
   25%   14.7716
   50%   15.0228
  100%   13.9917

FD001 → Battery  (scratch=1.2281)
  Frac     RMSE
   10%   20.4087
   25%    8.4871
   50%    4.0837
  100%    3.6558


In [ ]:
# ============================================================
# CELL 13 — Master Results Summary
# ============================================================
print("\n" + "=" * 72)
print("TINYPROGNOSTICS — MASTER RESULTS TABLE")
print("=" * 72)

tiny_kb = sum(p.numel()*p.element_size() for p in TinyPrognostics().parameters())/1024
lstm_kb = sum(p.numel()*p.element_size() for p in LSTMBaseline().parameters())/1024
cnn_kb  = sum(p.numel()*p.element_size() for p in CNNBaseline().parameters())/1024
print(f"Model sizes:  TinyPrognostics={tiny_kb:.1f}KB  LSTM-64={lstm_kb:.1f}KB  CNN-32={cnn_kb:.1f}KB\n")

print(f"{'Dataset':8} {'Tiny':>8} {'Score':>10} {'LSTM':>8} {'CNN':>8} {'Ridge':>8}")
print("-" * 56)
for fd in [1, 2, 3, 4]:
    k = f'FD{fd:03d}'
    r = results[k]
    print(f"{k:8} {r['rmse']:8.4f} {r['score']:10.2f} "
          f"{r.get('LSTM-64_rmse', float('nan')):8.4f} "
          f"{r.get('CNN-32_rmse', float('nan')):8.4f} "
          f"{r.get('ridge_rmse', float('nan')):8.4f}")

r = results['battery']
print(f"{'Battery':8} {r['rmse']:8.4f}  RMSE(cycles)  MAE:{r['mae']:.4f}")
r = results['cwru']
print(f"{'CWRU':8} {'Acc:':>8} {r['acc']:.2f}%  ({N_CWRU_CLS}-class)")

print("\nABLATION (FD001 / FD003 RMSE)")
print(f"{'Config':12} {'FD001':>8} {'FD003':>8} {'ΔRMSE FD001':>12} {'ΔRMSE FD003':>12}")
bf1, bf3 = abl['Full'][1], abl['Full'][3]
for name, row in abl.items():
    print(f"{name:12} {row[1]:8.4f} {row[3]:8.4f} {row[1]-bf1:+12.4f} {row[3]-bf3:+12.4f}")

print("\nTRANSFER LEARNING (RMSE)")
print(f"{'Target':10} {'10%':>8} {'25%':>8} {'50%':>8} {'100%':>8} {'Scratch':>8}")
for tgt, fd in [('FD003', 'FD003'), ('Battery', 'battery')]:
    scratch = results[fd]['rmse']
    vals    = [transfer_results[tgt].get(f, float('nan')) for f in fracs]
    print(f"{tgt:10} {vals[0]:8.4f} {vals[1]:8.4f} {vals[2]:8.4f} {vals[3]:8.4f} {scratch:8.4f}")

print("=" * 72)


TINYPROGNOSTICS — MASTER RESULTS TABLE
Model sizes:  TinyPrognostics=47.1KB  LSTM-64=210.3KB  CNN-32=30.6KB

Dataset      Tiny      Score     LSTM      CNN    Ridge
--------------------------------------------------------
FD001     15.1547     359.19  16.3336  27.2845  43.2855
FD002     27.1246    6590.53  28.6739  41.2742  55.2645
FD003     14.0647     571.30  15.4139  25.9456  42.4939
FD004     28.5491    7296.92  29.3802  36.4140  55.4057
Battery    1.2281  RMSE(cycles)  MAE:0.2456
CWRU         Acc: 99.83%  (10-class)

ABLATION (FD001 / FD003 RMSE)
Config          FD001    FD003  ΔRMSE FD001  ΔRMSE FD003
Full          16.2166  14.4355      +0.0000      +0.0000
w/o Gate      18.6918  14.2138      +2.4752      -0.2216
w/o Skip      14.5053  14.6780      -1.7113      +0.2425
w/o Dil.      14.8423  14.0499      -1.3743      -0.3855

TRANSFER LEARNING (RMSE)
Target          10%      25%      50%     100%  Scratch
FD003       15.4123  14.7716  15.0228  13.9917  14.0647
Battery     20.408

In [3]:
# ============================================================
# CELL 1 — Imports & Global Config
# ============================================================
import os, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import scipy.io
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ      = 64
BATCH    = 256
SEED     = 42
EPOCHS   = 150    # single value used everywhere
PATIENCE = 30
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

Device: cpu


In [4]:
# ============================================================
# CELL 2 — C-MAPSS Data Pipeline (with condition-aware norm)
# ============================================================
CMAPSS_DIR = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps'
SENSORS    = [6,7,8,11,12,13,15,16,17,18,19,21,24,25]  # 14 informative sensors
OP_COLS    = [2, 3, 4]   # 3 operating setting columns
RUL_CAP    = 125

def normalize_cmapss(train_df, test_df, fd):
    """
    FD001, FD003 (single condition): global z-score normalization.
    FD002, FD004 (6 conditions):     K-means cluster-wise z-score normalization.
    Fit only on train, apply to test.
    """
    if fd in [1, 3]:
        mu  = train_df[SENSORS].mean()
        std = train_df[SENSORS].std().replace(0, 1)
        train_df = train_df.copy()
        test_df  = test_df.copy()
        train_df[SENSORS] = (train_df[SENSORS] - mu) / std
        test_df[SENSORS]  = (test_df[SENSORS]  - mu) / std
        return train_df, test_df

    # FD002, FD004: fit K-means on training op settings, normalize per cluster
    km = KMeans(n_clusters=6, random_state=SEED, n_init=10)
    km.fit(train_df[OP_COLS].values)

    train_df = train_df.copy()
    test_df  = test_df.copy()

    tr_labels = km.predict(train_df[OP_COLS].values)
    te_labels = km.predict(test_df[OP_COLS].values)

    cluster_stats = {}
    for c in range(6):
        mask = tr_labels == c
        if mask.sum() < 2:
            continue
        mu  = train_df.loc[mask, SENSORS].mean()
        std = train_df.loc[mask, SENSORS].std().replace(0, 1)
        cluster_stats[c] = (mu, std)
        train_df.loc[mask, SENSORS] = (train_df.loc[mask, SENSORS] - mu) / std

    for c in range(6):
        mask = te_labels == c
        if mask.sum() == 0 or c not in cluster_stats:
            continue
        mu, std = cluster_stats[c]
        test_df.loc[mask, SENSORS] = (test_df.loc[mask, SENSORS] - mu) / std

    return train_df, test_df


def load_cmapss(fd):
    train_df = pd.read_csv(f'{CMAPSS_DIR}/train_FD{fd:03d}.txt', sep=r'\s+', header=None)
    test_df  = pd.read_csv(f'{CMAPSS_DIR}/test_FD{fd:03d}.txt',  sep=r'\s+', header=None)
    rul_df   = pd.read_csv(f'{CMAPSS_DIR}/RUL_FD{fd:03d}.txt',   header=None, names=['RUL'])

    # Piecewise-linear RUL label
    max_cyc  = train_df.groupby(0)[1].max().rename('max_cycle')
    train_df = train_df.join(max_cyc, on=0)
    train_df['RUL'] = (train_df['max_cycle'] - train_df[1]).clip(upper=RUL_CAP)

    # Condition-aware normalization
    train_df, test_df = normalize_cmapss(train_df, test_df, fd)

    def make_windows(df, engine_list):
        Xs, yr = [], []
        for eng in engine_list:
            sub   = df[df[0] == eng].reset_index(drop=True)
            feats = sub[SENSORS].values.astype(np.float32)
            n     = len(feats)
            if n < SEQ:
                feats = np.vstack([np.zeros((SEQ - n, len(SENSORS)), np.float32), feats])
                n     = SEQ
            for t in range(n - SEQ + 1):
                Xs.append(feats[t:t + SEQ])
                yr.append(float(sub.loc[min(t + SEQ - 1, len(sub) - 1), 'RUL']))
        return torch.tensor(np.array(Xs)), torch.tensor(yr, dtype=torch.float32)

    engines  = train_df[0].unique()
    n_tr_eng = int(len(engines) * 0.8)
    Xtr, ytr = make_windows(train_df, engines[:n_tr_eng])
    Xvl, yvl = make_windows(train_df, engines[n_tr_eng:])

    # Test: last SEQ window per engine + provided RUL
    test_engines = test_df[0].unique()
    rul_test     = rul_df['RUL'].values.astype(np.float32)
    Xte_list, yte_list = [], []
    for i, eng in enumerate(test_engines):
        sub   = test_df[test_df[0] == eng].reset_index(drop=True)
        feats = sub[SENSORS].values.astype(np.float32)
        n     = len(feats)
        if n < SEQ:
            feats = np.vstack([np.zeros((SEQ - n, len(SENSORS)), np.float32), feats])
        Xte_list.append(feats[-SEQ:])
        yte_list.append(float(rul_test[i]))

    Xte = torch.tensor(np.array(Xte_list))
    yte = torch.tensor(yte_list, dtype=torch.float32)

    def loader(X, y, shuffle=False):
        return DataLoader(TensorDataset(X, y), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    print(f"FD{fd:03d}  tr:{len(Xtr):>6}  val:{len(Xvl):>5}  te:{len(Xte):>4}")
    return loader(Xtr, ytr, True), loader(Xvl, yvl), loader(Xte, yte)

cmapss = {}
for fd in [1, 2, 3, 4]:
    cmapss[fd] = load_cmapss(fd)

FD001  tr: 11098  val: 3233  te: 100
FD002  tr: 29999  val: 7380  te: 259
FD003  tr: 14739  val: 3681  te: 100
FD004  tr: 36535  val: 9027  te: 248


In [5]:
# ============================================================
# CELL 3 — Battery Data Pipeline
# ============================================================
BAT_DIR  = '/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset'
BAT_META = BAT_DIR + '/metadata.csv'

def load_battery():
    meta      = pd.read_csv(BAT_META)
    discharge = meta[meta['type'] == 'discharge'].copy()
    batteries = discharge['battery_id'].unique()

    all_seqs = []
    for bid in batteries:
        rows = discharge[discharge['battery_id'] == bid]\
               .sort_values('start_time').reset_index(drop=True)
        cap_vals = []
        for _, r in rows.iterrows():
            try:
                cap_vals.append(float(r['Capacity']))
            except:
                continue

        n = len(cap_vals)
        if n < SEQ + 5:
            continue

        cap_arr = np.array(cap_vals, dtype=np.float32)
        max_cap = cap_arr[0] if cap_arr[0] > 0 else cap_arr.max()
        soh     = np.clip(cap_arr / max_cap, 0.0, 1.0)
        delta   = np.diff(soh, prepend=soh[0])
        cyc_n   = np.arange(n, dtype=np.float32) / n
        feats   = np.stack([soh, delta, cyc_n, cap_arr / (max_cap + 1e-8)], axis=1)

        eol_idx = int(np.where(soh < 0.8)[0][0]) if np.any(soh < 0.8) else n
        rul     = np.clip((eol_idx - np.arange(n)).astype(np.float32), 0, 200)

        for t in range(n - SEQ + 1):
            all_seqs.append((feats[t:t + SEQ], rul[t + SEQ - 1]))

    np.random.RandomState(SEED).shuffle(all_seqs)
    n_total = len(all_seqs)
    n_tr    = int(n_total * 0.6)
    n_vl    = int(n_total * 0.2)

    def to_loader(seqs, shuffle=False):
        X  = torch.tensor(np.array([s[0] for s in seqs]))
        yr = torch.tensor([s[1] for s in seqs], dtype=torch.float32)
        return DataLoader(TensorDataset(X, yr), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    print(f"Battery  tr:{n_tr}  val:{n_vl}  te:{n_total-n_tr-n_vl}  total:{n_total}")
    return to_loader(all_seqs[:n_tr], True), to_loader(all_seqs[n_tr:n_tr+n_vl]), \
           to_loader(all_seqs[n_tr+n_vl:])

bat_tr, bat_va, bat_te = load_battery()
BAT_FEAT = 4

Battery  tr:652  val:217  te:218  total:1087


In [6]:
# ============================================================
# CELL 4 — CWRU Data Pipeline
# ============================================================
CWRU_DIR   = '/kaggle/input/datasets/sufian79/cwru-mat-full-dataset'
N_CWRU_CLS = 10
CWRU_SEQ   = 1024

CWRU_FILES = {
    '97.mat':0,'98.mat':0,'99.mat':0,'100.mat':0,
    '105.mat':1,'106.mat':1,'107.mat':1,'108.mat':1,
    '169.mat':2,'170.mat':2,'171.mat':2,'172.mat':2,
    '209.mat':3,'210.mat':3,'211.mat':3,'212.mat':3,
    '118.mat':4,'119.mat':4,'120.mat':4,'121.mat':4,
    '185.mat':5,'186.mat':5,'187.mat':5,'188.mat':5,
    '222.mat':6,'223.mat':6,'224.mat':6,'225.mat':6,
    '130.mat':7,'131.mat':7,'132.mat':7,'133.mat':7,
    '197.mat':8,'198.mat':8,'199.mat':8,'200.mat':8,
    '234.mat':9,'235.mat':9,'236.mat':9,'237.mat':9,
}

def load_cwru():
    all_X, all_y = [], []
    for fname, cls_id in CWRU_FILES.items():
        fpath = os.path.join(CWRU_DIR, fname)
        if not os.path.exists(fpath):
            continue
        mat  = scipy.io.loadmat(fpath)
        keys = [k for k in mat.keys() if 'DE_time' in k]
        if not keys:
            continue
        sig  = mat[keys[0]].flatten().astype(np.float32)
        sig  = (sig - sig.mean()) / (sig.std() + 1e-8)
        step = CWRU_SEQ // 2
        for start in range(0, len(sig) - CWRU_SEQ + 1, step):
            all_X.append(sig[start:start + CWRU_SEQ].reshape(CWRU_SEQ, 1))
            all_y.append(cls_id)

    idx   = np.random.RandomState(SEED).permutation(len(all_X))
    all_X = np.array(all_X)[idx]
    all_y = np.array(all_y)[idx]

    n_total = len(all_X)
    n_tr    = int(n_total * 0.6)
    n_vl    = int(n_total * 0.2)

    def to_loader(X, y, shuffle=False):
        return DataLoader(TensorDataset(torch.tensor(X),
                                        torch.tensor(y, dtype=torch.long)),
                          batch_size=BATCH, shuffle=shuffle,
                          num_workers=2, pin_memory=True)

    print(f"CWRU     tr:{n_tr}  val:{n_vl}  te:{n_total-n_tr-n_vl}  classes:{N_CWRU_CLS}")
    return to_loader(all_X[:n_tr], all_y[:n_tr], True), \
           to_loader(all_X[n_tr:n_tr+n_vl], all_y[n_tr:n_tr+n_vl]), \
           to_loader(all_X[n_tr+n_vl:], all_y[n_tr+n_vl:])

cwru_tr, cwru_va, cwru_te = load_cwru()

CWRU     tr:7099  val:2366  te:2367  classes:10


In [7]:
# ============================================================
# CELL 5 — Architecture (unchanged, verified 47.1 KB)
# ============================================================
class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()
        self.conv = nn.Conv1d(d, d, kernel_size=3, padding=dilation,
                              dilation=dilation, padding_mode='zeros')
        self.bn   = nn.BatchNorm1d(d)
        self.act  = nn.GELU()
    def forward(self, x):
        return x + self.act(self.bn(self.conv(x)))

class CrossSensorGate(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x.mean(dim=1, keepdim=True))

class TinyPrognostics(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3):
        super().__init__()
        self.embed      = nn.Linear(n_sensors, d)
        self.tcn        = nn.Sequential(DilatedBlock(d,1), DilatedBlock(d,2), DilatedBlock(d,4))
        self.gate       = CrossSensorGate(d)
        self.skip       = nn.Linear(n_sensors, d)
        self.gru        = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head   = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32,1))
        self.state_head = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32,n_classes))
    def forward(self, x):
        h = self.tcn(self.embed(x).permute(0,2,1)).permute(0,2,1)
        h = self.gate(h) + self.skip(x)
        _, hT = self.gru(h)
        hT = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

params = sum(p.numel() for p in TinyPrognostics().parameters())
kb     = sum(p.numel()*p.element_size() for p in TinyPrognostics().parameters()) / 1024
print(f"TinyPrognostics: {params:,} params  {kb:.1f} KB")

TinyPrognostics: 12,052 params  47.1 KB


In [8]:
# ============================================================
# CELL 6 — Training & Evaluation Utilities
# ============================================================
def nasa_score(pred, true):
    d = pred - true
    s = np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)
    return float(s.sum()) if np.isfinite(s.sum()) else float('nan')

# ── Regression (C-MAPSS, Battery) ───────────────────────────
def train_reg(model, tr, va, te, epochs=EPOCHS, lr=1e-3, patience=PATIENCE):
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    mse   = nn.MSELoss()
    best_val, best_sd, wait = float('inf'), None, 0
    for _ in range(epochs):
        model.train()
        for X, yr in tr:
            X, yr = X.to(DEVICE), yr.to(DEVICE)
            opt.zero_grad()
            mse(model(X)[0], yr).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            p = np.concatenate([model(X.to(DEVICE))[0].cpu().numpy() for X, _ in va])
            t = np.concatenate([y.numpy() for _, y in va])
        v = float(np.sqrt(np.mean((p-t)**2)))
        if v < best_val:
            best_val = v
            best_sd  = {k: v_.clone() for k, v_ in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_sd)
    model.eval()
    with torch.no_grad():
        p = np.concatenate([model(X.to(DEVICE))[0].cpu().numpy() for X, _ in te])
        t = np.concatenate([y.numpy() for _, y in te])
    return float(np.sqrt(np.mean((p-t)**2))), float(np.mean(np.abs(p-t))), nasa_score(p, t)

# ── Classification (CWRU) ────────────────────────────────────
def train_cls(model, tr, va, te, epochs=EPOCHS, lr=1e-3, patience=PATIENCE):
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    ce    = nn.CrossEntropyLoss()
    best_val, best_sd, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train()
        for X, ys in tr:
            X, ys = X.to(DEVICE), ys.to(DEVICE)
            opt.zero_grad()
            ce(model(X)[1], ys).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            p = np.concatenate([model(X.to(DEVICE))[1].argmax(1).cpu().numpy() for X, _ in va])
            t = np.concatenate([y.numpy() for _, y in va])
        v = float((p==t).mean())
        if v > best_val:
            best_val = v
            best_sd  = {k: v_.clone() for k, v_ in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_sd)
    model.eval()
    with torch.no_grad():
        p = np.concatenate([model(X.to(DEVICE))[1].argmax(1).cpu().numpy() for X, _ in te])
        t = np.concatenate([y.numpy() for _, y in te])
    return float((p==t).mean() * 100)

In [9]:
# ============================================================
# CELL 7 — Main Results: TinyPrognostics on C-MAPSS
# ============================================================
results = {}
print(f"{'Dataset':8} {'RMSE':>8} {'MAE':>8} {'NASAScore':>12}")
print("-" * 42)
for fd in [1, 2, 3, 4]:
    torch.manual_seed(SEED)
    model = TinyPrognostics(n_sensors=14, d=24, n_classes=3).to(DEVICE)
    rmse, mae, score = train_reg(model, *cmapss[fd])
    results[f'FD{fd:03d}'] = dict(rmse=rmse, mae=mae, score=score)
    torch.save(model.state_dict(), f'/kaggle/working/tiny_fd{fd:03d}.pt')
    print(f"FD{fd:03d}   {rmse:8.4f} {mae:8.4f} {score:12.2f}")

Dataset      RMSE      MAE    NASAScore
------------------------------------------
FD001    15.1646  11.2276       358.96
FD002    26.0796  17.2055      6559.41
FD003    13.8201   9.9881       281.83
FD004    27.0798  18.2385      6295.72


In [11]:
# ============================================================
# CELL 8 — Battery & CWRU Results
# ============================================================
torch.manual_seed(SEED)
bat_model = TinyPrognostics(n_sensors=BAT_FEAT, d=24, n_classes=3).to(DEVICE)
rmse, mae, score = train_reg(bat_model, bat_tr, bat_va, bat_te)
results['battery'] = dict(rmse=rmse, mae=mae, score=score)
torch.save(bat_model.state_dict(), '/kaggle/working/tiny_battery.pt')
print(f"Battery  RMSE:{rmse:.4f}  MAE:{mae:.4f}  Score:{score:.4f}")

torch.manual_seed(SEED)
cwru_model = TinyPrognostics(n_sensors=1, d=24, n_classes=N_CWRU_CLS).to(DEVICE)
acc = train_cls(cwru_model, cwru_tr, cwru_va, cwru_te)
results['cwru'] = dict(acc=acc)
torch.save(cwru_model.state_dict(), '/kaggle/working/tiny_cwru.pt')
print(f"CWRU     Acc:{acc:.2f}%  ({N_CWRU_CLS}-class fault classification)")

Battery  RMSE:1.8747  MAE:0.3861  Score:9.9967
CWRU     Acc:99.79%  (10-class fault classification)


In [1]:
# ============================================================
# CELL 1 — Imports & Config
# ============================================================
import os, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import scipy.io
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ      = 64
BATCH    = 256
SEED     = 42
EPOCHS   = 150
PATIENCE = 30
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

Device: cpu


In [2]:
# ============================================================
# CELL 2 — C-MAPSS (condition-aware normalization)
# ============================================================
CMAPSS_DIR = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps'
SENSORS    = [6,7,8,11,12,13,15,16,17,18,19,21,24,25]
OP_COLS    = [2,3,4]
RUL_CAP    = 125

def normalize_cmapss(train_df, test_df, fd):
    if fd in [1, 3]:
        mu  = train_df[SENSORS].mean()
        std = train_df[SENSORS].std().replace(0, 1)
        train_df = train_df.copy(); test_df = test_df.copy()
        train_df[SENSORS] = (train_df[SENSORS] - mu) / std
        test_df[SENSORS]  = (test_df[SENSORS]  - mu) / std
        return train_df, test_df
    # FD002/FD004: 6 operating condition clusters
    km        = KMeans(n_clusters=6, random_state=SEED, n_init=10)
    km.fit(train_df[OP_COLS].values)
    train_df  = train_df.copy(); test_df = test_df.copy()
    tr_labels = km.predict(train_df[OP_COLS].values)
    te_labels = km.predict(test_df[OP_COLS].values)
    stats = {}
    for c in range(6):
        mask = tr_labels == c
        if mask.sum() < 2: continue
        mu  = train_df.loc[mask, SENSORS].mean()
        std = train_df.loc[mask, SENSORS].std().replace(0, 1)
        stats[c] = (mu, std)
        train_df.loc[mask, SENSORS] = (train_df.loc[mask, SENSORS] - mu) / std
    for c in range(6):
        mask = te_labels == c
        if mask.sum() == 0 or c not in stats: continue
        mu, std = stats[c]
        test_df.loc[mask, SENSORS] = (test_df.loc[mask, SENSORS] - mu) / std
    return train_df, test_df

def load_cmapss(fd):
    train_df = pd.read_csv(f'{CMAPSS_DIR}/train_FD{fd:03d}.txt', sep=r'\s+', header=None)
    test_df  = pd.read_csv(f'{CMAPSS_DIR}/test_FD{fd:03d}.txt',  sep=r'\s+', header=None)
    rul_df   = pd.read_csv(f'{CMAPSS_DIR}/RUL_FD{fd:03d}.txt',   header=None, names=['RUL'])

    max_cyc  = train_df.groupby(0)[1].max().rename('max_cycle')
    train_df = train_df.join(max_cyc, on=0)
    train_df['RUL'] = (train_df['max_cycle'] - train_df[1]).clip(upper=RUL_CAP)
    train_df, test_df = normalize_cmapss(train_df, test_df, fd)

    def make_windows(df, engine_list):
        Xs, yr = [], []
        for eng in engine_list:
            sub   = df[df[0]==eng].reset_index(drop=True)
            feats = sub[SENSORS].values.astype(np.float32)
            n     = len(feats)
            if n < SEQ:
                feats = np.vstack([np.zeros((SEQ-n, len(SENSORS)), np.float32), feats])
                n     = SEQ
            for t in range(n - SEQ + 1):
                Xs.append(feats[t:t+SEQ])
                yr.append(float(sub.loc[min(t+SEQ-1, len(sub)-1), 'RUL']))
        return torch.tensor(np.array(Xs)), torch.tensor(yr, dtype=torch.float32)

    engines  = train_df[0].unique()
    n_tr_eng = int(len(engines) * 0.8)
    Xtr, ytr = make_windows(train_df, engines[:n_tr_eng])
    Xvl, yvl = make_windows(train_df, engines[n_tr_eng:])

    test_engines = test_df[0].unique()
    rul_test     = rul_df['RUL'].values.astype(np.float32)
    Xte_list, yte_list = [], []
    for i, eng in enumerate(test_engines):
        sub   = test_df[test_df[0]==eng].reset_index(drop=True)
        feats = sub[SENSORS].values.astype(np.float32)
        n     = len(feats)
        if n < SEQ:
            feats = np.vstack([np.zeros((SEQ-n, len(SENSORS)), np.float32), feats])
        Xte_list.append(feats[-SEQ:])
        yte_list.append(float(rul_test[i]))

    Xte = torch.tensor(np.array(Xte_list))
    yte = torch.tensor(yte_list, dtype=torch.float32)

    def loader(X, y, shuffle=False):
        return DataLoader(TensorDataset(X, y), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)

    print(f"FD{fd:03d}  tr:{len(Xtr):>6}  val:{len(Xvl):>5}  te:{len(Xte):>4}")
    return loader(Xtr, ytr, True), loader(Xvl, yvl), loader(Xte, yte)

cmapss = {}
for fd in [1, 2, 3, 4]:
    cmapss[fd] = load_cmapss(fd)

FD001  tr: 11098  val: 3233  te: 100
FD002  tr: 29999  val: 7380  te: 259
FD003  tr: 14739  val: 3681  te: 100
FD004  tr: 36535  val: 9027  te: 248


In [3]:
# ============================================================
# CELL 3 — Battery
# ============================================================
BAT_DIR  = '/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset'
BAT_META = BAT_DIR + '/metadata.csv'

def load_battery():
    meta      = pd.read_csv(BAT_META)
    discharge = meta[meta['type'] == 'discharge'].copy()
    batteries = discharge['battery_id'].unique()
    all_seqs  = []
    for bid in batteries:
        rows = discharge[discharge['battery_id']==bid]\
               .sort_values('start_time').reset_index(drop=True)
        cap_vals = []
        for _, r in rows.iterrows():
            try: cap_vals.append(float(r['Capacity']))
            except: continue
        n = len(cap_vals)
        if n < SEQ + 5: continue
        cap_arr = np.array(cap_vals, dtype=np.float32)
        max_cap = cap_arr[0] if cap_arr[0] > 0 else cap_arr.max()
        soh     = np.clip(cap_arr / max_cap, 0.0, 1.0)
        delta   = np.diff(soh, prepend=soh[0])
        cyc_n   = np.arange(n, dtype=np.float32) / n
        feats   = np.stack([soh, delta, cyc_n, cap_arr/(max_cap+1e-8)], axis=1)
        eol_idx = int(np.where(soh<0.8)[0][0]) if np.any(soh<0.8) else n
        rul     = np.clip((eol_idx - np.arange(n)).astype(np.float32), 0, 200)
        for t in range(n - SEQ + 1):
            all_seqs.append((feats[t:t+SEQ], rul[t+SEQ-1]))
    np.random.RandomState(SEED).shuffle(all_seqs)
    n_total = len(all_seqs); n_tr = int(n_total*0.6); n_vl = int(n_total*0.2)
    def to_loader(seqs, shuffle=False):
        X  = torch.tensor(np.array([s[0] for s in seqs]))
        yr = torch.tensor([s[1] for s in seqs], dtype=torch.float32)
        return DataLoader(TensorDataset(X, yr), batch_size=BATCH,
                          shuffle=shuffle, num_workers=2, pin_memory=True)
    print(f"Battery  tr:{n_tr}  val:{n_vl}  te:{n_total-n_tr-n_vl}  total:{n_total}")
    return to_loader(all_seqs[:n_tr], True), \
           to_loader(all_seqs[n_tr:n_tr+n_vl]), \
           to_loader(all_seqs[n_tr+n_vl:])

bat_tr, bat_va, bat_te = load_battery()
BAT_FEAT = 4

Battery  tr:652  val:217  te:218  total:1087


In [4]:
# ============================================================
# CELL 4 — CWRU
# ============================================================
CWRU_DIR   = '/kaggle/input/datasets/sufian79/cwru-mat-full-dataset'
N_CWRU_CLS = 10
CWRU_SEQ   = 1024

CWRU_FILES = {
    '97.mat':0,'98.mat':0,'99.mat':0,'100.mat':0,
    '105.mat':1,'106.mat':1,'107.mat':1,'108.mat':1,
    '169.mat':2,'170.mat':2,'171.mat':2,'172.mat':2,
    '209.mat':3,'210.mat':3,'211.mat':3,'212.mat':3,
    '118.mat':4,'119.mat':4,'120.mat':4,'121.mat':4,
    '185.mat':5,'186.mat':5,'187.mat':5,'188.mat':5,
    '222.mat':6,'223.mat':6,'224.mat':6,'225.mat':6,
    '130.mat':7,'131.mat':7,'132.mat':7,'133.mat':7,
    '197.mat':8,'198.mat':8,'199.mat':8,'200.mat':8,
    '234.mat':9,'235.mat':9,'236.mat':9,'237.mat':9,
}

def load_cwru():
    all_X, all_y = [], []
    for fname, cls_id in CWRU_FILES.items():
        fpath = os.path.join(CWRU_DIR, fname)
        if not os.path.exists(fpath): continue
        mat  = scipy.io.loadmat(fpath)
        keys = [k for k in mat.keys() if 'DE_time' in k]
        if not keys: continue
        sig  = mat[keys[0]].flatten().astype(np.float32)
        sig  = (sig - sig.mean()) / (sig.std() + 1e-8)
        step = CWRU_SEQ // 2
        for start in range(0, len(sig)-CWRU_SEQ+1, step):
            all_X.append(sig[start:start+CWRU_SEQ].reshape(CWRU_SEQ, 1))
            all_y.append(cls_id)
    idx   = np.random.RandomState(SEED).permutation(len(all_X))
    all_X = np.array(all_X)[idx]; all_y = np.array(all_y)[idx]
    n_total = len(all_X); n_tr = int(n_total*0.6); n_vl = int(n_total*0.2)
    def to_loader(X, y, shuffle=False):
        return DataLoader(TensorDataset(torch.tensor(X),
                                        torch.tensor(y, dtype=torch.long)),
                          batch_size=BATCH, shuffle=shuffle,
                          num_workers=2, pin_memory=True)
    print(f"CWRU     tr:{n_tr}  val:{n_vl}  te:{n_total-n_tr-n_vl}  classes:{N_CWRU_CLS}")
    return (to_loader(all_X[:n_tr], all_y[:n_tr], True),
            to_loader(all_X[n_tr:n_tr+n_vl], all_y[n_tr:n_tr+n_vl]),
            to_loader(all_X[n_tr+n_vl:], all_y[n_tr+n_vl:]))

cwru_tr, cwru_va, cwru_te = load_cwru()

CWRU     tr:7099  val:2366  te:2367  classes:10


In [5]:
# ============================================================
# CELL 5 — Architecture
# ============================================================
class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()
        self.conv = nn.Conv1d(d, d, kernel_size=3, padding=dilation,
                              dilation=dilation, padding_mode='zeros')
        self.bn   = nn.BatchNorm1d(d)
        self.act  = nn.GELU()
    def forward(self, x):
        return x + self.act(self.bn(self.conv(x)))

class CrossSensorGate(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x.mean(dim=1, keepdim=True))

class TinyPrognostics(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3):
        super().__init__()
        self.embed      = nn.Linear(n_sensors, d)
        self.tcn        = nn.Sequential(DilatedBlock(d,1),DilatedBlock(d,2),DilatedBlock(d,4))
        self.gate       = CrossSensorGate(d)
        self.skip       = nn.Linear(n_sensors, d)
        self.gru        = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head   = nn.Sequential(nn.Linear(d,32),nn.GELU(),nn.Dropout(0.1),nn.Linear(32,1))
        self.state_head = nn.Sequential(nn.Linear(d,32),nn.GELU(),nn.Dropout(0.1),nn.Linear(32,n_classes))
    def forward(self, x):
        h = self.tcn(self.embed(x).permute(0,2,1)).permute(0,2,1)
        h = self.gate(h) + self.skip(x)
        _, hT = self.gru(h)
        hT = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

params = sum(p.numel() for p in TinyPrognostics().parameters())
kb     = sum(p.numel()*p.element_size() for p in TinyPrognostics().parameters()) / 1024
print(f"TinyPrognostics: {params:,} params  {kb:.1f} KB")

TinyPrognostics: 12,052 params  47.1 KB


In [6]:
# ============================================================
# CELL 6 — Training & Evaluation Utilities
# ============================================================
def nasa_score(pred, true):
    d = pred - true
    s = np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)
    return float(s.sum()) if np.isfinite(s.sum()) else float('nan')

def _eval_reg(model, loader):
    model.eval()
    ps, ts = [], []
    with torch.no_grad():
        for X, yr in loader:
            ps.append(model(X.to(DEVICE))[0].cpu().numpy())
            ts.append(yr.numpy())
    p = np.concatenate(ps); t = np.concatenate(ts)
    return float(np.sqrt(np.mean((p-t)**2))), float(np.mean(np.abs(p-t))), nasa_score(p,t)

def _eval_cls(model, loader):
    model.eval()
    ps, ts = [], []
    with torch.no_grad():
        for X, ys in loader:
            ps.append(model(X.to(DEVICE))[1].argmax(1).cpu().numpy())
            ts.append(ys.numpy())
    p = np.concatenate(ps); t = np.concatenate(ts)
    return float((p==t).mean()*100)

def train_reg(model, tr, va, te, lr=1e-3, epochs=EPOCHS, patience=PATIENCE):
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    mse   = nn.MSELoss()
    best_val, best_sd, wait = float('inf'), None, 0
    for _ in range(epochs):
        model.train()
        for X, yr in tr:
            X, yr = X.to(DEVICE), yr.to(DEVICE)
            opt.zero_grad()
            mse(model(X)[0], yr).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        v = _eval_reg(model, va)[0]
        if v < best_val:
            best_val = v
            best_sd  = {k: v_.clone() for k, v_ in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_sd)
    return _eval_reg(model, te)

def train_cls(model, tr, va, te, lr=1e-3, epochs=EPOCHS, patience=PATIENCE):
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    ce    = nn.CrossEntropyLoss()
    best_val, best_sd, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train()
        for X, ys in tr:
            X, ys = X.to(DEVICE), ys.to(DEVICE)
            opt.zero_grad()
            ce(model(X)[1], ys).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        v = _eval_cls(model, va)
        if v > best_val:
            best_val = v
            best_sd  = {k: v_.clone() for k, v_ in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_sd)
    return _eval_cls(model, te)

In [7]:
# ============================================================
# CELL 7 — Main Results: TinyPrognostics on C-MAPSS
# ============================================================
results = {}
print(f"{'Dataset':8} {'RMSE':>8} {'MAE':>8} {'NASAScore':>12}")
print("-" * 42)
for fd in [1, 2, 3, 4]:
    torch.manual_seed(SEED)
    model = TinyPrognostics(n_sensors=14, d=24, n_classes=3).to(DEVICE)
    rmse, mae, score = train_reg(model, *cmapss[fd])
    results[f'FD{fd:03d}'] = dict(rmse=rmse, mae=mae, score=score)
    torch.save(model.state_dict(), f'/kaggle/working/tiny_fd{fd:03d}.pt')
    print(f"FD{fd:03d}   {rmse:8.4f} {mae:8.4f} {score:12.2f}")

Dataset      RMSE      MAE    NASAScore
------------------------------------------
FD001    14.7494  11.1443       350.47
FD002    26.7423  18.1006      7405.94
FD003    13.9485  10.4141       315.86
FD004    27.6702  18.6630      8677.78


In [10]:
# ============================================================
# CELL 8 — Battery & CWRU
# ============================================================
torch.manual_seed(SEED)
bat_model = TinyPrognostics(n_sensors=BAT_FEAT, d=24, n_classes=3).to(DEVICE)
rmse, mae, score = train_reg(bat_model, bat_tr, bat_va, bat_te)
results['battery'] = dict(rmse=rmse, mae=mae, score=score)
torch.save(bat_model.state_dict(), '/kaggle/working/tiny_battery.pt')
print(f"Battery  RMSE:{rmse:.4f}  MAE:{mae:.4f}  Score:{score:.4f}")

torch.manual_seed(SEED)
cwru_model = TinyPrognostics(n_sensors=1, d=24, n_classes=N_CWRU_CLS).to(DEVICE)
acc = train_cls(cwru_model, cwru_tr, cwru_va, cwru_te)
results['cwru'] = dict(acc=acc)
torch.save(cwru_model.state_dict(), '/kaggle/working/tiny_cwru.pt')
print(f"CWRU     Acc:{acc:.2f}%  ({N_CWRU_CLS}-class fault classification)")

Battery  RMSE:1.8696  MAE:0.4151  Score:10.6099
CWRU     Acc:99.96%  (10-class fault classification)


In [11]:
# ============================================================
# CELL 9 — Baselines  (Ridge + LSTM-64 + CNN-32)
#           ALL using same EPOCHS/PATIENCE as main — no .pipe() bug
# ============================================================
# Ridge
print("RIDGE BASELINE")
for fd in [1, 2, 3, 4]:
    tr, va, te = cmapss[fd]
    Xtr_np = torch.cat([b[0] for b in tr]).numpy().reshape(-1, SEQ*14)
    ytr_np = torch.cat([b[1] for b in tr]).numpy()
    Xte_np = torch.cat([b[0] for b in te]).numpy().reshape(-1, SEQ*14)
    yte_np = torch.cat([b[1] for b in te]).numpy()
    ridge  = Ridge(alpha=1.0).fit(Xtr_np, ytr_np)
    rmse   = float(np.sqrt(mean_squared_error(yte_np, ridge.predict(Xte_np))))
    results[f'FD{fd:03d}']['ridge_rmse'] = rmse
    print(f"  FD{fd:03d}  {rmse:.4f}")

# Neural baselines — clean implementations, no .pipe()
class LSTMBaseline(nn.Module):
    def __init__(self, n_sensors=14, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, hidden, 2, batch_first=True, dropout=0.2)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(h[-1]).squeeze(-1), None

class CNNBaseline(nn.Module):
    def __init__(self, n_sensors=14, filters=32):
        super().__init__()
        self.net  = nn.Sequential(
            nn.Conv1d(n_sensors, filters,   3, padding=1), nn.ReLU(), nn.BatchNorm1d(filters),
            nn.Conv1d(filters,   filters*2, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(filters*2),
            nn.AdaptiveAvgPool1d(1))
        self.head = nn.Linear(filters*2, 1)
    def forward(self, x):
        h = self.net(x.permute(0,2,1)).squeeze(-1)   # (B, filters*2)
        return self.head(h).squeeze(-1), None         # NO .pipe()

def train_reg_bl(model, tr, va, te):
    """Train a baseline that returns (pred, None)."""
    opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    mse   = nn.MSELoss()
    best_val, best_sd, wait = float('inf'), None, 0
    for _ in range(EPOCHS):
        model.train()
        for X, yr in tr:
            X, yr = X.to(DEVICE), yr.to(DEVICE)
            opt.zero_grad()
            pred, _ = model(X)
            mse(pred, yr).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        # validate
        model.eval()
        ps, ts = [], []
        with torch.no_grad():
            for X, yr in va:
                p, _ = model(X.to(DEVICE))
                ps.append(p.cpu().numpy()); ts.append(yr.numpy())
        v = float(np.sqrt(np.mean((np.concatenate(ps)-np.concatenate(ts))**2)))
        if v < best_val:
            best_val = v
            best_sd  = {k: v_.clone() for k, v_ in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(best_sd)
    model.eval()
    ps, ts = [], []
    with torch.no_grad():
        for X, yr in te:
            p, _ = model(X.to(DEVICE))
            ps.append(p.cpu().numpy()); ts.append(yr.numpy())
    p = np.concatenate(ps); t = np.concatenate(ts)
    return float(np.sqrt(np.mean((p-t)**2))), nasa_score(p,t)

print(f"\n{'Model':16} {'FD001':>8} {'FD002':>8} {'FD003':>8} {'FD004':>8} {'KB':>6}")
for name, Cls, kw in [('LSTM-64', LSTMBaseline, {'hidden':64}),
                      ('CNN-32',  CNNBaseline,  {'filters':32})]:
    row = []
    for fd in [1, 2, 3, 4]:
        torch.manual_seed(SEED)
        m    = Cls(**kw).to(DEVICE)
        rmse, _ = train_reg_bl(m, *cmapss[fd])
        results[f'FD{fd:03d}'][f'{name}_rmse'] = rmse
        row.append(rmse)
    kb = sum(p.numel()*p.element_size() for p in Cls(**kw).parameters())/1024
    print(f"{name:16} {row[0]:8.4f} {row[1]:8.4f} {row[2]:8.4f} {row[3]:8.4f} {kb:6.1f}")

RIDGE BASELINE
  FD001  44.1351
  FD002  54.4415
  FD003  42.4331
  FD004  55.0704

Model               FD001    FD002    FD003    FD004     KB
LSTM-64           16.2756  27.9466  16.0186  27.1470  210.3
CNN-32            26.4671  40.8416  24.7879  39.6203   30.6


In [12]:
# ============================================================
# CELL 10 — Ablation  (SAME EPOCHS/PATIENCE as main)
# ============================================================
class TinyAblation(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3,
                 no_gate=False, no_skip=False, no_dilation=False):
        super().__init__()
        self.no_gate = no_gate; self.no_skip = no_skip
        self.embed   = nn.Linear(n_sensors, d)
        dilations    = [1,1,1] if no_dilation else [1,2,4]
        self.tcn     = nn.Sequential(*[DilatedBlock(d, dil) for dil in dilations])
        if not no_gate: self.gate = CrossSensorGate(d)
        if not no_skip: self.skip = nn.Linear(n_sensors, d)
        self.gru     = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head   = nn.Sequential(nn.Linear(d,32),nn.GELU(),nn.Linear(32,1))
        self.state_head = nn.Sequential(nn.Linear(d,32),nn.GELU(),nn.Linear(32,n_classes))
    def forward(self, x):
        h = self.tcn(self.embed(x).permute(0,2,1)).permute(0,2,1)
        if not self.no_gate: h = self.gate(h)
        if not self.no_skip: h = h + self.skip(x)
        _, hT = self.gru(h)
        hT = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

abl_cfgs = {
    'Full':     dict(no_gate=False, no_skip=False, no_dilation=False),
    'w/o Gate': dict(no_gate=True,  no_skip=False, no_dilation=False),
    'w/o Skip': dict(no_gate=False, no_skip=True,  no_dilation=False),
    'w/o Dil.': dict(no_gate=False, no_skip=False, no_dilation=True),
}

abl = {}
print(f"{'Config':12} {'FD001':>8} {'FD003':>8}  {'Δ FD001':>10} {'Δ FD003':>10}")
print("-"*56)
base1 = base3 = None
for name, cfg in abl_cfgs.items():
    row = {}
    for fd in [1, 3]:
        torch.manual_seed(SEED)
        m = TinyAblation(n_sensors=14, **cfg).to(DEVICE)
        rmse, _, _ = train_reg(m, *cmapss[fd])
        row[fd] = rmse
    abl[name] = row
    if base1 is None: base1, base3 = row[1], row[3]
    print(f"{name:12} {row[1]:8.4f} {row[3]:8.4f}  {row[1]-base1:+10.4f} {row[3]-base3:+10.4f}")

Config          FD001    FD003     Δ FD001    Δ FD003
--------------------------------------------------------
Full          16.2766  12.8954     +0.0000    +0.0000
w/o Gate      18.2704  15.7474     +1.9938    +2.8520
w/o Skip      15.0416  13.2856     -1.2349    +0.3902
w/o Dil.      15.1258  13.9712     -1.1508    +1.0758


In [13]:
# ============================================================
# CELL 11 — Transfer Learning
# ============================================================
fracs = [0.10, 0.25, 0.50, 1.00]
transfer_results = {}

def run_transfer(source_pt, tr_full, va, te,
                 src_sensors, tgt_sensors, frac, freeze_backbone):
    src = TinyPrognostics(n_sensors=src_sensors, d=24, n_classes=3).to(DEVICE)
    src.load_state_dict(torch.load(source_pt, map_location=DEVICE))
    tgt = TinyPrognostics(n_sensors=tgt_sensors, d=24, n_classes=3).to(DEVICE)
    src_sd, tgt_sd = src.state_dict(), tgt.state_dict()
    for k in tgt_sd:
        if k in src_sd and src_sd[k].shape == tgt_sd[k].shape:
            tgt_sd[k] = src_sd[k].clone()
    tgt.load_state_dict(tgt_sd)
    if freeze_backbone:
        for p in tgt.tcn.parameters(): p.requires_grad = False
        for p in tgt.gru.parameters(): p.requires_grad = False
    if frac < 1.0:
        ds     = tr_full.dataset
        n_keep = max(64, int(len(ds)*frac))
        idx    = torch.randperm(len(ds), generator=torch.Generator().manual_seed(SEED))[:n_keep]
        tr     = DataLoader(Subset(ds, idx), batch_size=BATCH,
                            shuffle=True, num_workers=2, pin_memory=True)
    else:
        tr = tr_full
    rmse, mae, score = train_reg(tgt, tr, va, te, lr=5e-4)
    return rmse

# FD001 → FD003 (same domain, freeze backbone)
scratch_fd3 = results['FD003']['rmse']
transfer_results['FD003'] = {}
print(f"FD001 → FD003  (scratch={scratch_fd3:.4f})")
print(f"{'Frac':>6} {'RMSE':>8} {'Δ vs Scratch':>14}")
tr3, va3, te3 = cmapss[3]
for frac in fracs:
    rmse = run_transfer('/kaggle/working/tiny_fd001.pt',
                        tr3, va3, te3, 14, 14, frac, freeze_backbone=True)
    transfer_results['FD003'][frac] = rmse
    print(f"{frac*100:5.0f}%  {rmse:8.4f}  {rmse-scratch_fd3:+14.4f}")

# FD001 → Battery (cross domain, full fine-tune)
scratch_bat = results['battery']['rmse']
transfer_results['Battery'] = {}
print(f"\nFD001 → Battery  (scratch={scratch_bat:.4f})")
print(f"{'Frac':>6} {'RMSE':>8} {'Δ vs Scratch':>14}")
for frac in fracs:
    rmse = run_transfer('/kaggle/working/tiny_fd001.pt',
                        bat_tr, bat_va, bat_te, 14, 4, frac, freeze_backbone=False)
    transfer_results['Battery'][frac] = rmse
    print(f"{frac*100:5.0f}%  {rmse:8.4f}  {rmse-scratch_bat:+14.4f}")

FD001 → FD003  (scratch=13.9485)
  Frac     RMSE   Δ vs Scratch
   10%   15.0757         +1.1272
   25%   15.3461         +1.3975
   50%   14.6416         +0.6930
  100%   15.3139         +1.3654

FD001 → Battery  (scratch=1.8696)
  Frac     RMSE   Δ vs Scratch
   10%    3.8407         +1.9711
   25%    3.8332         +1.9635
   50%    2.4829         +0.6133
  100%    3.7491         +1.8795


In [14]:
# ============================================================
# CELL 12 — Master Results Summary
# ============================================================
print("\n" + "="*72)
print("TINYPROGNOSTICS — FINAL RESULTS")
print("="*72)

tiny_kb = sum(p.numel()*p.element_size() for p in TinyPrognostics().parameters())/1024
lstm_kb = sum(p.numel()*p.element_size() for p in LSTMBaseline().parameters())/1024
cnn_kb  = sum(p.numel()*p.element_size() for p in CNNBaseline().parameters())/1024
print(f"Sizes:  Tiny={tiny_kb:.1f}KB  LSTM-64={lstm_kb:.1f}KB  CNN-32={cnn_kb:.1f}KB\n")

print(f"{'Dataset':8} {'Tiny':>8} {'Score':>10} {'LSTM':>8} {'CNN':>8} {'Ridge':>8}")
print("-"*58)
for fd in [1,2,3,4]:
    k = f'FD{fd:03d}'
    r = results[k]
    print(f"{k:8} {r['rmse']:8.4f} {r['score']:10.2f} "
          f"{r.get('LSTM-64_rmse',float('nan')):8.4f} "
          f"{r.get('CNN-32_rmse',float('nan')):8.4f} "
          f"{r.get('ridge_rmse',float('nan')):8.4f}")

r = results['battery']
print(f"{'Battery':8} {r['rmse']:8.4f}  MAE:{r['mae']:.4f}")
print(f"{'CWRU':8} Acc:{results['cwru']['acc']:.2f}%  ({N_CWRU_CLS}-class)")

print(f"\nABLATION")
print(f"{'Config':12} {'FD001':>8} {'FD003':>8} {'Δ FD001':>10} {'Δ FD003':>10}")
b1, b3 = abl['Full'][1], abl['Full'][3]
for name, row in abl.items():
    print(f"{name:12} {row[1]:8.4f} {row[3]:8.4f} {row[1]-b1:+10.4f} {row[3]-b3:+10.4f}")

print(f"\nTRANSFER LEARNING")
print(f"{'Target':10} {'10%':>8} {'25%':>8} {'50%':>8} {'100%':>8} {'Scratch':>8}")
for tgt in ['FD003','Battery']:
    sc = results['FD003']['rmse'] if tgt=='FD003' else results['battery']['rmse']
    v  = [transfer_results[tgt].get(f, float('nan')) for f in fracs]
    print(f"{tgt:10} {v[0]:8.4f} {v[1]:8.4f} {v[2]:8.4f} {v[3]:8.4f} {sc:8.4f}")

print("="*72)


TINYPROGNOSTICS — FINAL RESULTS
Sizes:  Tiny=47.1KB  LSTM-64=210.3KB  CNN-32=30.6KB

Dataset      Tiny      Score     LSTM      CNN    Ridge
----------------------------------------------------------
FD001     14.7494     350.47  16.2756  26.4671  44.1351
FD002     26.7423    7405.94  27.9466  40.8416  54.4415
FD003     13.9485     315.86  16.0186  24.7879  42.4331
FD004     27.6702    8677.78  27.1470  39.6203  55.0704
Battery    1.8696  MAE:0.4151
CWRU     Acc:99.96%  (10-class)

ABLATION
Config          FD001    FD003    Δ FD001    Δ FD003
Full          16.2766  12.8954    +0.0000    +0.0000
w/o Gate      18.2704  15.7474    +1.9938    +2.8520
w/o Skip      15.0416  13.2856    -1.2349    +0.3902
w/o Dil.      15.1258  13.9712    -1.1508    +1.0758

TRANSFER LEARNING
Target          10%      25%      50%     100%  Scratch
FD003       15.0757  15.3461  14.6416  15.3139  13.9485
Battery      3.8407   3.8332   2.4829   3.7491   1.8696


In [15]:
# ============================================================
# ADDITION 1 — Inference Latency Benchmark (30 min)
# ============================================================
import time

def benchmark_latency(model, n_sensors, seq_len=64, n_runs=1000):
    model.eval()
    x = torch.randn(1, seq_len, n_sensors)
    # warmup
    for _ in range(50):
        _ = model(x)
    t0 = time.perf_counter()
    for _ in range(n_runs):
        with torch.no_grad():
            _ = model(x)
    elapsed = (time.perf_counter() - t0) / n_runs * 1000  # ms per sample
    return elapsed

models_to_bench = {
    'TinyPrognostics': TinyPrognostics(n_sensors=14),
    'LSTM-64':         LSTMBaseline(n_sensors=14, hidden=64),
    'CNN-32':          CNNBaseline(n_sensors=14, filters=32),
}
print(f"\n{'Model':18} {'Size(KB)':>10} {'Latency(ms)':>13}")
print("-"*44)
for name, m in models_to_bench.items():
    kb  = sum(p.numel()*p.element_size() for p in m.parameters())/1024
    lat = benchmark_latency(m, 14)
    print(f"{name:18} {kb:10.1f} {lat:13.3f}")


Model                Size(KB)   Latency(ms)
--------------------------------------------
TinyPrognostics          47.1         1.800
LSTM-64                 210.3         1.048
CNN-32                   30.6         0.206


In [16]:
# ============================================================
# ADDITION 2 — Critical-Zone RMSE (RUL ≤ 30) (30 min)
# ============================================================
def evaluate_with_zones(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, yr in loader:
            preds.extend(model(X.to(DEVICE))[0].cpu().numpy())
            trues.extend(yr.numpy())
    p, t = np.array(preds), np.array(trues)
    rmse_all   = float(np.sqrt(np.mean((p-t)**2)))
    # Critical zone: last 30 cycles before failure — matters most for maintenance
    mask_crit  = t <= 30
    rmse_crit  = float(np.sqrt(np.mean((p[mask_crit]-t[mask_crit])**2))) if mask_crit.sum()>0 else float('nan')
    mask_early = t > 30
    rmse_early = float(np.sqrt(np.mean((p[mask_early]-t[mask_early])**2))) if mask_early.sum()>0 else float('nan')
    return rmse_all, rmse_crit, rmse_early, nasa_score(p,t)

# Replace your main loop in Cell 7 with:
print(f"{'Dataset':8} {'RMSE':>8} {'RMSE_crit':>10} {'RMSE_early':>11} {'NASAScore':>11}")
print("-"*54)
for fd in [1, 2, 3, 4]:
    torch.manual_seed(SEED)
    model = TinyPrognostics(n_sensors=14, d=24, n_classes=3).to(DEVICE)
    rmse, mae, score = train_reg(model, *cmapss[fd])
    # Re-evaluate with zones using test loader
    _, te = cmapss[fd][1], cmapss[fd][2]
    rmse_a, rmse_c, rmse_e, sc = evaluate_with_zones(model, cmapss[fd][2])
    results[f'FD{fd:03d}'].update(rmse_crit=rmse_c, rmse_early=rmse_e)
    torch.save(model.state_dict(), f'/kaggle/working/tiny_fd{fd:03d}.pt')
    print(f"FD{fd:03d}   {rmse_a:8.4f} {rmse_c:10.4f} {rmse_e:11.4f} {sc:11.2f}")

Dataset      RMSE  RMSE_crit  RMSE_early   NASAScore
------------------------------------------------------
FD001    14.7494     4.6302     16.8201      350.47
FD002    26.7423     5.5173     30.4318     7405.94
FD003    13.9485     3.1427     15.5156      315.86
FD004    27.6702     6.6224     31.0132     8677.78


In [17]:
# ============================================================
# ADDITION 3 — GRU-32 Baseline (same size class) (1 hour)
# ============================================================
class GRUBaseline(nn.Module):
    """Fair same-size-class comparison: ~35KB, simple GRU, no architecture innovations."""
    def __init__(self, n_sensors=14, hidden=24):
        super().__init__()
        self.gru  = nn.GRU(n_sensors, hidden, 2, batch_first=True, dropout=0.1)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        _, h = self.gru(x)
        return self.head(h[-1]).squeeze(-1), None

# Add to your baseline loop:
for name, Cls, kw in [('LSTM-64',  LSTMBaseline, {'hidden':64}),
                      ('GRU-32',   GRUBaseline,  {'hidden':24}),   # ← NEW
                      ('CNN-32',   CNNBaseline,  {'filters':32})]:
    ...

In [20]:
# ============================================================
# FINAL CELL — Critical-Zone RMSE for ALL baselines
# This is your headline result. Run this once.
# ============================================================

def eval_zones(model, loader, device='cpu'):
    """Compute RMSE overall and in critical zone (RUL ≤ 30)."""
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, yr in loader:
            preds.append(model(X.to(device))[0].cpu().numpy())
            trues.append(yr.numpy())
    p = np.concatenate(preds)
    t = np.concatenate(trues)
    rmse_all  = float(np.sqrt(np.mean((p - t)**2)))
    mask      = t <= 30
    rmse_crit = float(np.sqrt(np.mean((p[mask]-t[mask])**2))) if mask.sum() > 0 else float('nan')
    return rmse_all, rmse_crit, int(mask.sum())

# Train fresh baselines with eval_zones — reuse your already-trained weights
# for TinyProg (already saved), retrain LSTM and CNN just for test evaluation

print(f"{'Model':20} {'FD001':>8} {'crit':>7} {'FD002':>8} {'crit':>7} "
      f"{'FD003':>8} {'crit':>7} {'FD004':>8} {'crit':>7}")
print("-"*80)

# TinyProg — load saved weights
row_tiny = {'all':[], 'crit':[]}
for fd in [1,2,3,4]:
    m = TinyPrognostics(n_sensors=14, d=24, n_classes=3).to(DEVICE)
    m.load_state_dict(torch.load(f'/kaggle/working/tiny_fd{fd:03d}.pt', map_location=DEVICE))
    _, te = cmapss[fd][1], cmapss[fd][2]
    ra, rc, n = eval_zones(m, cmapss[fd][2], DEVICE)
    row_tiny['all'].append(ra); row_tiny['crit'].append(rc)
vals = ''.join([f"{row_tiny['all'][i]:8.4f} {row_tiny['crit'][i]:7.4f} " for i in range(4)])
print(f"{'TinyPrognostics':20} {vals}")

# LSTM-64 and CNN-32 — retrain for each FD, evaluate zones
for name, Cls, kw in [('LSTM-64', LSTMBaseline, {'hidden':64}),
                      ('CNN-32',  CNNBaseline,  {'filters':32})]:
    row = {'all':[], 'crit':[]}
    for fd in [1,2,3,4]:
        torch.manual_seed(SEED)
        m = Cls(**kw).to(DEVICE)
        train_reg_bl(m, *cmapss[fd])
        ra, rc, n = eval_zones(m, cmapss[fd][2], DEVICE)
        row['all'].append(ra); row['crit'].append(rc)
    vals = ''.join([f"{row['all'][i]:8.4f} {row['crit'][i]:7.4f} " for i in range(4)])
    print(f"{name:20} {vals}")

Model                   FD001    crit    FD002    crit    FD003    crit    FD004    crit
--------------------------------------------------------------------------------
TinyPrognostics       14.7494  4.6302  26.7423  5.5173  13.9485  3.1427  27.6702  6.6224 
LSTM-64               16.2756  5.2707  27.9466  4.4167  16.0186  2.6583  27.1470  5.9448 
CNN-32                26.4671 29.0510  40.8416 20.6667  24.7879 21.5038  39.6203 28.1839 


In [2]:
pip install torch numpy pandas scipy

Note: you may need to restart the kernel to use updated packages.


In [4]:
# ── STEP 1: Find exact paths (run this cell first to confirm) ──────────────
import os

def find_file(name, search_root="/kaggle/input"):
    for root, dirs, files in os.walk(search_root):
        if name in files:
            return root
    return None

# Check C-MAPSS
cmapss = find_file("train_FD001.txt")
print("C-MAPSS dir:", cmapss)

# Check Battery
battery = find_file("metadata.csv")
print("Battery dir:", battery)

# Check CWRU
cwru = find_file("97.mat")
print("CWRU dir:", cwru)

C-MAPSS dir: /kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps
Battery dir: /kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset
CWRU dir: /kaggle/input/datasets/sufian79/cwru-mat-full-dataset


In [6]:
# ══════════════════════════════════════════════════════════════════
# NanoSentry — 5-Seed Evaluation  (Kaggle version)
# ══════════════════════════════════════════════════════════════════
CMAPSS_DIR  = "/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps"
BATTERY_DIR = "/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset"
CWRU_DIR    = "/kaggle/input/datasets/sufian79/cwru-mat-full-dataset"

SEEDS = [42, 7, 13, 99, 2025]
BATCH = 128
SEQ   = 64
SENSORS = [6,7,8,11,12,13,15,16,17,18,19,21,24,25]

import os, csv, warnings
import numpy as np
import pandas as pd
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── MODEL ──────────────────────────────────────────────────────────
class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()
        self.conv = nn.Conv1d(d, d, kernel_size=3, padding=dilation,
                              dilation=dilation, padding_mode="zeros")
        self.bn  = nn.BatchNorm1d(d)
        self.act = nn.GELU()
    def forward(self, x):
        return x + self.act(self.bn(self.conv(x)))

class CrossSensorGate(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x.mean(dim=1, keepdim=True))

class NanoSentry(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3):
        super().__init__()
        self.embed = nn.Linear(n_sensors, d)
        self.tcn   = nn.Sequential(DilatedBlock(d,1), DilatedBlock(d,2), DilatedBlock(d,4))
        self.gate  = CrossSensorGate(d)
        self.skip  = nn.Linear(n_sensors, d)
        self.gru   = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head   = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32,1))
        self.state_head = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32,n_classes))
    def forward(self, x):
        h = self.embed(x).permute(0,2,1)
        h = self.tcn(h).permute(0,2,1)
        h = self.gate(h) + self.skip(x)
        _, hT = self.gru(h)
        hT = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

# ── DATA LOADERS ───────────────────────────────────────────────────
def load_cmapss(fd, seed):
    train_df = pd.read_csv(f"{CMAPSS_DIR}/train_FD{fd:03d}.txt", sep=r'\s+', header=None)
    test_df  = pd.read_csv(f"{CMAPSS_DIR}/test_FD{fd:03d}.txt",  sep=r'\s+', header=None)
    rul_df   = pd.read_csv(f"{CMAPSS_DIR}/RUL_FD{fd:03d}.txt",   header=None, names=["RUL"])
    max_cycles = train_df.groupby(0)[1].max().rename("max_cycle")
    train_df   = train_df.join(max_cycles, on=0)
    train_df["RUL"]   = (train_df["max_cycle"] - train_df[1]).clip(upper=125)
    train_df["state"] = pd.cut(train_df["RUL"], bins=[-1,25,50,200], labels=[0,1,2]).astype(int)
    feat_mean = train_df[SENSORS].mean()
    feat_std  = train_df[SENSORS].std().replace(0, 1)

    def make_windows(df, eng_list):
        Xs, yr, ys = [], [], []
        for eng in eng_list:
            sub   = df[df[0]==eng].reset_index(drop=True)
            feats = ((sub[SENSORS] - feat_mean) / feat_std).values.astype(np.float32)
            n = len(feats)
            if n < SEQ:
                feats = np.vstack([np.zeros((SEQ-n,14),dtype=np.float32), feats])
                n = SEQ
            for t in range(n-SEQ+1):
                Xs.append(feats[t:t+SEQ])
                yr.append(float(sub.loc[min(t+SEQ-1,len(sub)-1),"RUL"]))
                ys.append(int(sub.loc[min(t+SEQ-1,len(sub)-1),"state"]))
        return torch.tensor(np.array(Xs)), torch.tensor(yr,dtype=torch.float32), torch.tensor(ys,dtype=torch.long)

    rng = np.random.RandomState(seed)
    engines = train_df[0].unique()
    idx = rng.permutation(len(engines))
    n_tr = int(len(engines)*0.8)
    Xtr,ytr_r,ytr_s = make_windows(train_df, engines[idx[:n_tr]])
    Xvl,yvl_r,yvl_s = make_windows(train_df, engines[idx[n_tr:]])

    rul_test = rul_df["RUL"].values.astype(np.float32)
    Xte_list,yte_r_list,yte_s_list = [],[],[]
    for i, eng in enumerate(test_df[0].unique()):
        sub   = test_df[test_df[0]==eng].reset_index(drop=True)
        feats = ((sub[SENSORS] - feat_mean) / feat_std).values.astype(np.float32)
        n = len(feats)
        if n < SEQ:
            feats = np.vstack([np.zeros((SEQ-n,14),dtype=np.float32), feats])
        Xte_list.append(feats[-SEQ:])
        rv = rul_test[i]
        yte_r_list.append(float(rv))
        yte_s_list.append(0 if rv>50 else (1 if rv>25 else 2))

    def ml(X,yr,ys,sh=False):
        return DataLoader(TensorDataset(X,yr,ys), batch_size=BATCH, shuffle=sh, num_workers=2, pin_memory=True)
    Xte  = torch.tensor(np.array(Xte_list))
    yte_r = torch.tensor(yte_r_list,dtype=torch.float32)
    yte_s = torch.tensor(yte_s_list,dtype=torch.long)
    return ml(Xtr,ytr_r,ytr_s,True), ml(Xvl,yvl_r,yvl_s), ml(Xte,yte_r,yte_s)


def load_battery(seed):
    meta      = pd.read_csv(os.path.join(BATTERY_DIR,"metadata.csv"))
    discharge = meta[meta["type"]=="discharge"].copy()
    rng = np.random.RandomState(seed)
    all_seqs  = []
    for bid in discharge["battery_id"].unique():
        rows = discharge[discharge["battery_id"]==bid].sort_values("start_time").reset_index(drop=True)
        cap_vals = []
        for _,r in rows.iterrows():
            try: cap_vals.append(float(r["Capacity"]))
            except: continue
        n = len(cap_vals)
        if n < SEQ+5: continue
        cap_arr = np.array(cap_vals,dtype=np.float32)
        max_cap = cap_arr[0] if cap_arr[0]>0 else cap_arr.max()
        soh     = np.clip(cap_arr/max_cap, 0.0, 1.0)
        delta   = np.diff(soh, prepend=soh[0])
        cyc_norm= np.arange(n,dtype=np.float32)/n
        feats   = np.stack([soh, delta, cyc_norm, cap_arr/(max_cap+1e-8)], axis=1)
        eol_idx = int(np.where(soh<0.8)[0][0]) if np.any(soh<0.8) else n
        rul     = np.clip((eol_idx-np.arange(n)).astype(np.float32),0,200)
        state   = (rul<=50).astype(np.int64)+(rul<=20).astype(np.int64)
        for t in range(n-SEQ+1):
            all_seqs.append((feats[t:t+SEQ], rul[t+SEQ-1], state[t+SEQ-1]))
    idx = rng.permutation(len(all_seqs))
    all_seqs = [all_seqs[i] for i in idx]
    n_total=len(all_seqs); n_tr=int(n_total*0.6); n_vl=int(n_total*0.2)
    def to_loader(seqs, sh=False):
        X  = torch.tensor(np.array([s[0] for s in seqs]))
        yr = torch.tensor([s[1] for s in seqs],dtype=torch.float32)
        ys = torch.tensor([s[2] for s in seqs],dtype=torch.long)
        return DataLoader(TensorDataset(X,yr,ys), batch_size=BATCH, shuffle=sh, num_workers=2, pin_memory=True)
    return to_loader(all_seqs[:n_tr],True), to_loader(all_seqs[n_tr:n_tr+n_vl]), to_loader(all_seqs[n_tr+n_vl:]), 4


CWRU_FILES = {
    "97.mat":0,"98.mat":0,"99.mat":0,"100.mat":0,
    "105.mat":1,"106.mat":1,"107.mat":1,"108.mat":1,
    "169.mat":2,"170.mat":2,"171.mat":2,"172.mat":2,
    "209.mat":3,"210.mat":3,"211.mat":3,"212.mat":3,
    "118.mat":4,"119.mat":4,"120.mat":4,"121.mat":4,
    "185.mat":5,"186.mat":5,"187.mat":5,"188.mat":5,
    "222.mat":6,"223.mat":6,"224.mat":6,"225.mat":6,
    "130.mat":7,"131.mat":7,"132.mat":7,"133.mat":7,
    "197.mat":8,"198.mat":8,"199.mat":8,"200.mat":8,
    "234.mat":9,"235.mat":9,"236.mat":9,"237.mat":9,
}

def load_cwru(seed):
    CWRU_SEQ = 1024
    all_X, all_y = [], []
    for fname, cls_id in CWRU_FILES.items():
        # search recursively in case files are in a subfolder
        fpath = None
        for root, dirs, files in os.walk(CWRU_DIR):
            if fname in files:
                fpath = os.path.join(root, fname)
                break
        if fpath is None: continue
        mat = scipy.io.loadmat(fpath)
        key = [k for k in mat.keys() if "DE_time" in k]
        if not key: continue
        sig = mat[key[0]].flatten().astype(np.float32)
        sig = (sig - sig.mean()) / (sig.std() + 1e-8)
        step = CWRU_SEQ // 2
        for start in range(0, len(sig)-CWRU_SEQ+1, step):
            all_X.append(sig[start:start+CWRU_SEQ].reshape(CWRU_SEQ,1))
            all_y.append(cls_id)
    idx   = np.random.RandomState(seed).permutation(len(all_X))
    all_X = np.array(all_X)[idx]; all_y = np.array(all_y)[idx]
    n_total=len(all_X); n_tr=int(n_total*0.6); n_vl=int(n_total*0.2)
    def to_loader(X,y,sh=False):
        return DataLoader(TensorDataset(torch.tensor(X),
            torch.zeros(len(y),dtype=torch.float32),
            torch.tensor(y,dtype=torch.long)),
            batch_size=BATCH, shuffle=sh, num_workers=2, pin_memory=True)
    return to_loader(all_X[:n_tr],all_y[:n_tr],True), to_loader(all_X[n_tr:n_tr+n_vl],all_y[n_tr:n_tr+n_vl]), to_loader(all_X[n_tr+n_vl:],all_y[n_tr+n_vl:])

# ── TRAINING ───────────────────────────────────────────────────────
def nasa_score(pred, true):
    d = pred - true
    s = np.where(d<0, np.exp(-d/13)-1, np.exp(d/10)-1)
    return float(s.sum()) if np.isfinite(s.sum()) else float("nan")

def critical_zone_rmse(pred, true, threshold=30):
    mask = true <= threshold
    if mask.sum()==0: return float("nan")
    return float(np.sqrt(np.mean((pred[mask]-true[mask])**2)))

def train_epoch(model, loader, optimizer, is_cwru=False):
    model.train()
    mse_fn=nn.MSELoss(); ce_fn=nn.CrossEntropyLoss()
    for X,yr,ys in loader:
        X,yr,ys = X.to(DEVICE),yr.to(DEVICE),ys.to(DEVICE)
        optimizer.zero_grad()
        pred_r,pred_s = model(X)
        loss = ce_fn(pred_s,ys) if is_cwru else mse_fn(pred_r,yr)+0.15*ce_fn(pred_s,ys)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()

def evaluate(model, loader, is_cwru=False):
    model.eval()
    preds_r,trues_r,preds_s,trues_s=[],[],[],[]
    with torch.no_grad():
        for X,yr,ys in loader:
            pred_r,pred_s = model(X.to(DEVICE))
            preds_r.extend(pred_r.cpu().numpy())
            trues_r.extend(yr.numpy())
            preds_s.extend(pred_s.argmax(1).cpu().numpy())
            trues_s.extend(ys.numpy())
    pr,tr_=np.array(preds_r),np.array(trues_r)
    ps,ts=np.array(preds_s),np.array(trues_s)
    return {"rmse":float(np.sqrt(np.mean((pr-tr_)**2))),
            "mae":float(np.mean(np.abs(pr-tr_))),
            "acc":float((ps==ts).mean()*100),
            "score":nasa_score(pr,tr_),
            "cz_rmse":critical_zone_rmse(pr,tr_)}

def run_one(model, tr, va, te, epochs=100, lr=1e-3, is_cwru=False, patience=20):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    best_val,best_state,wait = float("inf"),None,0
    for epoch in range(1, epochs+1):
        train_epoch(model, tr, optimizer, is_cwru=is_cwru)
        m = evaluate(model, va, is_cwru=is_cwru)
        val_metric = -m["acc"] if is_cwru else m["rmse"]
        scheduler.step()
        if val_metric < best_val:
            best_val=val_metric
            best_state={k:v.clone() for k,v in model.state_dict().items()}
            wait=0
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(best_state)
    return evaluate(model, te, is_cwru=is_cwru)

# ── MAIN LOOP ──────────────────────────────────────────────────────
TASKS = [
    ("FD001","cmapss",1), ("FD002","cmapss",2),
    ("FD003","cmapss",3), ("FD004","cmapss",4),
    ("Battery","battery",None), ("CWRU","cwru",None),
]

all_rows = []

for task_name, task_type, fd in TASKS:
    print(f"\n{'='*55}\nTASK: {task_name}\n{'='*55}")
    task_results = []

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)
        try:
            if task_type=="cmapss":
                tr,va,te = load_cmapss(fd, seed)
                model = NanoSentry(14,24,3).to(DEVICE); is_cwru=False
            elif task_type=="battery":
                tr,va,te,nf = load_battery(seed)
                model = NanoSentry(nf,24,3).to(DEVICE); is_cwru=False
            else:
                tr,va,te = load_cwru(seed)
                model = NanoSentry(1,24,10).to(DEVICE); is_cwru=True
        except Exception as e:
            print(f"  [ERROR] Seed {seed}: {e}"); continue

        print(f"  Seed {seed} | params={sum(p.numel() for p in model.parameters()):,}", end=" | ", flush=True)
        m = run_one(model, tr, va, te, epochs=100, lr=1e-3, is_cwru=is_cwru, patience=20)
        print(f"RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  Acc={m['acc']:.2f}%  Score={m['score']:.1f}  CZ={m['cz_rmse']:.4f}")
        all_rows.append({"task":task_name,"seed":seed,**m})
        task_results.append(m)

    if len(task_results)>1:
        for key in ["rmse","mae","acc","score","cz_rmse"]:
            vals=[r[key] for r in task_results if not np.isnan(r.get(key,float("nan")))]
            if vals: print(f"  >>> {key:8s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

# ── SAVE ───────────────────────────────────────────────────────────
if all_rows:
    import csv
    with open("/kaggle/working/results_five_seeds.csv","w",newline="") as f:
        w=csv.DictWriter(f,fieldnames=["task","seed","rmse","mae","acc","score","cz_rmse"])
        w.writeheader(); w.writerows(all_rows)

    print("\n" + "="*65)
    print("SUMMARY (mean ± std across 5 seeds)")
    print("="*65)
    print(f"{'Task':<12} {'RMSE':>14} {'MAE':>12} {'CZ-RMSE':>12} {'Acc%':>10} {'Score':>12}")
    print("-"*65)

    tasks_seen = list(dict.fromkeys(r["task"] for r in all_rows))
    with open("/kaggle/working/results_summary.csv","w",newline="") as f:
        w=csv.writer(f)
        w.writerow(["Task","RMSE_mean","RMSE_std","MAE_mean","MAE_std","CZ_RMSE_mean","CZ_RMSE_std","Acc_mean","Acc_std","Score_mean","Score_std"])
        for tname in tasks_seen:
            rows=[r for r in all_rows if r["task"]==tname]
            def ms(key):
                vals=[r[key] for r in rows if not np.isnan(r.get(key,float("nan")))]
                return (np.mean(vals),np.std(vals)) if vals else (float("nan"),float("nan"))
            r_m,r_s=ms("rmse"); a_m,a_s=ms("mae"); c_m,c_s=ms("cz_rmse"); ac_m,ac_s=ms("acc"); sc_m,sc_s=ms("score")
            w.writerow([tname,f"{r_m:.4f}",f"{r_s:.4f}",f"{a_m:.4f}",f"{a_s:.4f}",f"{c_m:.4f}",f"{c_s:.4f}",f"{ac_m:.2f}",f"{ac_s:.2f}",f"{sc_m:.1f}",f"{sc_s:.1f}"])
            print(f"{tname:<12} {r_m:.2f}±{r_s:.2f}   {a_m:.2f}±{a_s:.2f}   {c_m:.2f}±{c_s:.2f}   {ac_m:.2f}±{ac_s:.2f}   {sc_m:.0f}±{sc_s:.0f}")

    print("\n✓ Saved to /kaggle/working/results_five_seeds.csv")
    print("✓ Saved to /kaggle/working/results_summary.csv")

Device: cpu

TASK: FD001
  Seed 42 | params=12,052 | RMSE=14.4232  MAE=10.5573  Acc=8.00%  Score=270.8  CZ=4.6194
  Seed 7 | params=12,052 | RMSE=15.7367  MAE=11.6950  Acc=11.00%  Score=434.2  CZ=4.2869
  Seed 13 | params=12,052 | RMSE=14.5464  MAE=10.7368  Acc=11.00%  Score=323.5  CZ=3.5108
  Seed 99 | params=12,052 | RMSE=15.5307  MAE=11.0300  Acc=7.00%  Score=355.0  CZ=3.4392
  Seed 2025 | params=12,052 | RMSE=13.4111  MAE=9.7841  Acc=9.00%  Score=264.7  CZ=3.2721
  >>> rmse    : 14.7296 ± 0.8393
  >>> mae     : 10.7606 ± 0.6231
  >>> acc     : 9.2000 ± 1.6000
  >>> score   : 329.6505 ± 62.1156
  >>> cz_rmse : 3.8257 ± 0.5287

TASK: FD002
  Seed 42 | params=12,052 | RMSE=25.5412  MAE=17.3382  Acc=8.11%  Score=5276.9  CZ=4.4273
  Seed 7 | params=12,052 | RMSE=26.6585  MAE=17.8490  Acc=8.11%  Score=7586.2  CZ=4.4522
  Seed 13 | params=12,052 | RMSE=28.6124  MAE=19.6352  Acc=8.11%  Score=9475.0  CZ=5.9489
  Seed 99 | params=12,052 | RMSE=24.8992  MAE=17.1361  Acc=8.49%  Score=4668.7  C

In [1]:
import os, csv, time, warnings, scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {DEVICE}")

# ==========================================================
# 1. CHECKPOINT SYSTEM (Disconnect-Proof)
# ==========================================================
RESULTS_FILE = '/kaggle/working/master_results.csv'
FIELDNAMES = ['dataset', 'subset', 'model', 'variant', 'seed', 'rmse', 'mae', 'score', 'cz_rmse', 'acc', 'params', 'kb']

def is_done(dataset, subset, model, variant, seed):
    if not os.path.exists(RESULTS_FILE): return False
    df = pd.read_csv(RESULTS_FILE)
    return len(df[(df['dataset']==dataset) & (df['subset']==subset) & 
                  (df['model']==model) & (df['variant']==variant) & (df['seed']==seed)]) > 0

def save_result(row):
    file_exists = os.path.exists(RESULTS_FILE)
    with open(RESULTS_FILE, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists: writer.writeheader()
        writer.writerow(row)
    print(f"✅ Saved: {row['dataset']} | {row['subset']} | {row['model']} | {row['variant']} | Seed {row['seed']}")

# ==========================================================
# 2. LEAKAGE-FREE DATA LOADERS
# ==========================================================
CMAPSS_DIR = "/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps"
BATTERY_DIR = "/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset"
CWRU_DIR = "/kaggle/input/datasets/sufian79/cwru-mat-full-dataset"

SENSORS = [6,7,8,11,12,13,15,16,17,18,19,21,24,25]
SEQ = 64
BATCH = 256

def load_cmapss(fd, seed):
    train_df = pd.read_csv(f"{CMAPSS_DIR}/train_FD{fd:03d}.txt", sep=r'\s+', header=None)
    test_df  = pd.read_csv(f"{CMAPSS_DIR}/test_FD{fd:03d}.txt",  sep=r'\s+', header=None)
    rul_df   = pd.read_csv(f"{CMAPSS_DIR}/RUL_FD{fd:03d}.txt",   header=None, names=["RUL"])
    
    max_cycles = train_df.groupby(0)[1].max().rename("max_cycle")
    train_df = train_df.join(max_cycles, on=0)
    train_df["RUL"] = (train_df["max_cycle"] - train_df[1]).clip(upper=125)
    
    # Condition-aware normalization for FD002/004
    if fd in [1, 3]:
        mu, std = train_df[SENSORS].mean(), train_df[SENSORS].std().replace(0, 1)
        train_df[SENSORS] = (train_df[SENSORS] - mu) / std
        test_df[SENSORS] = (test_df[SENSORS] - mu) / std
    else:
        km = KMeans(n_clusters=6, random_state=seed, n_init=10)
        km.fit(train_df[[2,3,4]].values)
        tr_lbls, te_lbls = km.predict(train_df[[2,3,4]].values), km.predict(test_df[[2,3,4]].values)
        for c in range(6):
            mask = tr_lbls == c
            if mask.sum() < 2: continue
            mu, std = train_df.loc[mask, SENSORS].mean(), train_df.loc[mask, SENSORS].std().replace(0, 1)
            train_df.loc[mask, SENSORS] = (train_df.loc[mask, SENSORS] - mu) / std
            te_mask = te_lbls == c
            if te_mask.sum() > 0: test_df.loc[te_mask, SENSORS] = (test_df.loc[te_mask, SENSORS] - mu) / std

    def make_windows(df, eng_list):
        Xs, yr, ys = [], [], []
        for eng in eng_list:
            sub = df[df[0]==eng].reset_index(drop=True)
            feats = sub[SENSORS].values.astype(np.float32)
            if len(feats) < SEQ: feats = np.vstack([np.zeros((SEQ-len(feats),14),dtype=np.float32), feats])
            for t in range(len(feats) - SEQ + 1):
                Xs.append(feats[t:t+SEQ])
                rul_val = sub.loc[min(t+SEQ-1, len(sub)-1), "RUL"]
                yr.append(float(rul_val))
                ys.append(0 if rul_val>50 else (1 if rul_val>25 else 2))
        return torch.tensor(np.array(Xs)), torch.tensor(yr,dtype=torch.float32), torch.tensor(ys,dtype=torch.long)

    rng = np.random.RandomState(seed)
    engines = train_df[0].unique()
    idx = rng.permutation(len(engines))
    n_tr = int(len(engines)*0.8)
    Xtr, ytr_r, ytr_s = make_windows(train_df, engines[idx[:n_tr]])
    Xvl, yvl_r, yvl_s = make_windows(train_df, engines[idx[n_tr:]])
    
    test_engines = test_df[0].unique()
    Xte_list, yte_r_list, yte_s_list = [], [], []
    for i, eng in enumerate(test_engines):
        sub = test_df[test_df[0]==eng].reset_index(drop=True)
        feats = sub[SENSORS].values.astype(np.float32)
        if len(feats) < SEQ: feats = np.vstack([np.zeros((SEQ-len(feats),14),dtype=np.float32), feats])
        Xte_list.append(feats[-SEQ:])
        rv = rul_df["RUL"].values.astype(np.float32)[i]
        yte_r_list.append(float(rv))
        yte_s_list.append(0 if rv>50 else (1 if rv>25 else 2))
        
    Xte = torch.tensor(np.array(Xte_list))
    yte_r = torch.tensor(yte_r_list, dtype=torch.float32)
    yte_s = torch.tensor(yte_s_list, dtype=torch.long)
    
    dl = lambda X,yr,ys,sh: DataLoader(TensorDataset(X,yr,ys), batch_size=BATCH, shuffle=sh, num_workers=0)
    return dl(Xtr,ytr_r,ytr_s,True), dl(Xvl,yvl_r,yvl_s,False), dl(Xte,yte_r,yte_s,False)

def load_battery(seed):
    meta = pd.read_csv(os.path.join(BATTERY_DIR, "metadata.csv"))
    discharge = meta[meta["type"]=="discharge"].copy()
    batteries = discharge["battery_id"].unique()
    
    # Leave-One-Cell-Out (LOCO) Folds
    folds = []
    for test_cell in batteries:
        train_cells = [b for b in batteries if b != test_cell]
        tr_seqs, vl_seqs, te_seqs = [], [], []
        
        # Process Train/Val Cells
        for bid in train_cells:
            rows = discharge[discharge["battery_id"]==bid].sort_values("start_time").reset_index(drop=True)
            cap_vals = [float(r['Capacity']) for _, r in rows.iterrows() if 'Capacity' in r and pd.notnull(r['Capacity'])]
            if len(cap_vals) < SEQ + 5: continue
            cap_arr = np.array(cap_vals, dtype=np.float32)
            max_cap = cap_arr[0] if cap_arr[0]>0 else cap_arr.max()
            soh = np.clip(cap_arr/max_cap, 0.0, 1.0)
            delta = np.diff(soh, prepend=soh[0])
            # REVIEWER FIX: t/H instead of t/n
            cyc_norm = np.arange(len(cap_vals), dtype=np.float32) / 200.0 
            feats = np.stack([soh, delta, cyc_norm, cap_arr/(max_cap+1e-8)], axis=1)
            eol_idx = int(np.where(soh<0.8)[0][0]) if np.any(soh<0.8) else len(cap_vals)
            rul = np.clip((eol_idx-np.arange(len(cap_vals))).astype(np.float32),0,200)
            state = (rul<=50).astype(np.int64)+(rul<=20).astype(np.int64)
            
            cell_seqs = [(feats[t:t+SEQ], rul[t+SEQ-1], state[t+SEQ-1]) for t in range(len(feats)-SEQ+1)]
            split_idx = int(len(cell_seqs) * 0.8)
            tr_seqs.extend(cell_seqs[:split_idx])
            vl_seqs.extend(cell_seqs[split_idx:])
            
        # Process Test Cell
        rows = discharge[discharge["battery_id"]==test_cell].sort_values("start_time").reset_index(drop=True)
        cap_vals = [float(r['Capacity']) for _, r in rows.iterrows() if 'Capacity' in r and pd.notnull(r['Capacity'])]
        if len(cap_vals) >= SEQ + 5:
            cap_arr = np.array(cap_vals, dtype=np.float32)
            max_cap = cap_arr[0] if cap_arr[0]>0 else cap_arr.max()
            soh = np.clip(cap_arr/max_cap, 0.0, 1.0)
            delta = np.diff(soh, prepend=soh[0])
            cyc_norm = np.arange(len(cap_vals), dtype=np.float32) / 200.0
            feats = np.stack([soh, delta, cyc_norm, cap_arr/(max_cap+1e-8)], axis=1)
            eol_idx = int(np.where(soh<0.8)[0][0]) if np.any(soh<0.8) else len(cap_vals)
            rul = np.clip((eol_idx-np.arange(len(cap_vals))).astype(np.float32),0,200)
            state = (rul<=50).astype(np.int64)+(rul<=20).astype(np.int64)
            te_seqs = [(feats[t:t+SEQ], rul[t+SEQ-1], state[t+SEQ-1]) for t in range(len(feats)-SEQ+1)]
        else:
            te_seqs = []
            
        if len(tr_seqs) > 0 and len(te_seqs) > 0:
            folds.append((test_cell, tr_seqs, vl_seqs, te_seqs))
            
    # Select fold based on seed
    fold_idx = seed % len(folds)
    test_cell, tr_seqs, vl_seqs, te_seqs = folds[fold_idx]
    
    rng = np.random.RandomState(seed)
    rng.shuffle(tr_seqs)
    
    dl = lambda seqs, sh: DataLoader(TensorDataset(
        torch.tensor(np.array([s[0] for s in seqs])),
        torch.tensor([s[1] for s in seqs], dtype=torch.float32),
        torch.tensor([s[2] for s in seqs], dtype=torch.long)
    ), batch_size=128, shuffle=sh, num_workers=0)
    
    return dl(tr_seqs, True), dl(vl_seqs, False), dl(te_seqs, False), 4, test_cell

def load_cwru(seed):
    CWRU_FILES = {
        '97.mat':0,'98.mat':0,'99.mat':0,'100.mat':0, '105.mat':1,'106.mat':1,'107.mat':1,'108.mat':1,
        '169.mat':2,'170.mat':2,'171.mat':2,'172.mat':2, '209.mat':3,'210.mat':3,'211.mat':3,'212.mat':3,
        '118.mat':4,'119.mat':4,'120.mat':4,'121.mat':4, '185.mat':5,'186.mat':5,'187.mat':5,'188.mat':5,
        '222.mat':6,'223.mat':6,'224.mat':6,'225.mat':6, '130.mat':7,'131.mat':7,'132.mat':7,'133.mat':7,
        '197.mat':8,'198.mat':8,'199.mat':8,'200.mat':8, '234.mat':9,'235.mat':9,'236.mat':9,'237.mat':9,
    }
    
    # REVIEWER FIX: Record-Level Split (No overlapping window leakage)
    classes = {}
    for f, c in CWRU_FILES.items():
        if c not in classes: classes[c] = []
        classes[c].append(f)
        
    tr_files, vl_files, te_files = [], [], []
    rng = np.random.RandomState(seed)
    for c, files in classes.items():
        rng.shuffle(files)
        tr_files.extend(files[:2])
        vl_files.extend(files[2:3])
        te_files.extend(files[3:])
        
    def extract_windows(file_list):
        X, y = [], []
        for fname in file_list:
            fpath = os.path.join(CWRU_DIR, fname)
            if not os.path.exists(fpath): continue
            mat = scipy.io.loadmat(fpath)
            key = [k for k in mat.keys() if "DE_time" in k][0]
            sig = mat[key].flatten().astype(np.float32)
            sig = (sig - sig.mean()) / (sig.std() + 1e-8)
            for start in range(0, len(sig)-1024+1, 512):
                X.append(sig[start:start+1024].reshape(1024, 1))
                y.append(CWRU_FILES[fname])
        return torch.tensor(np.array(X)), torch.tensor(y, dtype=torch.long)
        
    Xtr, ytr = extract_windows(tr_files)
    Xvl, yvl = extract_windows(vl_files)
    Xte, yte = extract_windows(te_files)
    
    dummy_rul = torch.zeros(len(ytr), dtype=torch.float32)
    dl = lambda X, y, sh: DataLoader(TensorDataset(X, dummy_rul[:len(X)], y), batch_size=128, shuffle=sh, num_workers=0)
    return dl(Xtr, ytr, True), dl(Xvl, yvl, False), dl(Xte, yte, False)

# ==========================================================
# 3. MODELS (NanoSentry, SE, ECA, CNN, LSTM)
# ==========================================================
class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()
        self.conv = nn.Conv1d(d, d, kernel_size=3, padding=dilation, dilation=dilation, padding_mode="zeros")
        self.bn, self.act = nn.BatchNorm1d(d), nn.GELU()
    def forward(self, x): return x + self.act(self.bn(self.conv(x)))

class CrossSensorGate(nn.Module):
    def __init__(self, d): super().__init__(); self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x): return x * self.gate(x.mean(dim=1, keepdim=True))

class SEAttention(nn.Module):
    def __init__(self, d): 
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d, d//2), nn.ReLU(), nn.Linear(d//2, d), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x.mean(dim=1, keepdim=True))

class ECAAttention(nn.Module):
    def __init__(self, d): 
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=3, padding=1, bias=False)
    def forward(self, x): 
        y = self.pool(x.permute(0,2,1)).permute(0,2,1)
        return x * torch.sigmoid(self.conv(y.permute(0,2,1)).permute(0,2,1))

class NanoSentry(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3, attn_type='CSG'):
        super().__init__()
        self.embed = nn.Linear(n_sensors, d)
        self.tcn = nn.Sequential(DilatedBlock(d,1), DilatedBlock(d,2), DilatedBlock(d,4))
        if attn_type == 'CSG': self.gate = CrossSensorGate(d)
        elif attn_type == 'SE': self.gate = SEAttention(d)
        elif attn_type == 'ECA': self.gate = ECAAttention(d)
        else: self.gate = nn.Identity()
        self.skip = nn.Linear(n_sensors, d)
        self.gru = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)
        self.rul_head = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32,1))
        self.state_head = nn.Sequential(nn.Linear(d,32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32,n_classes))
    def forward(self, x):
        h = self.tcn(self.embed(x).permute(0,2,1)).permute(0,2,1)
        h = self.gate(h) + self.skip(x)
        _, hT = self.gru(h)
        hT = hT.squeeze(0)
        return self.rul_head(hT).squeeze(-1), self.state_head(hT)

class BaselineCNN(nn.Module):
    def __init__(self, n_sensors=14):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_sensors, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64), nn.AdaptiveAvgPool1d(1))
        self.head = nn.Linear(64, 1)
    def forward(self, x): return self.head(self.net(x.permute(0,2,1)).squeeze(-1)).squeeze(-1), torch.zeros(x.size(0), 3).to(x.device)

class BaselineLSTM(nn.Module):
    def __init__(self, n_sensors=14):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, 64, 2, batch_first=True, dropout=0.2)
        self.head = nn.Linear(64, 1)
    def forward(self, x): _, (h, _) = self.lstm(x); return self.head(h[-1]).squeeze(-1), torch.zeros(x.size(0), 3).to(x.device)

# ==========================================================
# 4. TRAINING & EVALUATION ENGINE
# ==========================================================
def nasa_score(p, t):
    d = p - t
    s = np.where(d<0, np.exp(-d/13)-1, np.exp(d/10)-1)
    return float(s.sum()) if np.isfinite(s.sum()) else 0.0

def train_and_eval(model, tr, va, te, is_cwru=False, is_multi=True, lr=1e-3, epochs=100, patience=20):
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    mse, ce = nn.MSELoss(), nn.CrossEntropyLoss()
    best_val, best_sd, wait = float('inf') if not is_cwru else -1.0, None, 0
    
    for _ in range(epochs):
        model.train()
        for X, yr, ys in tr:
            X, yr, ys = X.to(DEVICE), yr.to(DEVICE), ys.to(DEVICE)
            opt.zero_grad()
            pr, ps = model(X)
            if is_cwru: loss = ce(ps, ys)
            else: loss = mse(pr, yr) + (0.15 * ce(ps, ys) if is_multi else 0.0)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        
        model.eval()
        preds_r, trues_r, preds_s, trues_s = [], [], [], []
        with torch.no_grad():
            for X, yr, ys in va:
                pr, ps = model(X.to(DEVICE))
                if not is_cwru: preds_r.extend(pr.cpu().numpy()); trues_r.extend(yr.numpy())
                preds_s.extend(ps.argmax(1).cpu().numpy()); trues_s.extend(ys.numpy())
        
        if is_cwru:
            val_metric = float((np.array(preds_s)==np.array(trues_s)).mean())
            if val_metric > best_val: best_val, best_sd, wait = val_metric, {k:v.clone() for k,v in model.state_dict().items()}, 0
            else: wait += 1
        else:
            val_metric = float(np.sqrt(np.mean((np.array(preds_r)-np.array(trues_r))**2)))
            if val_metric < best_val: best_val, best_sd, wait = val_metric, {k:v.clone() for k,v in model.state_dict().items()}, 0
            else: wait += 1
        if wait >= patience: break
        
    model.load_state_dict(best_sd)
    model.eval()
    preds_r, trues_r, preds_s, trues_s = [], [], [], []
    with torch.no_grad():
        for X, yr, ys in te:
            pr, ps = model(X.to(DEVICE))
            if not is_cwru: preds_r.extend(pr.cpu().numpy()); trues_r.extend(yr.numpy())
            preds_s.extend(ps.argmax(1).cpu().numpy()); trues_s.extend(ys.numpy())
            
    pr, tr_ = np.array(preds_r), np.array(trues_r)
    ps, ts = np.array(preds_s), np.array(trues_s)
    
    rmse = float(np.sqrt(np.mean((pr-tr_)**2))) if not is_cwru else 0.0
    mae = float(np.mean(np.abs(pr-tr_))) if not is_cwru else 0.0
    score = nasa_score(pr, tr_) if not is_cwru else 0.0
    cz_mask = tr_ <= 30
    cz_rmse = float(np.sqrt(np.mean((pr[cz_mask]-tr_[cz_mask])**2))) if cz_mask.sum()>0 and not is_cwru else 0.0
    acc = float((ps==ts).mean()*100)
    
    params = sum(p.numel() for p in model.parameters())
    kb = sum(p.numel()*p.element_size() for p in model.parameters())/1024
    return rmse, mae, score, cz_rmse, acc, params, kb

# ==========================================================
# 5. MASTER EXECUTION LOOP
# ==========================================================
SEEDS = [42, 7, 13, 99, 2025]

print("🔍 Scanning for already completed experiments...")
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    
    # --- C-MAPSS ---
    for fd in [1, 2, 3, 4]:
        tr, va, te = load_cmapss(fd, seed)
        subset = f"FD00{fd}"
        
        # NanoSentry Multi-Task
        if not is_done("C-MAPSS", subset, "NanoSentry", "multi_task", seed):
            m = NanoSentry(14, 24, 3, 'CSG').to(DEVICE)
            rmse, mae, score, cz, acc, p, kb = train_and_eval(m, tr, va, te, is_multi=True)
            save_result({'dataset':'C-MAPSS', 'subset':subset, 'model':'NanoSentry', 'variant':'multi_task', 'seed':seed, 'rmse':rmse, 'mae':mae, 'score':score, 'cz_rmse':cz, 'acc':acc, 'params':p, 'kb':kb})
            
        # NanoSentry Single-Task
        if not is_done("C-MAPSS", subset, "NanoSentry", "single_task", seed):
            m = NanoSentry(14, 24, 3, 'CSG').to(DEVICE)
            rmse, mae, score, cz, acc, p, kb = train_and_eval(m, tr, va, te, is_multi=False)
            save_result({'dataset':'C-MAPSS', 'subset':subset, 'model':'NanoSentry', 'variant':'single_task', 'seed':seed, 'rmse':rmse, 'mae':mae, 'score':score, 'cz_rmse':cz, 'acc':acc, 'params':p, 'kb':kb})

        # Baselines (3 seeds to save time)
        if seed in [42, 7, 13]:
            if not is_done("C-MAPSS", subset, "CNN-32", "single_task", seed):
                m = BaselineCNN(14).to(DEVICE)
                rmse, mae, score, cz, acc, p, kb = train_and_eval(m, tr, va, te, is_multi=False)
                save_result({'dataset':'C-MAPSS', 'subset':subset, 'model':'CNN-32', 'variant':'single_task', 'seed':seed, 'rmse':rmse, 'mae':mae, 'score':score, 'cz_rmse':cz, 'acc':acc, 'params':p, 'kb':kb})
                
            if not is_done("C-MAPSS", subset, "LSTM-64", "single_task", seed):
                m = BaselineLSTM(14).to(DEVICE)
                rmse, mae, score, cz, acc, p, kb = train_and_eval(m, tr, va, te, is_multi=False)
                save_result({'dataset':'C-MAPSS', 'subset':subset, 'model':'LSTM-64', 'variant':'single_task', 'seed':seed, 'rmse':rmse, 'mae':mae, 'score':score, 'cz_rmse':cz, 'acc':acc, 'params':p, 'kb':kb})

        # Attention Variants (FD001 only, 3 seeds)
        if fd == 1 and seed in [42, 7, 13]:
            for attn in ['SE', 'ECA', 'None']:
                if not is_done("C-MAPSS", subset, f"NanoSentry_{attn}", "multi_task", seed):
                    m = NanoSentry(14, 24, 3, attn).to(DEVICE)
                    rmse, mae, score, cz, acc, p, kb = train_and_eval(m, tr, va, te, is_multi=True)
                    save_result({'dataset':'C-MAPSS', 'subset':subset, 'model':f'NanoSentry_{attn}', 'variant':'multi_task', 'seed':seed, 'rmse':rmse, 'mae':mae, 'score':score, 'cz_rmse':cz, 'acc':acc, 'params':p, 'kb':kb})

    # --- BATTERY (LOCO) ---
    tr, va, te, nf, test_cell = load_battery(seed)
    subset = f"LOCO_{test_cell}"
    if not is_done("Battery", subset, "NanoSentry", "multi_task", seed):
        m = NanoSentry(nf, 24, 3, 'CSG').to(DEVICE)
        rmse, mae, score, cz, acc, p, kb = train_and_eval(m, tr, va, te, is_multi=True)
        save_result({'dataset':'Battery', 'subset':subset, 'model':'NanoSentry', 'variant':'multi_task', 'seed':seed, 'rmse':rmse, 'mae':mae, 'score':score, 'cz_rmse':cz, 'acc':acc, 'params':p, 'kb':kb})

    # --- CWRU (Record-Level) ---
    tr, va, te = load_cwru(seed)
    if not is_done("CWRU", "10-class", "NanoSentry", "classifier", seed):
        m = NanoSentry(1, 24, 10, 'CSG').to(DEVICE)
        rmse, mae, score, cz, acc, p, kb = train_and_eval(m, tr, va, te, is_cwru=True, is_multi=False)
        save_result({'dataset':'CWRU', 'subset':'10-class', 'model':'NanoSentry', 'variant':'classifier', 'seed':seed, 'rmse':rmse, 'mae':mae, 'score':score, 'cz_rmse':cz, 'acc':acc, 'params':p, 'kb':kb})

print("\n🎉 ALL EXPERIMENTS COMPLETED SUCCESSFULLY!")
print(f"📊 Download your results from: {RESULTS_FILE}")

🚀 Running on: cpu
🔍 Scanning for already completed experiments...
✅ Saved: C-MAPSS | FD001 | NanoSentry | multi_task | Seed 42
✅ Saved: C-MAPSS | FD001 | NanoSentry | single_task | Seed 42
✅ Saved: C-MAPSS | FD001 | CNN-32 | single_task | Seed 42
✅ Saved: C-MAPSS | FD001 | LSTM-64 | single_task | Seed 42
✅ Saved: C-MAPSS | FD001 | NanoSentry_SE | multi_task | Seed 42


RuntimeError: Given groups=1, weight of size [1, 1, 3], expected input[256, 24, 1] to have 1 channels, but got 24 channels instead

In [1]:
# ============================================================
# NanoSentry Revised Master Experiment Script
# Fixed ECA bug + resume-friendly + error-safe
# ============================================================

import os
import csv
import time
import warnings
import gc
import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

# ============================================================
# GLOBAL CONFIG
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"🚀 Running on: {DEVICE}")

if DEVICE == "cpu":
    print("⚠ WARNING: CPU mode is much slower. Enable GPU in Kaggle if possible.")

CMAPSS_DIR = "/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps"
BATTERY_DIR = "/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset"
CWRU_DIR = "/kaggle/input/datasets/sufian79/cwru-mat-full-dataset"

RESULTS_FILE = "/kaggle/working/master_results_fixed.csv"
ERROR_FILE = "/kaggle/working/master_errors.log"

os.makedirs("/kaggle/working", exist_ok=True)

FIELDNAMES = [
    "dataset",
    "subset",
    "model",
    "variant",
    "seed",
    "fold",
    "rmse",
    "mae",
    "score",
    "cz_rmse",
    "acc",
    "f1",
    "params",
    "kb"
]

SEQ = 64
CWRU_SEQ = 1024
BATTERY_HORIZON = 200.0

SENSORS = [6, 7, 8, 11, 12, 13, 15, 16, 17, 18, 19, 21, 24, 25]
OP_COLS = [2, 3, 4]

# Stronger settings on GPU, safer settings on CPU
if DEVICE == "cuda":
    BATCH = 128
    EPOCHS = 150
    PATIENCE = 30
    MAIN_SEEDS = [42, 7, 13, 99, 2025]
    BASE_SEEDS = [42, 7, 13]
    CWRU_MAIN_SEEDS = [42, 7, 13, 99, 2025]
    ATTENTION_FDS = [1, 3]
else:
    BATCH = 64
    EPOCHS = 100
    PATIENCE = 20
    MAIN_SEEDS = [42, 7, 13]
    BASE_SEEDS = [42, 7, 13]
    CWRU_MAIN_SEEDS = [42, 7, 13]
    ATTENTION_FDS = [1]

# For single-task vs multi-task comparison, we do FD001 and FD003 fully.
# This saves time while still giving the reviewer the required comparison.
SINGLE_TASK_FDS = [1, 3]

CWRU_FILES = {
    "97.mat": 0, "98.mat": 0, "99.mat": 0, "100.mat": 0,
    "105.mat": 1, "106.mat": 1, "107.mat": 1, "108.mat": 1,
    "169.mat": 2, "170.mat": 2, "171.mat": 2, "172.mat": 2,
    "209.mat": 3, "210.mat": 3, "211.mat": 3, "212.mat": 3,
    "118.mat": 4, "119.mat": 4, "120.mat": 4, "121.mat": 4,
    "185.mat": 5, "186.mat": 5, "187.mat": 5, "188.mat": 5,
    "222.mat": 6, "223.mat": 6, "224.mat": 6, "225.mat": 6,
    "130.mat": 7, "131.mat": 7, "132.mat": 7, "133.mat": 7,
    "197.mat": 8, "198.mat": 8, "199.mat": 8, "200.mat": 8,
    "234.mat": 9, "235.mat": 9, "236.mat": 9, "237.mat": 9,
}


# ============================================================
# CHECKPOINT / LOGGING SYSTEM
# ============================================================

def is_done(dataset, subset, model, variant, seed, fold=""):
    if not os.path.exists(RESULTS_FILE):
        return False
    try:
        df = pd.read_csv(RESULTS_FILE)
        mask = (
            (df["dataset"] == dataset) &
            (df["subset"] == subset) &
            (df["model"] == model) &
            (df["variant"] == variant) &
            (df["seed"] == seed) &
            (df["fold"].astype(str) == str(fold))
        )
        return len(df[mask]) > 0
    except Exception:
        return False


def save_result(row):
    for k in FIELDNAMES:
        row.setdefault(k, "")

    file_exists = os.path.exists(RESULTS_FILE)
    with open(RESULTS_FILE, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in FIELDNAMES})

    print(
        f"✅ Saved: {row['dataset']} | {row['subset']} | {row['model']} | "
        f"{row['variant']} | Seed {row['seed']} | Fold {row['fold']}"
    )


def log_error(message):
    with open(ERROR_FILE, "a") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S") + " | " + message + "\n")
    print(f"[ERROR] {message}")


def cleanup():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


# ============================================================
# METRICS
# ============================================================

def nasa_score(pred, true):
    pred = np.array(pred, dtype=np.float64)
    true = np.array(true, dtype=np.float64)
    d = pred - true
    s = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
    total = float(s.sum())
    return total if np.isfinite(total) else np.nan


def critical_zone_rmse(pred, true, threshold=30):
    pred = np.array(pred, dtype=np.float64)
    true = np.array(true, dtype=np.float64)
    mask = true <= threshold
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((pred[mask] - true[mask]) ** 2)))


def macro_f1(pred, true, num_classes):
    pred = np.array(pred)
    true = np.array(true)
    f1s = []
    for c in range(num_classes):
        tp = int(np.sum((pred == c) & (true == c)))
        fp = int(np.sum((pred == c) & (true != c)))
        fn = int(np.sum((pred != c) & (true == c)))

        if tp == 0:
            f1s.append(0.0)
            continue

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        f1s.append(f1)

    return float(np.mean(f1s)) if len(f1s) > 0 else np.nan


def rul_to_state(rul):
    """
    Unified health-state definition:
    0 = healthy       RUL > 60
    1 = transitional  31 <= RUL <= 60
    2 = critical      RUL <= 30
    """
    if rul > 60:
        return 0
    elif rul > 30:
        return 1
    else:
        return 2


def count_params(model):
    params = sum(p.numel() for p in model.parameters())
    kb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024.0
    return params, kb


# ============================================================
# C-MAPSS DATA LOADER
# ============================================================

def load_cmapss(fd, seed):
    train_df = pd.read_csv(f"{CMAPSS_DIR}/train_FD{fd:03d}.txt", sep=r"\s+", header=None)
    test_df = pd.read_csv(f"{CMAPSS_DIR}/test_FD{fd:03d}.txt", sep=r"\s+", header=None)
    rul_df = pd.read_csv(f"{CMAPSS_DIR}/RUL_FD{fd:03d}.txt", header=None, names=["RUL"])

    max_cycles = train_df.groupby(0)[1].max().rename("max_cycle")
    train_df = train_df.join(max_cycles, on=0)
    train_df["RUL"] = (train_df["max_cycle"] - train_df[1]).clip(upper=125)

    engines = train_df[0].unique()
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(engines))
    n_tr = int(len(engines) * 0.8)

    tr_eng = engines[idx[:n_tr]]
    vl_eng = engines[idx[n_tr:]]

    # Fit normalization only on training engines
    fit_df = train_df[train_df[0].isin(tr_eng)]

    norm_train = train_df.copy()
    norm_test = test_df.copy()

    mu_global = fit_df[SENSORS].mean()
    std_global = fit_df[SENSORS].std().replace(0, 1)

    norm_train[SENSORS] = (train_df[SENSORS] - mu_global) / std_global
    norm_test[SENSORS] = (test_df[SENSORS] - mu_global) / std_global

    # Condition-aware normalization for multi-condition subsets
    if fd in [2, 4]:
        km = KMeans(n_clusters=6, random_state=seed, n_init=10)
        km.fit(fit_df[OP_COLS].values)

        fit_labels = km.predict(fit_df[OP_COLS].values)
        all_train_labels = km.predict(train_df[OP_COLS].values)
        test_labels = km.predict(test_df[OP_COLS].values)

        for c in range(6):
            fit_mask = fit_labels == c
            if fit_mask.sum() < 2:
                continue

            mu_c = fit_df.loc[fit_mask, SENSORS].mean()
            std_c = fit_df.loc[fit_mask, SENSORS].std().replace(0, 1)

            train_mask = all_train_labels == c
            test_mask = test_labels == c

            norm_train.loc[train_mask, SENSORS] = (
                train_df.loc[train_mask, SENSORS] - mu_c
            ) / std_c

            if test_mask.sum() > 0:
                norm_test.loc[test_mask, SENSORS] = (
                    test_df.loc[test_mask, SENSORS] - mu_c
                ) / std_c

    def make_windows(df, engine_list):
        Xs, yr, ys = [], [], []
        for eng in engine_list:
            sub = df[df[0] == eng].reset_index(drop=True)
            feats = sub[SENSORS].values.astype(np.float32)
            n = len(feats)

            if n < SEQ:
                pad = np.zeros((SEQ - n, len(SENSORS)), dtype=np.float32)
                feats = np.vstack([pad, feats])
                n = SEQ

            for t in range(n - SEQ + 1):
                Xs.append(feats[t:t + SEQ])
                rul_val = float(sub.loc[min(t + SEQ - 1, len(sub) - 1), "RUL"])
                yr.append(rul_val)
                ys.append(rul_to_state(rul_val))

        return (
            torch.tensor(np.array(Xs), dtype=torch.float32),
            torch.tensor(yr, dtype=torch.float32),
            torch.tensor(ys, dtype=torch.long)
        )

    Xtr, ytr_r, ytr_s = make_windows(norm_train, tr_eng)
    Xvl, yvl_r, yvl_s = make_windows(norm_train, vl_eng)

    rul_test = rul_df["RUL"].values.astype(np.float32)
    Xte_list, yte_r_list, yte_s_list = [], [], []

    for i, eng in enumerate(test_df[0].unique()):
        sub = norm_test[norm_test[0] == eng].reset_index(drop=True)
        feats = sub[SENSORS].values.astype(np.float32)
        n = len(feats)

        if n < SEQ:
            pad = np.zeros((SEQ - n, len(SENSORS)), dtype=np.float32)
            feats = np.vstack([pad, feats])

        Xte_list.append(feats[-SEQ:])
        rv = float(rul_test[i])
        yte_r_list.append(rv)
        yte_s_list.append(rul_to_state(rv))

    Xte = torch.tensor(np.array(Xte_list), dtype=torch.float32)
    yte_r = torch.tensor(yte_r_list, dtype=torch.float32)
    yte_s = torch.tensor(yte_s_list, dtype=torch.long)

    pin = DEVICE == "cuda"

    tr = DataLoader(TensorDataset(Xtr, ytr_r, ytr_s), batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=pin)
    va = DataLoader(TensorDataset(Xvl, yvl_r, yvl_s), batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=pin)
    te = DataLoader(TensorDataset(Xte, yte_r, yte_s), batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=pin)

    return tr, va, te


# ============================================================
# BATTERY DATA LOADER: LEAVE-ONE-CELL-OUT
# ============================================================

def seqs_to_loader(seqs, shuffle=False):
    X = torch.tensor(np.array([s[0] for s in seqs]), dtype=torch.float32)
    yr = torch.tensor(np.array([s[1] for s in seqs]), dtype=torch.float32)
    ys = torch.tensor(np.array([s[2] for s in seqs]), dtype=torch.long)

    return DataLoader(
        TensorDataset(X, yr, ys),
        batch_size=BATCH,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(DEVICE == "cuda")
    )


def build_battery_cell_sequences(cap_vals):
    cap_arr = np.array(cap_vals, dtype=np.float32)
    n = len(cap_arr)

    if n < SEQ + 5:
        return None

    max_cap = cap_arr[0] if cap_arr[0] > 0 else cap_arr.max()
    soh = np.clip(cap_arr / max_cap, 0.0, 1.0)
    delta = np.diff(soh, prepend=soh[0])

    # Reviewer-safe feature: t / H, not t / n
    cyc_norm = np.clip(np.arange(n, dtype=np.float32) / BATTERY_HORIZON, 0.0, 1.0)

    feats = np.stack(
        [soh, delta, cyc_norm, cap_arr / (max_cap + 1e-8)],
        axis=1
    ).astype(np.float32)

    eol = np.where(soh < 0.8)[0]
    eol_idx = int(eol[0]) if len(eol) > 0 else n

    rul = np.clip((eol_idx - np.arange(n)).astype(np.float32), 0.0, 200.0)
    states = np.array([rul_to_state(float(r)) for r in rul], dtype=np.int64)

    seqs = []
    for t in range(n - SEQ + 1):
        seqs.append(
            (
                feats[t:t + SEQ],
                float(rul[t + SEQ - 1]),
                int(states[t + SEQ - 1])
            )
        )

    return seqs


def load_battery_folds():
    meta = pd.read_csv(os.path.join(BATTERY_DIR, "metadata.csv"))
    discharge = meta[meta["type"] == "discharge"].copy()
    batteries = sorted(discharge["battery_id"].unique())

    folds = []

    for test_cell in batteries:
        train_cells = [b for b in batteries if b != test_cell]

        tr_seqs = []
        vl_seqs = []

        for bid in train_cells:
            rows = discharge[discharge["battery_id"] == bid].sort_values("start_time")
            cap_vals = pd.to_numeric(rows["Capacity"], errors="coerce").dropna().values

            cell_seqs = build_battery_cell_sequences(cap_vals)
            if cell_seqs is None:
                continue

            split_idx = int(len(cell_seqs) * 0.8)
            tr_seqs.extend(cell_seqs[:split_idx])
            vl_seqs.extend(cell_seqs[split_idx:])

        rows = discharge[discharge["battery_id"] == test_cell].sort_values("start_time")
        cap_vals = pd.to_numeric(rows["Capacity"], errors="coerce").dropna().values
        te_seqs = build_battery_cell_sequences(cap_vals)

        if te_seqs is None:
            continue

        if len(tr_seqs) > 0 and len(vl_seqs) > 0 and len(te_seqs) > 0:
            folds.append(
                {
                    "test_cell": str(test_cell),
                    "tr": tr_seqs,
                    "vl": vl_seqs,
                    "te": te_seqs
                }
            )

    return folds


# ============================================================
# CWRU DATA LOADER: RECORD-LEVEL SPLIT
# ============================================================

def find_cwru_file(fname):
    for root, _, files in os.walk(CWRU_DIR):
        if fname in files:
            return os.path.join(root, fname)
    return None


def load_cwru(seed):
    class_files = {c: [] for c in range(10)}
    for fname, cls_id in CWRU_FILES.items():
        class_files[cls_id].append(fname)

    rng = np.random.RandomState(seed)

    tr_files, vl_files, te_files = [], [], []

    for cls_id, files in class_files.items():
        files = files.copy()
        rng.shuffle(files)

        tr_files.extend(files[:2])
        vl_files.extend(files[2:3])
        te_files.extend(files[3:4])

    def extract_windows(file_list):
        X, y = [], []

        for fname in file_list:
            fpath = find_cwru_file(fname)
            if fpath is None:
                continue

            mat = scipy.io.loadmat(fpath)
            keys = [k for k in mat.keys() if "DE_time" in k]
            if len(keys) == 0:
                continue

            sig = mat[keys[0]].flatten().astype(np.float32)
            if len(sig) < CWRU_SEQ:
                continue

            sig = (sig - sig.mean()) / (sig.std() + 1e-8)

            stride = CWRU_SEQ // 2
            for start in range(0, len(sig) - CWRU_SEQ + 1, stride):
                X.append(sig[start:start + CWRU_SEQ].reshape(CWRU_SEQ, 1))
                y.append(CWRU_FILES[fname])

        if len(X) == 0:
            return torch.empty(0, CWRU_SEQ, 1), torch.empty(0, dtype=torch.long)

        return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(y, dtype=torch.long)

    Xtr, ytr = extract_windows(tr_files)
    Xvl, yvl = extract_windows(vl_files)
    Xte, yte = extract_windows(te_files)

    dummy_tr = torch.zeros(len(ytr), dtype=torch.float32)
    dummy_vl = torch.zeros(len(yvl), dtype=torch.float32)
    dummy_te = torch.zeros(len(yte), dtype=torch.float32)

    pin = DEVICE == "cuda"

    tr = DataLoader(TensorDataset(Xtr, dummy_tr, ytr), batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=pin)
    va = DataLoader(TensorDataset(Xvl, dummy_vl, yvl), batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=pin)
    te = DataLoader(TensorDataset(Xte, dummy_te, yte), batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=pin)

    return tr, va, te


# ============================================================
# MODELS
# ============================================================

class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()
        self.conv = nn.Conv1d(
            d, d,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
            padding_mode="zeros"
        )
        self.bn = nn.BatchNorm1d(d)
        self.act = nn.GELU()

    def forward(self, x):
        return x + self.act(self.bn(self.conv(x)))


class CrossSensorGate(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())

    def forward(self, x):
        g = self.gate(x.mean(dim=1, keepdim=True))
        return x * g


class SEAttention(nn.Module):
    def __init__(self, d, reduction=2):
        super().__init__()
        hidden = max(2, d // reduction)
        self.fc = nn.Sequential(
            nn.Linear(d, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d),
            nn.Sigmoid()
        )

    def forward(self, x):
        g = self.fc(x.mean(dim=1, keepdim=True))
        return x * g


class ECAAttention(nn.Module):
    """
    FIXED ECA implementation.
    Input x shape: (B, T, d)
    Global average pool over time -> (B, d)
    Treat channel dimension as 1D sequence -> (B, 1, d)
    Conv1d with 1 input channel and 1 output channel.
    """
    def __init__(self, d, k_size=3):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=1,
            kernel_size=k_size,
            padding=(k_size - 1) // 2,
            bias=False
        )

    def forward(self, x):
        # x: (B, T, d)
        y = x.mean(dim=1)              # (B, d)
        y = y.unsqueeze(1)             # (B, 1, d)
        y = self.conv(y)               # (B, 1, d)
        y = torch.sigmoid(y)
        y = y.squeeze(1).unsqueeze(-1) # (B, d, 1)
        return x * y


class NanoSentry(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3, attn_type="CSG"):
        super().__init__()

        self.embed = nn.Linear(n_sensors, d)
        self.tcn = nn.Sequential(
            DilatedBlock(d, 1),
            DilatedBlock(d, 2),
            DilatedBlock(d, 4)
        )

        if attn_type == "CSG":
            self.gate = CrossSensorGate(d)
        elif attn_type == "SE":
            self.gate = SEAttention(d)
        elif attn_type == "ECA":
            self.gate = ECAAttention(d)
        else:
            self.gate = nn.Identity()

        self.skip = nn.Linear(n_sensors, d)
        self.gru = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)

        self.rul_head = nn.Sequential(
            nn.Linear(d, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )

        self.state_head = nn.Sequential(
            nn.Linear(d, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        h = self.embed(x).permute(0, 2, 1)
        h = self.tcn(h).permute(0, 2, 1)
        h = self.gate(h) + self.skip(x)

        _, hT = self.gru(h)
        hT = hT.squeeze(0)

        rul = self.rul_head(hT).squeeze(-1)
        state = self.state_head(hT)

        return rul, state


class CNNBaseline(nn.Module):
    def __init__(self, n_channels=14, filters=32, n_classes=3, classification=False):
        super().__init__()
        self.classification = classification

        self.net = nn.Sequential(
            nn.Conv1d(n_channels, filters, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(filters),
            nn.Conv1d(filters, filters * 2, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(filters * 2),
            nn.AdaptiveAvgPool1d(1)
        )

        self.rul_head = nn.Linear(filters * 2, 1)

        if classification:
            self.cls_head = nn.Linear(filters * 2, n_classes)
        else:
            self.cls_head = None

    def forward(self, x):
        h = self.net(x.permute(0, 2, 1)).squeeze(-1)
        rul = self.rul_head(h).squeeze(-1)

        if self.cls_head is not None:
            cls = self.cls_head(h)
        else:
            cls = torch.zeros(x.size(0), 3, device=x.device)

        return rul, cls


class LSTMBaseline(nn.Module):
    def __init__(self, n_channels=14, hidden=64, n_classes=3, classification=False):
        super().__init__()
        self.classification = classification

        self.lstm = nn.LSTM(
            n_channels,
            hidden,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        self.rul_head = nn.Linear(hidden, 1)

        if classification:
            self.cls_head = nn.Linear(hidden, n_classes)
        else:
            self.cls_head = None

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        z = h[-1]

        rul = self.rul_head(z).squeeze(-1)

        if self.cls_head is not None:
            cls = self.cls_head(z)
        else:
            cls = torch.zeros(x.size(0), 3, device=x.device)

        return rul, cls


# ============================================================
# EVALUATION AND TRAINING
# ============================================================

def evaluate(model, loader, is_cwru=False, is_multi=True, num_classes=3):
    model.eval()

    preds_r, trues_r = [], []
    preds_s, trues_s = [], []

    with torch.no_grad():
        for X, yr, ys in loader:
            X = X.to(DEVICE)
            pr, ps = model(X)

            preds_r.extend(pr.cpu().numpy())
            trues_r.extend(yr.numpy())

            preds_s.extend(ps.argmax(1).cpu().numpy())
            trues_s.extend(ys.numpy())

    pr = np.array(preds_r, dtype=np.float64)
    tr = np.array(trues_r, dtype=np.float64)
    ps = np.array(preds_s, dtype=np.int64)
    ts = np.array(trues_s, dtype=np.int64)

    if is_cwru:
        acc = float((ps == ts).mean() * 100.0)
        f1 = macro_f1(ps, ts, num_classes)

        return {
            "rmse": np.nan,
            "mae": np.nan,
            "score": np.nan,
            "cz_rmse": np.nan,
            "acc": acc,
            "f1": f1
        }

    rmse = float(np.sqrt(np.mean((pr - tr) ** 2)))
    mae = float(np.mean(np.abs(pr - tr)))
    score = nasa_score(pr, tr)
    cz = critical_zone_rmse(pr, tr, threshold=30)

    if is_multi:
        acc = float((ps == ts).mean() * 100.0)
        f1 = macro_f1(ps, ts, num_classes)
    else:
        acc = np.nan
        f1 = np.nan

    return {
        "rmse": rmse,
        "mae": mae,
        "score": score,
        "cz_rmse": cz,
        "acc": acc,
        "f1": f1
    }


def train_and_eval(
    model,
    tr,
    va,
    te,
    is_cwru=False,
    is_multi=True,
    num_classes=3,
    epochs=EPOCHS,
    patience=PATIENCE,
    lr=1e-3
):
    model = model.to(DEVICE)

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)

    mse = nn.MSELoss()
    ce = nn.CrossEntropyLoss()

    best_val = -1.0 if is_cwru else float("inf")
    best_sd = None
    wait = 0

    for epoch in range(epochs):
        model.train()

        for X, yr, ys in tr:
            X = X.to(DEVICE)
            yr = yr.to(DEVICE)
            ys = ys.to(DEVICE)

            opt.zero_grad()

            pr, ps = model(X)

            if is_cwru:
                loss = ce(ps, ys)
            else:
                loss = mse(pr, yr)
                if is_multi:
                    loss = loss + 0.15 * ce(ps, ys)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()

        val_metrics = evaluate(model, va, is_cwru=is_cwru, is_multi=is_multi, num_classes=num_classes)

        if is_cwru:
            val_metric = -val_metrics["f1"]
        else:
            val_metric = val_metrics["rmse"]

        if val_metric < best_val:
            best_val = val_metric
            best_sd = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    if best_sd is not None:
        model.load_state_dict(best_sd)

    test_metrics = evaluate(model, te, is_cwru=is_cwru, is_multi=is_multi, num_classes=num_classes)
    params, kb = count_params(model)

    return test_metrics, params, kb


# ============================================================
# RIDGE BASELINE
# ============================================================

def loader_to_Xy(loader):
    Xs, ys = [], []
    for batch in loader:
        Xs.append(batch[0])
        ys.append(batch[1])
    return torch.cat(Xs, dim=0), torch.cat(ys, dim=0)


def window_summary_features(X):
    X = X.numpy()
    return np.concatenate(
        [
            X.mean(axis=1),
            X.std(axis=1),
            X.min(axis=1),
            X.max(axis=1)
        ],
        axis=1
    )


def run_ridge(tr, te):
    Xtr, ytr = loader_to_Xy(tr)
    Xte, yte = loader_to_Xy(te)

    Xtr_s = window_summary_features(Xtr)
    Xte_s = window_summary_features(Xte)

    model = Ridge(alpha=1.0)
    model.fit(Xtr_s, ytr.numpy())

    pred = model.predict(Xte_s)
    true = yte.numpy()

    rmse = float(np.sqrt(np.mean((pred - true) ** 2)))
    mae = float(np.mean(np.abs(pred - true)))
    score = nasa_score(pred, true)
    cz = critical_zone_rmse(pred, true, threshold=30)

    return {
        "rmse": rmse,
        "mae": mae,
        "score": score,
        "cz_rmse": cz,
        "acc": np.nan,
        "f1": np.nan
    }


# ============================================================
# MAIN EXPERIMENT LOOP
# ============================================================

print("🔍 Starting/resuming experiments...")

# ------------------------------------------------------------
# 1. C-MAPSS
# ------------------------------------------------------------

print("\n=== C-MAPSS ===")

for fd in [1, 2, 3, 4]:
    subset = f"FD{fd:03d}"

    for seed in MAIN_SEEDS:
        try:
            tr, va, te = load_cmapss(fd, seed)
        except Exception as e:
            log_error(f"C-MAPSS load failed | {subset} | seed {seed} | {e}")
            continue

        # NanoSentry multi-task
        if not is_done("C-MAPSS", subset, "NanoSentry", "multi_task", seed):
            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = NanoSentry(14, 24, 3, "CSG")
                metrics, params, kb = train_and_eval(
                    model, tr, va, te,
                    is_cwru=False,
                    is_multi=True,
                    num_classes=3
                )

                save_result({
                    "dataset": "C-MAPSS",
                    "subset": subset,
                    "model": "NanoSentry",
                    "variant": "multi_task",
                    "seed": seed,
                    "fold": "",
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"],
                    "score": metrics["score"],
                    "cz_rmse": metrics["cz_rmse"],
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"C-MAPSS NanoSentry multi_task | {subset} | seed {seed} | {e}")
            finally:
                cleanup()

        # NanoSentry single-task only for selected FDs to save time
        if fd in SINGLE_TASK_FDS:
            if not is_done("C-MAPSS", subset, "NanoSentry", "single_task", seed):
                try:
                    torch.manual_seed(seed)
                    np.random.seed(seed)

                    model = NanoSentry(14, 24, 3, "CSG")
                    metrics, params, kb = train_and_eval(
                        model, tr, va, te,
                        is_cwru=False,
                        is_multi=False,
                        num_classes=3
                    )

                    save_result({
                        "dataset": "C-MAPSS",
                        "subset": subset,
                        "model": "NanoSentry",
                        "variant": "single_task",
                        "seed": seed,
                        "fold": "",
                        "rmse": metrics["rmse"],
                        "mae": metrics["mae"],
                        "score": metrics["score"],
                        "cz_rmse": metrics["cz_rmse"],
                        "acc": metrics["acc"],
                        "f1": metrics["f1"],
                        "params": params,
                        "kb": kb
                    })

                except Exception as e:
                    log_error(f"C-MAPSS NanoSentry single_task | {subset} | seed {seed} | {e}")
                finally:
                    cleanup()

        # Baselines: use fewer seeds to save time
        if seed in BASE_SEEDS:

            # Ridge
            if not is_done("C-MAPSS", subset, "Ridge", "single_task", seed):
                try:
                    metrics = run_ridge(tr, te)

                    save_result({
                        "dataset": "C-MAPSS",
                        "subset": subset,
                        "model": "Ridge",
                        "variant": "single_task",
                        "seed": seed,
                        "fold": "",
                        "rmse": metrics["rmse"],
                        "mae": metrics["mae"],
                        "score": metrics["score"],
                        "cz_rmse": metrics["cz_rmse"],
                        "acc": "",
                        "f1": "",
                        "params": 0,
                        "kb": 0
                    })

                except Exception as e:
                    log_error(f"C-MAPSS Ridge | {subset} | seed {seed} | {e}")
                finally:
                    cleanup()

            # CNN-32
            if not is_done("C-MAPSS", subset, "CNN-32", "single_task", seed):
                try:
                    torch.manual_seed(seed)
                    np.random.seed(seed)

                    model = CNNBaseline(14, 32, 3, classification=False)
                    metrics, params, kb = train_and_eval(
                        model, tr, va, te,
                        is_cwru=False,
                        is_multi=False,
                        num_classes=3
                    )

                    save_result({
                        "dataset": "C-MAPSS",
                        "subset": subset,
                        "model": "CNN-32",
                        "variant": "single_task",
                        "seed": seed,
                        "fold": "",
                        "rmse": metrics["rmse"],
                        "mae": metrics["mae"],
                        "score": metrics["score"],
                        "cz_rmse": metrics["cz_rmse"],
                        "acc": metrics["acc"],
                        "f1": metrics["f1"],
                        "params": params,
                        "kb": kb
                    })

                except Exception as e:
                    log_error(f"C-MAPSS CNN-32 | {subset} | seed {seed} | {e}")
                finally:
                    cleanup()

            # LSTM-64
            if not is_done("C-MAPSS", subset, "LSTM-64", "single_task", seed):
                try:
                    torch.manual_seed(seed)
                    np.random.seed(seed)

                    model = LSTMBaseline(14, 64, 3, classification=False)
                    metrics, params, kb = train_and_eval(
                        model, tr, va, te,
                        is_cwru=False,
                        is_multi=False,
                        num_classes=3
                    )

                    save_result({
                        "dataset": "C-MAPSS",
                        "subset": subset,
                        "model": "LSTM-64",
                        "variant": "single_task",
                        "seed": seed,
                        "fold": "",
                        "rmse": metrics["rmse"],
                        "mae": metrics["mae"],
                        "score": metrics["score"],
                        "cz_rmse": metrics["cz_rmse"],
                        "acc": metrics["acc"],
                        "f1": metrics["f1"],
                        "params": params,
                        "kb": kb
                    })

                except Exception as e:
                    log_error(f"C-MAPSS LSTM-64 | {subset} | seed {seed} | {e}")
                finally:
                    cleanup()


# ------------------------------------------------------------
# 2. Attention comparison
# ------------------------------------------------------------

print("\n=== Attention Comparison ===")

for fd in ATTENTION_FDS:
    subset = f"FD{fd:03d}"

    for seed in BASE_SEEDS:
        try:
            tr, va, te = load_cmapss(fd, seed)
        except Exception as e:
            log_error(f"Attention load failed | {subset} | seed {seed} | {e}")
            continue

        for attn in ["SE", "ECA", "None"]:
            model_name = f"NanoSentry_{attn}"

            if not is_done("C-MAPSS", subset, model_name, "multi_task", seed):
                try:
                    torch.manual_seed(seed)
                    np.random.seed(seed)

                    model = NanoSentry(14, 24, 3, attn)
                    metrics, params, kb = train_and_eval(
                        model, tr, va, te,
                        is_cwru=False,
                        is_multi=True,
                        num_classes=3
                    )

                    save_result({
                        "dataset": "C-MAPSS",
                        "subset": subset,
                        "model": model_name,
                        "variant": "multi_task",
                        "seed": seed,
                        "fold": "",
                        "rmse": metrics["rmse"],
                        "mae": metrics["mae"],
                        "score": metrics["score"],
                        "cz_rmse": metrics["cz_rmse"],
                        "acc": metrics["acc"],
                        "f1": metrics["f1"],
                        "params": params,
                        "kb": kb
                    })

                except Exception as e:
                    log_error(f"Attention comparison | {model_name} | {subset} | seed {seed} | {e}")
                finally:
                    cleanup()


# ------------------------------------------------------------
# 3. Battery leave-one-cell-out
# ------------------------------------------------------------

print("\n=== Battery LOCO ===")

try:
    battery_folds = load_battery_folds()
except Exception as e:
    battery_folds = []
    log_error(f"Battery fold loading failed | {e}")

for fold in battery_folds:
    fold_id = fold["test_cell"]

    try:
        tr = seqs_to_loader(fold["tr"], shuffle=True)
        va = seqs_to_loader(fold["vl"], shuffle=False)
        te = seqs_to_loader(fold["te"], shuffle=False)
    except Exception as e:
        log_error(f"Battery loader failed | fold {fold_id} | {e}")
        continue

    for seed in BASE_SEEDS:

        # NanoSentry multi-task
        if not is_done("Battery", fold_id, "NanoSentry", "multi_task", seed, fold_id):
            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = NanoSentry(4, 24, 3, "CSG")
                metrics, params, kb = train_and_eval(
                    model, tr, va, te,
                    is_cwru=False,
                    is_multi=True,
                    num_classes=3
                )

                save_result({
                    "dataset": "Battery",
                    "subset": fold_id,
                    "model": "NanoSentry",
                    "variant": "multi_task",
                    "seed": seed,
                    "fold": fold_id,
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"],
                    "score": metrics["score"],
                    "cz_rmse": metrics["cz_rmse"],
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"Battery NanoSentry multi_task | fold {fold_id} | seed {seed} | {e}")
            finally:
                cleanup()

        # NanoSentry single-task
        if not is_done("Battery", fold_id, "NanoSentry", "single_task", seed, fold_id):
            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = NanoSentry(4, 24, 3, "CSG")
                metrics, params, kb = train_and_eval(
                    model, tr, va, te,
                    is_cwru=False,
                    is_multi=False,
                    num_classes=3
                )

                save_result({
                    "dataset": "Battery",
                    "subset": fold_id,
                    "model": "NanoSentry",
                    "variant": "single_task",
                    "seed": seed,
                    "fold": fold_id,
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"],
                    "score": metrics["score"],
                    "cz_rmse": metrics["cz_rmse"],
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"Battery NanoSentry single_task | fold {fold_id} | seed {seed} | {e}")
            finally:
                cleanup()

        # Ridge: deterministic for a fixed fold, run once only
        if seed == BASE_SEEDS[0]:
            if not is_done("Battery", fold_id, "Ridge", "single_task", seed, fold_id):
                try:
                    metrics = run_ridge(tr, te)

                    save_result({
                        "dataset": "Battery",
                        "subset": fold_id,
                        "model": "Ridge",
                        "variant": "single_task",
                        "seed": seed,
                        "fold": fold_id,
                        "rmse": metrics["rmse"],
                        "mae": metrics["mae"],
                        "score": metrics["score"],
                        "cz_rmse": metrics["cz_rmse"],
                        "acc": "",
                        "f1": "",
                        "params": 0,
                        "kb": 0
                    })

                except Exception as e:
                    log_error(f"Battery Ridge | fold {fold_id} | {e}")
                finally:
                    cleanup()

        # CNN-32
        if not is_done("Battery", fold_id, "CNN-32", "single_task", seed, fold_id):
            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = CNNBaseline(4, 32, 3, classification=False)
                metrics, params, kb = train_and_eval(
                    model, tr, va, te,
                    is_cwru=False,
                    is_multi=False,
                    num_classes=3
                )

                save_result({
                    "dataset": "Battery",
                    "subset": fold_id,
                    "model": "CNN-32",
                    "variant": "single_task",
                    "seed": seed,
                    "fold": fold_id,
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"],
                    "score": metrics["score"],
                    "cz_rmse": metrics["cz_rmse"],
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"Battery CNN-32 | fold {fold_id} | seed {seed} | {e}")
            finally:
                cleanup()

        # LSTM-64
        if not is_done("Battery", fold_id, "LSTM-64", "single_task", seed, fold_id):
            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = LSTMBaseline(4, 64, 3, classification=False)
                metrics, params, kb = train_and_eval(
                    model, tr, va, te,
                    is_cwru=False,
                    is_multi=False,
                    num_classes=3
                )

                save_result({
                    "dataset": "Battery",
                    "subset": fold_id,
                    "model": "LSTM-64",
                    "variant": "single_task",
                    "seed": seed,
                    "fold": fold_id,
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"],
                    "score": metrics["score"],
                    "cz_rmse": metrics["cz_rmse"],
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"Battery LSTM-64 | fold {fold_id} | seed {seed} | {e}")
            finally:
                cleanup()


# ------------------------------------------------------------
# 4. CWRU record-level classification
# ------------------------------------------------------------

print("\n=== CWRU Record-Level Split ===")

for seed in CWRU_MAIN_SEEDS:
    try:
        tr, va, te = load_cwru(seed)
    except Exception as e:
        log_error(f"CWRU load failed | seed {seed} | {e}")
        continue

    # NanoSentry classifier
    if not is_done("CWRU", "10-class", "NanoSentry", "classifier", seed):
        try:
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = NanoSentry(1, 24, 10, "CSG")
            metrics, params, kb = train_and_eval(
                model, tr, va, te,
                is_cwru=True,
                is_multi=False,
                num_classes=10
            )

            save_result({
                "dataset": "CWRU",
                "subset": "10-class",
                "model": "NanoSentry",
                "variant": "classifier",
                "seed": seed,
                "fold": "",
                "rmse": "",
                "mae": "",
                "score": "",
                "cz_rmse": "",
                "acc": metrics["acc"],
                "f1": metrics["f1"],
                "params": params,
                "kb": kb
            })

        except Exception as e:
            log_error(f"CWRU NanoSentry | seed {seed} | {e}")
        finally:
            cleanup()

    if seed in BASE_SEEDS:

        # CNN classifier
        if not is_done("CWRU", "10-class", "CNN-32", "classifier", seed):
            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = CNNBaseline(1, 32, 10, classification=True)
                metrics, params, kb = train_and_eval(
                    model, tr, va, te,
                    is_cwru=True,
                    is_multi=False,
                    num_classes=10
                )

                save_result({
                    "dataset": "CWRU",
                    "subset": "10-class",
                    "model": "CNN-32",
                    "variant": "classifier",
                    "seed": seed,
                    "fold": "",
                    "rmse": "",
                    "mae": "",
                    "score": "",
                    "cz_rmse": "",
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"CWRU CNN-32 | seed {seed} | {e}")
            finally:
                cleanup()

        # LSTM classifier
        if not is_done("CWRU", "10-class", "LSTM-64", "classifier", seed):
            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = LSTMBaseline(1, 64, 10, classification=True)
                metrics, params, kb = train_and_eval(
                    model, tr, va, te,
                    is_cwru=True,
                    is_multi=False,
                    num_classes=10
                )

                save_result({
                    "dataset": "CWRU",
                    "subset": "10-class",
                    "model": "LSTM-64",
                    "variant": "classifier",
                    "seed": seed,
                    "fold": "",
                    "rmse": "",
                    "mae": "",
                    "score": "",
                    "cz_rmse": "",
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"CWRU LSTM-64 | seed {seed} | {e}")
            finally:
                cleanup()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n=== EXPERIMENT RUN FINISHED ===")
print(f"Results file: {RESULTS_FILE}")
print(f"Error log: {ERROR_FILE}")

if os.path.exists(RESULTS_FILE):
    try:
        df = pd.read_csv(RESULTS_FILE)
        print(f"Total completed experiment rows: {len(df)}")
        print("\nPreview:")
        print(df.tail(20))
    except Exception as e:
        print(f"Could not print summary: {e}")

🚀 Running on: cuda
🔍 Starting/resuming experiments...

=== C-MAPSS ===
✅ Saved: C-MAPSS | FD001 | NanoSentry | multi_task | Seed 42 | Fold 
✅ Saved: C-MAPSS | FD001 | NanoSentry | single_task | Seed 42 | Fold 
✅ Saved: C-MAPSS | FD001 | Ridge | single_task | Seed 42 | Fold 
✅ Saved: C-MAPSS | FD001 | CNN-32 | single_task | Seed 42 | Fold 
✅ Saved: C-MAPSS | FD001 | LSTM-64 | single_task | Seed 42 | Fold 
✅ Saved: C-MAPSS | FD001 | NanoSentry | multi_task | Seed 7 | Fold 
✅ Saved: C-MAPSS | FD001 | NanoSentry | single_task | Seed 7 | Fold 
✅ Saved: C-MAPSS | FD001 | Ridge | single_task | Seed 7 | Fold 
✅ Saved: C-MAPSS | FD001 | CNN-32 | single_task | Seed 7 | Fold 
✅ Saved: C-MAPSS | FD001 | LSTM-64 | single_task | Seed 7 | Fold 
✅ Saved: C-MAPSS | FD001 | NanoSentry | multi_task | Seed 13 | Fold 
✅ Saved: C-MAPSS | FD001 | NanoSentry | single_task | Seed 13 | Fold 
✅ Saved: C-MAPSS | FD001 | Ridge | single_task | Seed 13 | Fold 
✅ Saved: C-MAPSS | FD001 | CNN-32 | single_task | Seed 1

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/working/master_results_fixed.csv')

# Convert numeric columns safely
for col in ['rmse', 'mae', 'score', 'cz_rmse', 'acc', 'f1']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

def fmt(mean, std):
    if pd.isna(mean): return "—"
    return f"{mean:.2f} ± {std:.2f}"

print("="*70)
print("TABLE 1: C-MAPSS MAIN RESULTS (Multi-Task NanoSentry vs Baselines)")
print("="*70)
cmapss = df[df['dataset'] == 'C-MAPSS']
# Get NanoSentry Multi-task
ns = cmapss[(cmapss['model'] == 'NanoSentry') & (cmapss['variant'] == 'multi_task')]
# Get Baselines (single_task)
base = cmapss[cmapss['variant'] == 'single_task']

for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    print(f"\n--- {fd} ---")
    for model_name in ['NanoSentry', 'LSTM-64', 'CNN-32', 'Ridge']:
        if model_name == 'NanoSentry':
            subset_df = ns[ns['subset'] == fd]
        else:
            subset_df = base[(base['subset'] == fd) & (base['model'] == model_name)]
            
        rmse = fmt(subset_df['rmse'].mean(), subset_df['rmse'].std())
        mae = fmt(subset_df['mae'].mean(), subset_df['mae'].std())
        score = fmt(subset_df['score'].mean(), subset_df['score'].std())
        cz = fmt(subset_df['cz_rmse'].mean(), subset_df['cz_rmse'].std())
        print(f"{model_name:<12} | RMSE: {rmse:<15} | MAE: {mae:<15} | Score: {score:<15} | Crit RMSE: {cz}")

print("\n" + "="*70)
print("TABLE 2: CWRU RECORD-LEVEL FAULT DIAGNOSIS (10-Class)")
print("="*70)
cwru = df[df['dataset'] == 'CWRU']
for model_name in ['NanoSentry', 'CNN-32', 'LSTM-64']:
    subset_df = cwru[cwru['model'] == model_name]
    acc = fmt(subset_df['acc'].mean(), subset_df['acc'].std())
    f1 = fmt(subset_df['f1'].mean(), subset_df['f1'].std())
    print(f"{model_name:<12} | Accuracy: {acc:<15} | Macro-F1: {f1}")

print("\n" + "="*70)
print("TABLE 3: BATTERY LOCO (Aggregated across all 17 cells)")
print("="*70)
bat = df[df['dataset'] == 'Battery']
for model_name in ['NanoSentry', 'LSTM-64', 'CNN-32', 'Ridge']:
    if model_name == 'NanoSentry':
        subset_df = bat[(bat['model'] == 'NanoSentry') & (bat['variant'] == 'multi_task')]
    else:
        subset_df = bat[(bat['model'] == model_name) & (bat['variant'] == 'single_task')]
        
    rmse = fmt(subset_df['rmse'].mean(), subset_df['rmse'].std())
    mae = fmt(subset_df['mae'].mean(), subset_df['mae'].std())
    print(f"{model_name:<12} | RMSE: {rmse:<15} | MAE: {mae}")


TABLE 1: C-MAPSS MAIN RESULTS (Multi-Task NanoSentry vs Baselines)

--- FD001 ---
NanoSentry   | RMSE: 15.26 ± 0.85    | MAE: 11.07 ± 0.65    | Score: 361.35 ± 50.81  | Crit RMSE: 4.72 ± 1.21
LSTM-64      | RMSE: 15.93 ± 0.83    | MAE: 11.85 ± 0.66    | Score: 377.65 ± 89.83  | Crit RMSE: 4.19 ± 0.34
CNN-32       | RMSE: 26.86 ± 0.86    | MAE: 21.47 ± 1.19    | Score: 2350.50 ± 182.68 | Crit RMSE: 27.24 ± 0.39
Ridge        | RMSE: 24.27 ± 0.17    | MAE: 19.62 ± 0.22    | Score: 2070.31 ± 257.26 | Crit RMSE: 19.51 ± 0.92

--- FD002 ---
NanoSentry   | RMSE: 25.57 ± 0.75    | MAE: 17.05 ± 0.52    | Score: 5782.16 ± 912.42 | Crit RMSE: 4.46 ± 0.62
LSTM-64      | RMSE: 27.19 ± 0.34    | MAE: 18.15 ± 0.26    | Score: 7197.46 ± 701.04 | Crit RMSE: 3.95 ± 0.80
CNN-32       | RMSE: 43.61 ± 3.22    | MAE: 31.52 ± 2.10    | Score: 299299.01 ± 252306.87 | Crit RMSE: 22.99 ± 2.67
Ridge        | RMSE: 36.19 ± 0.30    | MAE: 24.96 ± 0.33    | Score: 88271.54 ± 9119.07 | Crit RMSE: 12.72 ± 0.07

--- F

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/working/master_results_fixed.csv')
for col in ['rmse', 'mae', 'score', 'cz_rmse', 'acc', 'f1']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

def fmt(mean, std):
    if pd.isna(mean): return "—"
    return f"{mean:.2f} ± {std:.2f}"

print("="*70)
print("TABLE A: SINGLE-TASK vs MULTI-TASK (Reviewer 1 Request)")
print("="*70)
for ds, subsets in [('C-MAPSS', ['FD001', 'FD003']), ('Battery', df[df['dataset']=='Battery']['subset'].unique())]:
    for sub in subsets:
        print(f"\n--- {ds} {sub} ---")
        for var in ['multi_task', 'single_task']:
            sub_df = df[(df['dataset']==ds) & (df['subset']==sub) & (df['model']=='NanoSentry') & (df['variant']==var)]
            rmse = fmt(sub_df['rmse'].mean(), sub_df['rmse'].std())
            acc = fmt(sub_df['acc'].mean(), sub_df['acc'].std())
            print(f"NanoSentry ({var:<11}) | RMSE: {rmse:<15} | State Acc: {acc}")

print("\n" + "="*70)
print("TABLE B: ATTENTION COMPARISON (Reviewer 1 Request)")
print("="*70)
for sub in ['FD001', 'FD003']:
    print(f"\n--- {sub} ---")
    for model in ['NanoSentry', 'NanoSentry_SE', 'NanoSentry_ECA', 'NanoSentry_None']:
        sub_df = df[(df['dataset']=='C-MAPSS') & (df['subset']==sub) & (df['model']==model)]
        if sub_df.empty:
            print(f"{model:<18} | (Skipped / ECA Bug)")
            continue
        rmse = fmt(sub_df['rmse'].mean(), sub_df['rmse'].std())
        cz = fmt(sub_df['cz_rmse'].mean(), sub_df['cz_rmse'].std())
        params = sub_df['params'].mean()
        print(f"{model:<18} | RMSE: {rmse:<15} | Crit RMSE: {cz:<15} | Params: {params:.0f}")

TABLE A: SINGLE-TASK vs MULTI-TASK (Reviewer 1 Request)

--- C-MAPSS FD001 ---
NanoSentry (multi_task ) | RMSE: 15.26 ± 0.85    | State Acc: 89.40 ± 1.82
NanoSentry (single_task) | RMSE: 15.40 ± 0.89    | State Acc: —

--- C-MAPSS FD003 ---
NanoSentry (multi_task ) | RMSE: 14.13 ± 0.27    | State Acc: 90.60 ± 1.67
NanoSentry (single_task) | RMSE: 14.14 ± 0.26    | State Acc: —

--- Battery B0005 ---
NanoSentry (multi_task ) | RMSE: 0.22 ± 0.10     | State Acc: 66.67 ± 57.74
NanoSentry (single_task) | RMSE: 0.22 ± 0.10     | State Acc: —

--- Battery B0006 ---
NanoSentry (multi_task ) | RMSE: 0.21 ± 0.09     | State Acc: 66.67 ± 57.74
NanoSentry (single_task) | RMSE: 0.21 ± 0.09     | State Acc: —

--- Battery B0007 ---
NanoSentry (multi_task ) | RMSE: 0.23 ± 0.10     | State Acc: 66.67 ± 57.74
NanoSentry (single_task) | RMSE: 0.23 ± 0.10     | State Acc: —

--- Battery B0018 ---
NanoSentry (multi_task ) | RMSE: 0.22 ± 0.11     | State Acc: 66.67 ± 57.74
NanoSentry (single_task) | RMSE:

In [5]:
# ============================================================
# NanoSentry Final Polish Experiments
# Fixes ECA, runs ablation, improves Battery LOCO, improves CWRU
# ============================================================

import os
import csv
import time
import warnings
import gc
import numpy as np
import pandas as pd
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"Device: {DEVICE}")

# ============================================================
# Global config
# ============================================================

SEQ = 64
CWRU_SEQ = 1024
BATCH = 128
BATTERY_HORIZON = 200.0

BASE_SEEDS = [42, 7, 13]

# CWRU polish is heavy. Run it only if GPU is available.
RUN_CWRU_POLISH = DEVICE == "cuda"
CWRU_SEEDS = [42, 7, 13] if DEVICE == "cuda" else [42]

RESULTS_FILE = "/kaggle/working/polished_results.csv"
ERROR_FILE = "/kaggle/working/polished_errors.log"

FIELDNAMES = [
    "dataset",
    "subset",
    "model",
    "variant",
    "seed",
    "fold",
    "rmse",
    "mae",
    "score",
    "cz_rmse",
    "acc",
    "f1",
    "params",
    "kb"
]


# ============================================================
# Checkpoint system
# ============================================================

def is_done(dataset, subset, model, variant, seed, fold=""):
    if not os.path.exists(RESULTS_FILE):
        return False

    try:
        df = pd.read_csv(RESULTS_FILE)
        mask = (
            (df["dataset"] == dataset) &
            (df["subset"] == subset) &
            (df["model"] == model) &
            (df["variant"] == variant) &
            (df["seed"] == seed) &
            (df["fold"].astype(str) == str(fold))
        )
        return len(df[mask]) > 0
    except Exception:
        return False


def save_result(row):
    for k in FIELDNAMES:
        row.setdefault(k, "")

    file_exists = os.path.exists(RESULTS_FILE)

    with open(RESULTS_FILE, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in FIELDNAMES})

    print(
        f"Saved: {row['dataset']} | {row['subset']} | {row['model']} | "
        f"{row['variant']} | seed {row['seed']} | fold {row['fold']}"
    )


def log_error(message):
    with open(ERROR_FILE, "a") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S") + " | " + message + "\n")
    print(f"[ERROR] {message}")


def cleanup():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

Device: cuda


In [7]:
# ============================================================
# Metrics
# ============================================================

def nasa_score(pred, true):
    pred = np.array(pred, dtype=np.float64)
    true = np.array(true, dtype=np.float64)

    d = pred - true
    s = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)

    total = float(s.sum())
    return total if np.isfinite(total) else np.nan


def critical_zone_rmse(pred, true, threshold=30):
    pred = np.array(pred, dtype=np.float64)
    true = np.array(true, dtype=np.float64)

    mask = true <= threshold
    if mask.sum() == 0:
        return np.nan

    return float(np.sqrt(np.mean((pred[mask] - true[mask]) ** 2)))


def macro_f1(pred, true, num_classes):
    pred = np.array(pred)
    true = np.array(true)

    f1s = []

    for c in range(num_classes):
        tp = int(np.sum((pred == c) & (true == c)))
        fp = int(np.sum((pred == c) & (true != c)))
        fn = int(np.sum((pred != c) & (true == c)))

        if tp == 0:
            f1s.append(0.0)
            continue

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)

        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        f1s.append(f1)

    return float(np.mean(f1s)) if len(f1s) > 0 else np.nan


def rul_to_state(rul):
    """
    Unified health-state definition:
    0 = healthy       RUL > 60
    1 = transitional  31 <= RUL <= 60
    2 = critical      RUL <= 30
    """
    if rul > 60:
        return 0
    elif rul > 30:
        return 1
    else:
        return 2


def count_params(model):
    params = sum(p.numel() for p in model.parameters())
    kb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024.0
    return params, kb

In [8]:
# ============================================================
# Dataset paths
# ============================================================

CMAPSS_DIR = "/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps"
BATTERY_DIR = "/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset"
CWRU_DIR = "/kaggle/input/datasets/sufian79/cwru-mat-full-dataset"

SENSORS = [6, 7, 8, 11, 12, 13, 15, 16, 17, 18, 19, 21, 24, 25]
OP_COLS = [2, 3, 4]
RUL_CAP = 125


# ============================================================
# C-MAPSS loader
# ============================================================

def normalize_cmapss(train_df, test_df, fd):
    """
    FD001/FD003: global normalization.
    FD002/FD004: condition-aware normalization using KMeans on operating settings.
    """

    if fd in [1, 3]:
        mu = train_df[SENSORS].mean()
        std = train_df[SENSORS].std().replace(0, 1)

        train_df = train_df.copy()
        test_df = test_df.copy()

        train_df[SENSORS] = (train_df[SENSORS] - mu) / std
        test_df[SENSORS] = (test_df[SENSORS] - mu) / std

        return train_df, test_df

    km = KMeans(n_clusters=6, random_state=42, n_init=10)
    km.fit(train_df[OP_COLS].values)

    train_df = train_df.copy()
    test_df = test_df.copy()

    tr_labels = km.predict(train_df[OP_COLS].values)
    te_labels = km.predict(test_df[OP_COLS].values)

    stats = {}

    for c in range(6):
        mask = tr_labels == c

        if mask.sum() < 2:
            continue

        mu = train_df.loc[mask, SENSORS].mean()
        std = train_df.loc[mask, SENSORS].std().replace(0, 1)

        stats[c] = (mu, std)
        train_df.loc[mask, SENSORS] = (train_df.loc[mask, SENSORS] - mu) / std

    for c in range(6):
        mask = te_labels == c

        if mask.sum() == 0 or c not in stats:
            continue

        mu, std = stats[c]
        test_df.loc[mask, SENSORS] = (test_df.loc[mask, SENSORS] - mu) / std

    return train_df, test_df


def load_cmapss(fd, seed):
    train_df = pd.read_csv(
        f"{CMAPSS_DIR}/train_FD{fd:03d}.txt",
        sep=r"\s+",
        header=None
    )

    test_df = pd.read_csv(
        f"{CMAPSS_DIR}/test_FD{fd:03d}.txt",
        sep=r"\s+",
        header=None
    )

    rul_df = pd.read_csv(
        f"{CMAPSS_DIR}/RUL_FD{fd:03d}.txt",
        header=None,
        names=["RUL"]
    )

    max_cycles = train_df.groupby(0)[1].max().rename("max_cycle")
    train_df = train_df.join(max_cycles, on=0)
    train_df["RUL"] = (train_df["max_cycle"] - train_df[1]).clip(upper=RUL_CAP)
    train_df["state"] = train_df["RUL"].apply(rul_to_state)

    train_df, test_df = normalize_cmapss(train_df, test_df, fd)

    def make_windows(df, engine_list):
        Xs, yr, ys = [], [], []

        for eng in engine_list:
            sub = df[df[0] == eng].reset_index(drop=True)
            feats = sub[SENSORS].values.astype(np.float32)
            n = len(feats)

            if n < SEQ:
                pad = np.zeros((SEQ - n, len(SENSORS)), dtype=np.float32)
                feats = np.vstack([pad, feats])
                n = SEQ

            for t in range(n - SEQ + 1):
                Xs.append(feats[t:t + SEQ])
                yr.append(float(sub.loc[min(t + SEQ - 1, len(sub) - 1), "RUL"]))
                ys.append(int(sub.loc[min(t + SEQ - 1, len(sub) - 1), "state"]))

        return (
            torch.tensor(np.array(Xs), dtype=torch.float32),
            torch.tensor(yr, dtype=torch.float32),
            torch.tensor(ys, dtype=torch.long)
        )

    engines = train_df[0].unique()
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(engines))

    n_tr = int(len(engines) * 0.8)

    tr_eng = engines[idx[:n_tr]]
    vl_eng = engines[idx[n_tr:]]

    Xtr, ytr_r, ytr_s = make_windows(train_df, tr_eng)
    Xvl, yvl_r, yvl_s = make_windows(train_df, vl_eng)

    rul_test = rul_df["RUL"].values.astype(np.float32)

    Xte_list = []
    yte_r_list = []
    yte_s_list = []

    for i, eng in enumerate(test_df[0].unique()):
        sub = test_df[test_df[0] == eng].reset_index(drop=True)
        feats = sub[SENSORS].values.astype(np.float32)

        if len(feats) < SEQ:
            pad = np.zeros((SEQ - len(feats), len(SENSORS)), dtype=np.float32)
            feats = np.vstack([pad, feats])

        Xte_list.append(feats[-SEQ:])

        rv = float(rul_test[i])
        yte_r_list.append(rv)
        yte_s_list.append(rul_to_state(rv))

    Xte = torch.tensor(np.array(Xte_list), dtype=torch.float32)
    yte_r = torch.tensor(yte_r_list, dtype=torch.float32)
    yte_s = torch.tensor(yte_s_list, dtype=torch.long)

    pin = DEVICE == "cuda"

    tr = DataLoader(
        TensorDataset(Xtr, ytr_r, ytr_s),
        batch_size=BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=pin
    )

    va = DataLoader(
        TensorDataset(Xvl, yvl_r, yvl_s),
        batch_size=BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=pin
    )

    te = DataLoader(
        TensorDataset(Xte, yte_r, yte_s),
        batch_size=BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=pin
    )

    return tr, va, te


# ============================================================
# Battery LOCO loader with improved robust preprocessing
# ============================================================

def build_battery_cell_sequences():
    meta = pd.read_csv(os.path.join(BATTERY_DIR, "metadata.csv"))
    discharge = meta[meta["type"] == "discharge"].copy()

    batteries = sorted(discharge["battery_id"].unique())
    all_cells = {}

    for bid in batteries:
        rows = discharge[discharge["battery_id"] == bid].sort_values("start_time")

        cap_vals = pd.to_numeric(rows["Capacity"], errors="coerce").dropna().values
        cap_vals = cap_vals.astype(np.float32)

        n = len(cap_vals)

        if n < SEQ + 10:
            continue

        # Robust smoothing to reduce capacity-regeneration noise
        cap_smooth = (
            pd.Series(cap_vals)
            .rolling(window=5, center=True, min_periods=1)
            .median()
            .values
        )

        # Use early-cycle median as rated capacity estimate
        max_cap = float(np.median(cap_smooth[:10]))

        if max_cap <= 0:
            max_cap = float(cap_smooth.max())

        soh = np.clip(cap_smooth / max_cap, 0.0, 1.0)

        # Robust EOL detection: require 3 consecutive cycles below 0.8
        below = soh < 0.8
        eol_idx = n

        for i in range(len(below) - 2):
            if below[i] and below[i + 1] and below[i + 2]:
                eol_idx = i
                break

        # Fallback: first below-threshold cycle
        if eol_idx == n and below.any():
            eol_idx = int(np.where(below)[0][0])

        rul = np.clip(eol_idx - np.arange(n), 0, BATTERY_HORIZON).astype(np.float32)

        delta = np.diff(soh, prepend=soh[0])
        cyc_norm = np.clip(np.arange(n, dtype=np.float32) / BATTERY_HORIZON, 0.0, 1.0)

        feats = np.stack(
            [
                soh,
                delta,
                cyc_norm,
                cap_smooth / (max_cap + 1e-8)
            ],
            axis=1
        ).astype(np.float32)

        states = np.array([rul_to_state(float(r)) for r in rul], dtype=np.int64)

        seqs = []

        for t in range(n - SEQ + 1):
            seqs.append(
                (
                    feats[t:t + SEQ],
                    float(rul[t + SEQ - 1]),
                    int(states[t + SEQ - 1])
                )
            )

        if len(seqs) > 0:
            all_cells[str(bid)] = seqs

    return all_cells


def build_battery_folds():
    all_cells = build_battery_cell_sequences()
    folds = []

    for test_cell in all_cells.keys():
        tr_seqs = []
        vl_seqs = []

        for cell_id, seqs in all_cells.items():
            if cell_id == test_cell:
                continue

            split_idx = int(len(seqs) * 0.8)

            tr_seqs.extend(seqs[:split_idx])
            vl_seqs.extend(seqs[split_idx:])

        te_seqs = all_cells[test_cell]

        if len(tr_seqs) < 10 or len(vl_seqs) < 5 or len(te_seqs) < 5:
            continue

        rng = np.random.RandomState(42)
        rng.shuffle(tr_seqs)

        y_train = np.array([s[1] for s in tr_seqs], dtype=np.float32)

        y_mean = float(y_train.mean())
        y_std = float(y_train.std() + 1e-8)

        state_train = np.array([s[2] for s in tr_seqs], dtype=np.int64)
        counts = np.bincount(state_train, minlength=3).astype(np.float32)

        weights = 1.0 / (counts + 1.0)
        weights = weights / weights.sum() * 3.0

        class_weight = torch.tensor(weights, dtype=torch.float32)

        def to_loader(seqs, shuffle):
            X = torch.tensor(np.array([s[0] for s in seqs]), dtype=torch.float32)
            yr = torch.tensor([s[1] for s in seqs], dtype=torch.float32)
            ys = torch.tensor([s[2] for s in seqs], dtype=torch.long)

            bs = min(BATCH, len(X))

            return DataLoader(
                TensorDataset(X, yr, ys),
                batch_size=bs,
                shuffle=shuffle,
                num_workers=0
            )

        folds.append(
            {
                "test_cell": test_cell,
                "tr": to_loader(tr_seqs, True),
                "vl": to_loader(vl_seqs, False),
                "te": to_loader(te_seqs, False),
                "y_mean": y_mean,
                "y_std": y_std,
                "class_weight": class_weight
            }
        )

    return folds


# ============================================================
# CWRU record-level loader
# ============================================================

CWRU_FILES = {
    "97.mat": 0, "98.mat": 0, "99.mat": 0, "100.mat": 0,
    "105.mat": 1, "106.mat": 1, "107.mat": 1, "108.mat": 1,
    "169.mat": 2, "170.mat": 2, "171.mat": 2, "172.mat": 2,
    "209.mat": 3, "210.mat": 3, "211.mat": 3, "212.mat": 3,
    "118.mat": 4, "119.mat": 4, "120.mat": 4, "121.mat": 4,
    "185.mat": 5, "186.mat": 5, "187.mat": 5, "188.mat": 5,
    "222.mat": 6, "223.mat": 6, "224.mat": 6, "225.mat": 6,
    "130.mat": 7, "131.mat": 7, "132.mat": 7, "133.mat": 7,
    "197.mat": 8, "198.mat": 8, "199.mat": 8, "200.mat": 8,
    "234.mat": 9, "235.mat": 9, "236.mat": 9, "237.mat": 9,
}


def find_cwru_file(fname):
    for root, _, files in os.walk(CWRU_DIR):
        if fname in files:
            return os.path.join(root, fname)
    return None


def load_cwru_split(seed):
    class_files = {c: [] for c in range(10)}

    for fname, cls_id in CWRU_FILES.items():
        class_files[cls_id].append(fname)

    rng = np.random.RandomState(seed)

    tr_files = []
    vl_files = []
    te_files = []

    for cls_id, files in class_files.items():
        files = files.copy()
        rng.shuffle(files)

        tr_files.extend(files[:2])
        vl_files.extend(files[2:3])
        te_files.extend(files[3:4])

    def extract_windows(file_list):
        X, y = [], []

        for fname in file_list:
            fpath = find_cwru_file(fname)

            if fpath is None:
                continue

            mat = scipy.io.loadmat(fpath)
            keys = [k for k in mat.keys() if "DE_time" in k]

            if len(keys) == 0:
                continue

            sig = mat[keys[0]].flatten().astype(np.float32)

            if len(sig) < CWRU_SEQ:
                continue

            sig = (sig - sig.mean()) / (sig.std() + 1e-8)

            stride = CWRU_SEQ // 2

            for start in range(0, len(sig) - CWRU_SEQ + 1, stride):
                X.append(sig[start:start + CWRU_SEQ].reshape(CWRU_SEQ, 1))
                y.append(CWRU_FILES[fname])

        if len(X) == 0:
            return torch.empty(0, CWRU_SEQ, 1), torch.empty(0, dtype=torch.long)

        return (
            torch.tensor(np.array(X), dtype=torch.float32),
            torch.tensor(y, dtype=torch.long)
        )

    Xtr, ytr = extract_windows(tr_files)
    Xvl, yvl = extract_windows(vl_files)
    Xte, yte = extract_windows(te_files)

    if len(Xtr) == 0:
        raise RuntimeError("CWRU training set is empty. Check dataset path.")

    bs_tr = min(BATCH, len(Xtr))
    bs_vl = min(BATCH, len(Xvl))
    bs_te = min(BATCH, len(Xte))

    tr = DataLoader(
        TensorDataset(Xtr, ytr),
        batch_size=bs_tr,
        shuffle=True,
        num_workers=0
    )

    va = DataLoader(
        TensorDataset(Xvl, yvl),
        batch_size=bs_vl,
        shuffle=False,
        num_workers=0
    )

    te = DataLoader(
        TensorDataset(Xte, yte),
        batch_size=bs_te,
        shuffle=False,
        num_workers=0
    )

    return tr, va, te

In [9]:
# ============================================================
# Model blocks
# ============================================================

class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()

        self.conv = nn.Conv1d(
            d,
            d,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
            padding_mode="zeros"
        )

        self.bn = nn.BatchNorm1d(d)
        self.act = nn.GELU()

    def forward(self, x):
        return x + self.act(self.bn(self.conv(x)))


class CrossSensorGate(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())

    def forward(self, x):
        g = self.gate(x.mean(dim=1, keepdim=True))
        return x * g


class SEAttention(nn.Module):
    def __init__(self, d, reduction=2):
        super().__init__()

        hidden = max(2, d // reduction)

        self.fc = nn.Sequential(
            nn.Linear(d, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d),
            nn.Sigmoid()
        )

    def forward(self, x):
        g = self.fc(x.mean(dim=1, keepdim=True))
        return x * g


class ECAAttention(nn.Module):
    """
    Fixed ECA implementation.
    Input x shape: (B, T, d)
    """
    def __init__(self, d, k_size=3):
        super().__init__()

        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=1,
            kernel_size=k_size,
            padding=(k_size - 1) // 2,
            bias=False
        )

    def forward(self, x):
        # x: (B, T, d)
        y = x.mean(dim=1)              # (B, d)
        y = y.unsqueeze(1)             # (B, 1, d)
        y = self.conv(y)               # (B, 1, d)
        y = torch.sigmoid(y)           # (B, 1, d)

        return x * y


class NanoSentry(nn.Module):
    def __init__(self, n_sensors=14, d=24, n_classes=3, attn_type="CSG"):
        super().__init__()

        self.embed = nn.Linear(n_sensors, d)

        self.tcn = nn.Sequential(
            DilatedBlock(d, 1),
            DilatedBlock(d, 2),
            DilatedBlock(d, 4)
        )

        if attn_type == "CSG":
            self.gate = CrossSensorGate(d)
        elif attn_type == "SE":
            self.gate = SEAttention(d)
        elif attn_type == "ECA":
            self.gate = ECAAttention(d)
        else:
            self.gate = nn.Identity()

        self.skip = nn.Linear(n_sensors, d)

        self.gru = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)

        self.rul_head = nn.Sequential(
            nn.Linear(d, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )

        self.state_head = nn.Sequential(
            nn.Linear(d, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        h = self.embed(x).permute(0, 2, 1)
        h = self.tcn(h).permute(0, 2, 1)

        h = self.gate(h) + self.skip(x)

        _, hT = self.gru(h)
        hT = hT.squeeze(0)

        rul = self.rul_head(hT).squeeze(-1)
        state = self.state_head(hT)

        return rul, state


class NanoSentryAblation(nn.Module):
    def __init__(
        self,
        n_sensors=14,
        d=24,
        n_classes=3,
        no_gate=False,
        no_skip=False,
        no_dilation=False
    ):
        super().__init__()

        self.no_gate = no_gate
        self.no_skip = no_skip

        self.embed = nn.Linear(n_sensors, d)

        dilations = [1, 1, 1] if no_dilation else [1, 2, 4]

        self.tcn = nn.Sequential(
            *[DilatedBlock(d, dil) for dil in dilations]
        )

        if not no_gate:
            self.gate = CrossSensorGate(d)

        if not no_skip:
            self.skip = nn.Linear(n_sensors, d)

        self.gru = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)

        self.rul_head = nn.Sequential(
            nn.Linear(d, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )

        self.state_head = nn.Sequential(
            nn.Linear(d, 32),
            nn.GELU(),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        h = self.embed(x).permute(0, 2, 1)
        h = self.tcn(h).permute(0, 2, 1)

        if not self.no_gate:
            h = self.gate(h)

        if not self.no_skip:
            h = h + self.skip(x)

        _, hT = self.gru(h)
        hT = hT.squeeze(0)

        rul = self.rul_head(hT).squeeze(-1)
        state = self.state_head(hT)

        return rul, state


class CNNBaseline(nn.Module):
    def __init__(
        self,
        n_channels=14,
        filters=32,
        n_classes=3,
        classification=False
    ):
        super().__init__()

        self.classification = classification

        self.net = nn.Sequential(
            nn.Conv1d(n_channels, filters, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(filters),
            nn.Conv1d(filters, filters * 2, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(filters * 2),
            nn.AdaptiveAvgPool1d(1)
        )

        self.rul_head = nn.Linear(filters * 2, 1)

        if classification:
            self.cls_head = nn.Linear(filters * 2, n_classes)
        else:
            self.cls_head = None

    def forward(self, x):
        h = self.net(x.permute(0, 2, 1)).squeeze(-1)

        rul = self.rul_head(h).squeeze(-1)

        if self.cls_head is not None:
            cls = self.cls_head(h)
        else:
            cls = torch.zeros(x.size(0), 3, device=x.device)

        return rul, cls


class LSTMBaseline(nn.Module):
    def __init__(
        self,
        n_channels=14,
        hidden=64,
        n_classes=3,
        classification=False
    ):
        super().__init__()

        self.classification = classification

        self.lstm = nn.LSTM(
            n_channels,
            hidden,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        self.rul_head = nn.Linear(hidden, 1)

        if classification:
            self.cls_head = nn.Linear(hidden, n_classes)
        else:
            self.cls_head = None

    def forward(self, x):
        _, (h, _) = self.lstm(x)

        z = h[-1]

        rul = self.rul_head(z).squeeze(-1)

        if self.cls_head is not None:
            cls = self.cls_head(z)
        else:
            cls = torch.zeros(x.size(0), 3, device=x.device)

        return rul, cls

In [10]:
# ============================================================
# Evaluation helpers
# ============================================================

def evaluate_regression(model, loader, y_mean=0.0, y_std=1.0):
    model.eval()

    preds = []
    trues = []

    with torch.no_grad():
        for batch in loader:
            X = batch[0].to(DEVICE)
            yr = batch[1]

            pr = model(X)[0].cpu().numpy()
            pr = pr * y_std + y_mean

            preds.extend(pr)
            trues.extend(yr.numpy())

    if len(preds) == 0:
        return {
            "rmse": np.nan,
            "mae": np.nan,
            "score": np.nan,
            "cz_rmse": np.nan
        }

    pr = np.array(preds, dtype=np.float64)
    tr = np.array(trues, dtype=np.float64)

    return {
        "rmse": float(np.sqrt(np.mean((pr - tr) ** 2))),
        "mae": float(np.mean(np.abs(pr - tr))),
        "score": nasa_score(pr, tr),
        "cz_rmse": critical_zone_rmse(pr, tr)
    }


def evaluate_classification(model, loader, num_classes):
    model.eval()

    preds = []
    trues = []

    with torch.no_grad():
        for batch in loader:
            X = batch[0].to(DEVICE)

            # CWRU loader: (X, y)
            # C-MAPSS/Battery loader: (X, yr, ys)
            if len(batch) == 2:
                ys = batch[1]
            else:
                ys = batch[2]

            logits = model(X)[1]

            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(ys.numpy())

    if len(preds) == 0:
        return {
            "acc": np.nan,
            "f1": np.nan
        }

    pred = np.array(preds)
    true = np.array(trues)

    acc = float((pred == true).mean() * 100.0)
    f1 = macro_f1(pred, true, num_classes)

    return {
        "acc": acc,
        "f1": f1
    }


# ============================================================
# C-MAPSS training
# ============================================================

def train_cmapss(
    model,
    tr,
    va,
    te,
    multi=True,
    num_classes=3,
    epochs=120,
    patience=25,
    lr=1e-3
):
    model = model.to(DEVICE)

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)

    mse = nn.MSELoss()
    ce = nn.CrossEntropyLoss()

    best_val = float("inf")
    best_sd = None
    wait = 0

    for epoch in range(epochs):
        model.train()

        for X, yr, ys in tr:
            X = X.to(DEVICE)
            yr = yr.to(DEVICE)
            ys = ys.to(DEVICE)

            opt.zero_grad()

            pr, ps = model(X)

            loss = mse(pr, yr)

            if multi:
                loss = loss + 0.15 * ce(ps, ys)

            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()

        val_metrics = evaluate_regression(model, va)
        val_rmse = val_metrics["rmse"]

        if val_rmse < best_val:
            best_val = val_rmse
            best_sd = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    if best_sd is not None:
        model.load_state_dict(best_sd)

    metrics = evaluate_regression(model, te)

    if multi:
        cls_metrics = evaluate_classification(model, te, num_classes)
        metrics["acc"] = cls_metrics["acc"]
        metrics["f1"] = cls_metrics["f1"]
    else:
        metrics["acc"] = np.nan
        metrics["f1"] = np.nan

    return metrics


# ============================================================
# Battery training with target scaling + Huber loss
# ============================================================

def train_battery(
    model,
    tr,
    va,
    te,
    y_mean,
    y_std,
    class_weight=None,
    multi=True,
    num_classes=3,
    epochs=120,
    patience=25,
    lr=1e-3
):
    model = model.to(DEVICE)

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)

    huber = nn.HuberLoss(delta=1.0)

    if class_weight is not None:
        ce = nn.CrossEntropyLoss(weight=class_weight.to(DEVICE))
    else:
        ce = nn.CrossEntropyLoss()

    best_val = float("inf")
    best_sd = None
    wait = 0

    for epoch in range(epochs):
        model.train()

        for X, yr, ys in tr:
            X = X.to(DEVICE)
            yr = yr.to(DEVICE)
            ys = ys.to(DEVICE)

            yr_scaled = (yr - y_mean) / y_std

            opt.zero_grad()

            pr, ps = model(X)

            loss = huber(pr, yr_scaled)

            if multi:
                loss = loss + 0.15 * ce(ps, ys)

            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()

        val_metrics = evaluate_regression(model, va, y_mean, y_std)
        val_rmse = val_metrics["rmse"]

        if val_rmse < best_val:
            best_val = val_rmse
            best_sd = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    if best_sd is not None:
        model.load_state_dict(best_sd)

    metrics = evaluate_regression(model, te, y_mean, y_std)

    if multi:
        cls_metrics = evaluate_classification(model, te, num_classes)
        metrics["acc"] = cls_metrics["acc"]
        metrics["f1"] = cls_metrics["f1"]
    else:
        metrics["acc"] = np.nan
        metrics["f1"] = np.nan

    return metrics


# ============================================================
# Ridge helper
# ============================================================

def loader_to_Xy(loader):
    Xs = []
    ys = []

    for batch in loader:
        Xs.append(batch[0])
        ys.append(batch[1])

    return torch.cat(Xs, dim=0), torch.cat(ys, dim=0)


def window_summary_features(X):
    X = X.numpy()

    return np.concatenate(
        [
            X.mean(axis=1),
            X.std(axis=1),
            X.min(axis=1),
            X.max(axis=1)
        ],
        axis=1
    )


def run_ridge_regression(tr, te):
    Xtr, ytr = loader_to_Xy(tr)
    Xte, yte = loader_to_Xy(te)

    Xtr_s = window_summary_features(Xtr)
    Xte_s = window_summary_features(Xte)

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr_s)
    Xte_s = scaler.transform(Xte_s)

    model = Ridge(alpha=1.0)
    model.fit(Xtr_s, ytr.numpy())

    pred = model.predict(Xte_s)
    true = yte.numpy()

    return {
        "rmse": float(np.sqrt(np.mean((pred - true) ** 2))),
        "mae": float(np.mean(np.abs(pred - true))),
        "score": nasa_score(pred, true),
        "cz_rmse": critical_zone_rmse(pred, true),
        "acc": np.nan,
        "f1": np.nan
    }


# ============================================================
# CWRU improved training
# ============================================================

def augment_batch(X):
    X = X.clone()

    # Gaussian noise
    noise = 0.01 * torch.randn_like(X)

    # Amplitude scaling
    scale = 1.0 + 0.05 * torch.randn(X.size(0), 1, 1, device=X.device)

    X = (X + noise) * scale

    # Time shift
    shifts = torch.randint(-64, 65, (X.size(0),), device=X.device)

    for i in range(X.size(0)):
        X[i] = torch.roll(X[i], shifts=int(shifts[i]), dims=0)

    return X


def train_cwru(
    model,
    tr,
    va,
    te,
    num_classes=10,
    epochs=80,
    patience=20,
    lr=1e-3,
    augment=True,
    label_smoothing=0.1
):
    model = model.to(DEVICE)

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)

    ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    best_val = -1.0
    best_sd = None
    wait = 0

    for epoch in range(epochs):
        model.train()

        for X, ys in tr:
            X = X.to(DEVICE)
            ys = ys.to(DEVICE)

            if augment:
                X = augment_batch(X)

            opt.zero_grad()

            _, logits = model(X)

            loss = ce(logits, ys)

            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()

        val_metrics = evaluate_classification(model, va, num_classes)
        val_f1 = val_metrics["f1"]

        if np.isnan(val_f1):
            val_f1 = -1.0

        if val_f1 > best_val:
            best_val = val_f1
            best_sd = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    if best_sd is not None:
        model.load_state_dict(best_sd)

    metrics = evaluate_classification(model, te, num_classes)

    return metrics

In [11]:
# ============================================================
# 1. Attention comparison: CSG vs SE vs ECA vs None
# ============================================================

print("\n=== Attention Comparison ===")

for fd in [1, 3]:
    subset = f"FD{fd:03d}"

    for seed in BASE_SEEDS:
        try:
            tr, va, te = load_cmapss(fd, seed)
        except Exception as e:
            log_error(f"C-MAPSS load failed | {subset} | seed {seed} | {e}")
            continue

        for attn in ["CSG", "SE", "ECA", "None"]:
            model_name = f"NanoSentry_{attn}"

            if is_done("C-MAPSS", subset, model_name, "multi_task", seed):
                continue

            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = NanoSentry(
                    n_sensors=14,
                    d=24,
                    n_classes=3,
                    attn_type=attn
                )

                metrics = train_cmapss(
                    model,
                    tr,
                    va,
                    te,
                    multi=True,
                    num_classes=3
                )

                params, kb = count_params(model)

                save_result({
                    "dataset": "C-MAPSS",
                    "subset": subset,
                    "model": model_name,
                    "variant": "multi_task",
                    "seed": seed,
                    "fold": "",
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"],
                    "score": metrics["score"],
                    "cz_rmse": metrics["cz_rmse"],
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"Attention | {model_name} | {subset} | seed {seed} | {e}")
            finally:
                cleanup()


# ============================================================
# 2. Ablation study under corrected C-MAPSS protocol
# ============================================================

print("\n=== Ablation Study ===")

ablation_variants = [
    ("full", {}),
    ("no_gate", {"no_gate": True}),
    ("no_skip", {"no_skip": True}),
    ("no_dilation", {"no_dilation": True})
]

for fd in [1, 3]:
    subset = f"FD{fd:03d}"

    for seed in BASE_SEEDS:
        try:
            tr, va, te = load_cmapss(fd, seed)
        except Exception as e:
            log_error(f"C-MAPSS ablation load failed | {subset} | seed {seed} | {e}")
            continue

        for variant_name, flags in ablation_variants:
            if is_done("C-MAPSS", subset, "NanoSentry_Ablation", variant_name, seed):
                continue

            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = NanoSentryAblation(
                    n_sensors=14,
                    d=24,
                    n_classes=3,
                    **flags
                )

                metrics = train_cmapss(
                    model,
                    tr,
                    va,
                    te,
                    multi=False,
                    num_classes=3
                )

                params, kb = count_params(model)

                save_result({
                    "dataset": "C-MAPSS",
                    "subset": subset,
                    "model": "NanoSentry_Ablation",
                    "variant": variant_name,
                    "seed": seed,
                    "fold": "",
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"],
                    "score": metrics["score"],
                    "cz_rmse": metrics["cz_rmse"],
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"Ablation | {variant_name} | {subset} | seed {seed} | {e}")
            finally:
                cleanup()


# ============================================================
# 3. Improved Battery LOCO experiments
# ============================================================

print("\n=== Improved Battery LOCO ===")

try:
    battery_folds = build_battery_folds()
except Exception as e:
    battery_folds = []
    log_error(f"Battery fold construction failed | {e}")

for fold in battery_folds:
    fold_id = fold["test_cell"]

    tr = fold["tr"]
    va = fold["vl"]
    te = fold["te"]

    y_mean = fold["y_mean"]
    y_std = fold["y_std"]
    class_weight = fold["class_weight"]

    # NanoSentry multi-task
    if not is_done("Battery", fold_id, "NanoSentry", "multi_task", 42, fold_id):
        try:
            torch.manual_seed(42)
            np.random.seed(42)

            model = NanoSentry(
                n_sensors=4,
                d=24,
                n_classes=3,
                attn_type="CSG"
            )

            metrics = train_battery(
                model,
                tr,
                va,
                te,
                y_mean=y_mean,
                y_std=y_std,
                class_weight=class_weight,
                multi=True,
                num_classes=3
            )

            params, kb = count_params(model)

            save_result({
                "dataset": "Battery",
                "subset": fold_id,
                "model": "NanoSentry",
                "variant": "multi_task",
                "seed": 42,
                "fold": fold_id,
                "rmse": metrics["rmse"],
                "mae": metrics["mae"],
                "score": metrics["score"],
                "cz_rmse": metrics["cz_rmse"],
                "acc": metrics["acc"],
                "f1": metrics["f1"],
                "params": params,
                "kb": kb
            })

        except Exception as e:
            log_error(f"Battery NanoSentry multi_task | {fold_id} | {e}")
        finally:
            cleanup()

    # NanoSentry single-task
    if not is_done("Battery", fold_id, "NanoSentry", "single_task", 42, fold_id):
        try:
            torch.manual_seed(42)
            np.random.seed(42)

            model = NanoSentry(
                n_sensors=4,
                d=24,
                n_classes=3,
                attn_type="CSG"
            )

            metrics = train_battery(
                model,
                tr,
                va,
                te,
                y_mean=y_mean,
                y_std=y_std,
                class_weight=None,
                multi=False,
                num_classes=3
            )

            params, kb = count_params(model)

            save_result({
                "dataset": "Battery",
                "subset": fold_id,
                "model": "NanoSentry",
                "variant": "single_task",
                "seed": 42,
                "fold": fold_id,
                "rmse": metrics["rmse"],
                "mae": metrics["mae"],
                "score": metrics["score"],
                "cz_rmse": metrics["cz_rmse"],
                "acc": metrics["acc"],
                "f1": metrics["f1"],
                "params": params,
                "kb": kb
            })

        except Exception as e:
            log_error(f"Battery NanoSentry single_task | {fold_id} | {e}")
        finally:
            cleanup()

    # Ridge
    if not is_done("Battery", fold_id, "Ridge", "single_task", 42, fold_id):
        try:
            metrics = run_ridge_regression(tr, te)

            save_result({
                "dataset": "Battery",
                "subset": fold_id,
                "model": "Ridge",
                "variant": "single_task",
                "seed": 42,
                "fold": fold_id,
                "rmse": metrics["rmse"],
                "mae": metrics["mae"],
                "score": metrics["score"],
                "cz_rmse": metrics["cz_rmse"],
                "acc": "",
                "f1": "",
                "params": 0,
                "kb": 0
            })

        except Exception as e:
            log_error(f"Battery Ridge | {fold_id} | {e}")
        finally:
            cleanup()

    # CNN-32
    if not is_done("Battery", fold_id, "CNN-32", "single_task", 42, fold_id):
        try:
            torch.manual_seed(42)
            np.random.seed(42)

            model = CNNBaseline(
                n_channels=4,
                filters=32,
                n_classes=3,
                classification=False
            )

            metrics = train_battery(
                model,
                tr,
                va,
                te,
                y_mean=y_mean,
                y_std=y_std,
                class_weight=None,
                multi=False,
                num_classes=3
            )

            params, kb = count_params(model)

            save_result({
                "dataset": "Battery",
                "subset": fold_id,
                "model": "CNN-32",
                "variant": "single_task",
                "seed": 42,
                "fold": fold_id,
                "rmse": metrics["rmse"],
                "mae": metrics["mae"],
                "score": metrics["score"],
                "cz_rmse": metrics["cz_rmse"],
                "acc": metrics["acc"],
                "f1": metrics["f1"],
                "params": params,
                "kb": kb
            })

        except Exception as e:
            log_error(f"Battery CNN-32 | {fold_id} | {e}")
        finally:
            cleanup()

    # LSTM-64
    if not is_done("Battery", fold_id, "LSTM-64", "single_task", 42, fold_id):
        try:
            torch.manual_seed(42)
            np.random.seed(42)

            model = LSTMBaseline(
                n_channels=4,
                hidden=64,
                n_classes=3,
                classification=False
            )

            metrics = train_battery(
                model,
                tr,
                va,
                te,
                y_mean=y_mean,
                y_std=y_std,
                class_weight=None,
                multi=False,
                num_classes=3
            )

            params, kb = count_params(model)

            save_result({
                "dataset": "Battery",
                "subset": fold_id,
                "model": "LSTM-64",
                "variant": "single_task",
                "seed": 42,
                "fold": fold_id,
                "rmse": metrics["rmse"],
                "mae": metrics["mae"],
                "score": metrics["score"],
                "cz_rmse": metrics["cz_rmse"],
                "acc": metrics["acc"],
                "f1": metrics["f1"],
                "params": params,
                "kb": kb
            })

        except Exception as e:
            log_error(f"Battery LSTM-64 | {fold_id} | {e}")
        finally:
            cleanup()


# ============================================================
# 4. Improved CWRU record-level experiments
# ============================================================

if RUN_CWRU_POLISH:
    print("\n=== Improved CWRU Record-Level Split ===")

    for seed in CWRU_SEEDS:
        try:
            tr, va, te = load_cwru_split(seed)
        except Exception as e:
            log_error(f"CWRU load failed | seed {seed} | {e}")
            continue

        models = [
            ("NanoSentry", NanoSentry(1, 24, 10, "CSG")),
            ("CNN-32", CNNBaseline(1, 32, 10, True)),
            ("LSTM-64", LSTMBaseline(1, 64, 10, True))
        ]

        for model_name, model in models:
            if is_done("CWRU", "10-class", model_name, "classifier_aug", seed):
                continue

            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                metrics = train_cwru(
                    model,
                    tr,
                    va,
                    te,
                    num_classes=10,
                    epochs=80,
                    patience=20,
                    augment=True,
                    label_smoothing=0.1
                )

                params, kb = count_params(model)

                save_result({
                    "dataset": "CWRU",
                    "subset": "10-class",
                    "model": model_name,
                    "variant": "classifier_aug",
                    "seed": seed,
                    "fold": "",
                    "rmse": "",
                    "mae": "",
                    "score": "",
                    "cz_rmse": "",
                    "acc": metrics["acc"],
                    "f1": metrics["f1"],
                    "params": params,
                    "kb": kb
                })

            except Exception as e:
                log_error(f"CWRU | {model_name} | seed {seed} | {e}")
            finally:
                cleanup()

else:
    print("\nCWRU polish skipped because GPU is not available.")


# ============================================================
# Final summary
# ============================================================

print("\n=== POLISH EXPERIMENTS FINISHED ===")
print(f"Results file: {RESULTS_FILE}")
print(f"Error file: {ERROR_FILE}")

if os.path.exists(RESULTS_FILE):
    try:
        df = pd.read_csv(RESULTS_FILE)

        print(f"Total completed polished rows: {len(df)}")

        numeric_cols = ["rmse", "mae", "score", "cz_rmse", "acc", "f1"]

        for c in numeric_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        summary = (
            df.groupby(["dataset", "model", "variant"])
            .agg(
                rmse_mean=("rmse", "mean"),
                rmse_std=("rmse", "std"),
                acc_mean=("acc", "mean"),
                acc_std=("acc", "std"),
                f1_mean=("f1", "mean"),
                f1_std=("f1", "std")
            )
            .reset_index()
        )

        print("\nSummary:")
        print(summary)

    except Exception as e:
        print(f"Could not print summary: {e}")


=== Attention Comparison ===
Saved: C-MAPSS | FD001 | NanoSentry_CSG | multi_task | seed 42 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_SE | multi_task | seed 42 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_ECA | multi_task | seed 42 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_None | multi_task | seed 42 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_CSG | multi_task | seed 7 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_SE | multi_task | seed 7 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_ECA | multi_task | seed 7 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_None | multi_task | seed 7 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_CSG | multi_task | seed 13 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_SE | multi_task | seed 13 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_ECA | multi_task | seed 13 | fold 
Saved: C-MAPSS | FD001 | NanoSentry_None | multi_task | seed 13 | fold 
Saved: C-MAPSS | FD003 | NanoSentry_CSG | multi_task | seed 42 | fold 
Saved: C-MAPSS | FD003 | NanoSentry_SE | multi_task

In [12]:
# ============================================================
# Battery Final Rescue — Safe LOCO Protocol
# ============================================================

import os
import csv
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Battery rescue device: {DEVICE}")

# ============================================================
# Config
# ============================================================

BATTERY_DIR = "/kaggle/input/datasets/patrickfleith/nasa-battery-dataset/cleaned_dataset"

SEQ = 64
BATCH = 64
H = 200.0
SEED = 42
EPOCHS = 150
PATIENCE = 30

RESULTS_FILE = "/kaggle/working/battery_final_results.csv"

FIELDNAMES = [
    "fold",
    "model",
    "variant",
    "rmse",
    "mae",
    "acc",
    "f1",
    "params",
    "kb"
]


# ============================================================
# Metrics
# ============================================================

def macro_f1(pred, true, num_classes=3):
    pred = np.array(pred)
    true = np.array(true)

    f1s = []

    for c in range(num_classes):
        tp = int(np.sum((pred == c) & (true == c)))
        fp = int(np.sum((pred == c) & (true != c)))
        fn = int(np.sum((pred != c) & (true == c)))

        if tp == 0:
            f1s.append(0.0)
            continue

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)

        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        f1s.append(f1)

    return float(np.mean(f1s)) if len(f1s) > 0 else np.nan


def count_params(model):
    params = sum(p.numel() for p in model.parameters())
    kb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024.0
    return params, kb


# ============================================================
# Models
# ============================================================

class DilatedBlock(nn.Module):
    def __init__(self, d, dilation):
        super().__init__()

        self.conv = nn.Conv1d(
            d,
            d,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
            padding_mode="zeros"
        )

        self.bn = nn.BatchNorm1d(d)
        self.act = nn.GELU()

    def forward(self, x):
        return x + self.act(self.bn(self.conv(x)))


class CrossSensorGate(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())

    def forward(self, x):
        g = self.gate(x.mean(dim=1, keepdim=True))
        return x * g


class NanoSentry(nn.Module):
    def __init__(self, n_sensors=4, d=24, n_classes=3):
        super().__init__()

        self.embed = nn.Linear(n_sensors, d)

        self.tcn = nn.Sequential(
            DilatedBlock(d, 1),
            DilatedBlock(d, 2),
            DilatedBlock(d, 4)
        )

        self.gate = CrossSensorGate(d)
        self.skip = nn.Linear(n_sensors, d)

        self.gru = nn.GRU(d, d, batch_first=True)
        nn.init.orthogonal_(self.gru.weight_hh_l0)

        self.rul_head = nn.Sequential(
            nn.Linear(d, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )

        self.state_head = nn.Sequential(
            nn.Linear(d, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        h = self.embed(x).permute(0, 2, 1)
        h = self.tcn(h).permute(0, 2, 1)

        h = self.gate(h) + self.skip(x)

        _, hT = self.gru(h)
        hT = hT.squeeze(0)

        rul = self.rul_head(hT).squeeze(-1)
        state = self.state_head(hT)

        return rul, state


class CNNBaseline(nn.Module):
    def __init__(self, n_channels=4, filters=32, n_classes=3):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(n_channels, filters, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(filters),
            nn.Conv1d(filters, filters * 2, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(filters * 2),
            nn.AdaptiveAvgPool1d(1)
        )

        self.rul_head = nn.Linear(filters * 2, 1)

        self.state_head = nn.Sequential(
            nn.Linear(filters * 2, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        h = self.net(x.permute(0, 2, 1)).squeeze(-1)

        rul = self.rul_head(h).squeeze(-1)
        state = self.state_head(h)

        return rul, state


class LSTMBaseline(nn.Module):
    def __init__(self, n_channels=4, hidden=64, n_classes=3):
        super().__init__()

        self.lstm = nn.LSTM(
            n_channels,
            hidden,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        self.rul_head = nn.Linear(hidden, 1)

        self.state_head = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        _, (h, _) = self.lstm(x)

        z = h[-1]

        rul = self.rul_head(z).squeeze(-1)
        state = self.state_head(z)

        return rul, state


# ============================================================
# Build battery cell sequences
# ============================================================

def build_cells():
    meta = pd.read_csv(os.path.join(BATTERY_DIR, "metadata.csv"))
    discharge = meta[meta["type"] == "discharge"].copy()

    batteries = sorted(discharge["battery_id"].unique())
    cells = {}

    for bid in batteries:
        rows = discharge[discharge["battery_id"] == bid].sort_values("start_time")

        cap_vals = pd.to_numeric(rows["Capacity"], errors="coerce").dropna().values
        cap_vals = cap_vals.astype(np.float32)

        n = len(cap_vals)

        if n < SEQ + 10:
            continue

        # Light smoothing only
        cap_smooth = (
            pd.Series(cap_vals)
            .rolling(window=3, center=True, min_periods=1)
            .mean()
            .values
        )

        max_cap = float(cap_smooth.max())

        if max_cap <= 0:
            continue

        soh = np.clip(cap_smooth / max_cap, 0.0, 1.0)

        # Safer EOL detection using a smoothed SOH curve
        soh_eval = (
            pd.Series(soh)
            .rolling(window=5, center=True, min_periods=1)
            .mean()
            .values
        )

        below = np.where(soh_eval < 0.8)[0]

        has_eol = len(below) > 0

        if has_eol:
            eol_idx = int(below[0])
        else:
            eol_idx = n

        rul = np.clip(eol_idx - np.arange(n), 0, H).astype(np.float32)

        delta = np.diff(soh, prepend=soh[0])
        cyc = np.clip(np.arange(n, dtype=np.float32) / H, 0.0, 1.0)

        feats = np.stack(
            [
                soh,
                delta,
                cyc,
                cap_smooth / max_cap
            ],
            axis=1
        ).astype(np.float32)

        seqs = []

        for t in range(n - SEQ + 1):
            idx = t + SEQ - 1

            rul_val = float(rul[idx])

            if rul_val > 60:
                state = 0
            elif rul_val > 30:
                state = 1
            else:
                state = 2

            seqs.append(
                (
                    feats[t:t + SEQ],
                    rul_val,
                    int(state)
                )
            )

        if len(seqs) > 0:
            cells[str(bid)] = {
                "seqs": seqs,
                "has_eol": has_eol
            }

    return cells


cells = build_cells()

all_cell_ids = list(cells.keys())
eol_cell_ids = [c for c in all_cell_ids if cells[c]["has_eol"]]

# If enough cells reached EOL, use only those.
# Otherwise, use all cells.
if len(eol_cell_ids) >= 8:
    use_cells = eol_cell_ids
    print(f"Using only cells with observed EOL: {len(use_cells)} cells")
else:
    use_cells = all_cell_ids
    print(f"Not enough EOL cells. Using all available cells: {len(use_cells)} cells")

print(f"Total cells found: {len(all_cell_ids)}")
print(f"Cells with EOL: {len(eol_cell_ids)}")
print(f"Cells used: {use_cells}")


# ============================================================
# Fold loaders
# ============================================================

def make_fold_loaders(test_cell):
    train_cells = [c for c in use_cells if c != test_cell]

    tr_seqs = []
    vl_seqs = []

    for c in train_cells:
        seqs = cells[c]["seqs"]
        split_idx = int(len(seqs) * 0.8)

        tr_seqs.extend(seqs[:split_idx])
        vl_seqs.extend(seqs[split_idx:])

    te_seqs = cells[test_cell]["seqs"]

    if len(tr_seqs) < 20 or len(vl_seqs) < 5 or len(te_seqs) < 5:
        return None

    # Feature scaler fitted only on training sequences
    Xtr_raw = np.array([s[0] for s in tr_seqs], dtype=np.float32)

    feat_scaler = StandardScaler()
    feat_scaler.fit(Xtr_raw.reshape(-1, Xtr_raw.shape[-1]))

    def transform(seqs):
        X = np.array([s[0] for s in seqs], dtype=np.float32)
        X = feat_scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape)

        yr = np.array([s[1] for s in seqs], dtype=np.float32)
        ys = np.array([s[2] for s in seqs], dtype=np.int64)

        return (
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(yr, dtype=torch.float32),
            torch.tensor(ys, dtype=torch.long)
        )

    Xtr, ytr, ys_tr = transform(tr_seqs)
    Xvl, yvl, ys_vl = transform(vl_seqs)
    Xte, yte, ys_te = transform(te_seqs)

    y_mean = float(ytr.mean())
    y_std = float(ytr.std() + 1e-8)

    counts = np.bincount(ys_tr.numpy(), minlength=3).astype(np.float32)
    weights = 1.0 / (counts + 1.0)
    weights = weights / weights.sum() * 3.0

    class_weight = torch.tensor(weights, dtype=torch.float32)

    def dl(X, yr, ys, shuffle):
        return DataLoader(
            TensorDataset(X, yr, ys),
            batch_size=min(BATCH, len(X)),
            shuffle=shuffle,
            num_workers=0
        )

    return {
        "tr": dl(Xtr, ytr, ys_tr, True),
        "vl": dl(Xvl, yvl, ys_vl, False),
        "te": dl(Xte, yte, ys_te, False),
        "y_mean": y_mean,
        "y_std": y_std,
        "class_weight": class_weight
    }


# ============================================================
# Evaluation and training
# ============================================================

def evaluate_net(model, loader, y_mean, y_std):
    model.eval()

    preds = []
    trues = []
    preds_s = []
    trues_s = []

    with torch.no_grad():
        for X, yr, ys in loader:
            X = X.to(DEVICE)

            pr, ps = model(X)

            pr = pr.cpu().numpy() * y_std + y_mean

            preds.extend(pr)
            trues.extend(yr.numpy())

            preds_s.extend(ps.argmax(1).cpu().numpy())
            trues_s.extend(ys.numpy())

    pred = np.array(preds, dtype=np.float64)
    true = np.array(trues, dtype=np.float64)

    rmse = float(np.sqrt(np.mean((pred - true) ** 2)))
    mae = float(np.mean(np.abs(pred - true)))

    pred_s = np.array(preds_s)
    true_s = np.array(trues_s)

    acc = float((pred_s == true_s).mean() * 100.0)
    f1 = macro_f1(pred_s, true_s, num_classes=3)

    return {
        "rmse": rmse,
        "mae": mae,
        "acc": acc,
        "f1": f1
    }


def train_net(model, fold, multi=False):
    model = model.to(DEVICE)

    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)

    mse = nn.MSELoss()

    if multi:
        ce = nn.CrossEntropyLoss(weight=fold["class_weight"].to(DEVICE))
    else:
        ce = None

    y_mean = fold["y_mean"]
    y_std = fold["y_std"]

    best_val = float("inf")
    best_sd = None
    wait = 0

    for epoch in range(EPOCHS):
        model.train()

        for X, yr, ys in fold["tr"]:
            X = X.to(DEVICE)
            yr = yr.to(DEVICE)
            ys = ys.to(DEVICE)

            yr_scaled = (yr - y_mean) / y_std

            opt.zero_grad()

            pr, ps = model(X)

            loss = mse(pr, yr_scaled)

            if multi:
                loss = loss + 0.15 * ce(ps, ys)

            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            opt.step()

        sched.step()

        val_metrics = evaluate_net(model, fold["vl"], y_mean, y_std)
        val_rmse = val_metrics["rmse"]

        if not np.isfinite(val_rmse):
            val_rmse = float("inf")

        if val_rmse < best_val:
            best_val = val_rmse
            best_sd = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    if best_sd is not None:
        model.load_state_dict(best_sd)

    test_metrics = evaluate_net(model, fold["te"], y_mean, y_std)

    return test_metrics


def loader_to_Xy(loader):
    Xs = []
    ys = []

    for batch in loader:
        Xs.append(batch[0])
        ys.append(batch[1])

    return torch.cat(Xs, dim=0), torch.cat(ys, dim=0)


def window_summary_features(X):
    X = X.numpy()

    return np.concatenate(
        [
            X.mean(axis=1),
            X.std(axis=1),
            X.min(axis=1),
            X.max(axis=1)
        ],
        axis=1
    )


def run_ridge(fold):
    Xtr, ytr = loader_to_Xy(fold["tr"])
    Xte, yte = loader_to_Xy(fold["te"])

    Xtr_s = window_summary_features(Xtr)
    Xte_s = window_summary_features(Xte)

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr_s)
    Xte_s = scaler.transform(Xte_s)

    y_mean = fold["y_mean"]
    y_std = fold["y_std"]

    ytr_scaled = (ytr.numpy() - y_mean) / y_std

    model = Ridge(alpha=1.0)
    model.fit(Xtr_s, ytr_scaled)

    pred_scaled = model.predict(Xte_s)
    pred = pred_scaled * y_std + y_mean
    true = yte.numpy()

    rmse = float(np.sqrt(np.mean((pred - true) ** 2)))
    mae = float(np.mean(np.abs(pred - true)))

    return {
        "rmse": rmse,
        "mae": mae,
        "acc": np.nan,
        "f1": np.nan
    }


# ============================================================
# Run all battery folds
# ============================================================

rows = []

for test_cell in use_cells:
    print(f"\nBattery fold: {test_cell}")

    fold = make_fold_loaders(test_cell)

    if fold is None:
        print(f"Skipping {test_cell}: not enough sequences.")
        continue

    # NanoSentry multi-task
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = NanoSentry(n_sensors=4, d=24, n_classes=3)
    metrics = train_net(model, fold, multi=True)
    params, kb = count_params(model)

    rows.append({
        "fold": test_cell,
        "model": "NanoSentry",
        "variant": "multi_task",
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "acc": metrics["acc"],
        "f1": metrics["f1"],
        "params": params,
        "kb": kb
    })

    print(
        f"  NanoSentry multi_task | "
        f"RMSE {metrics['rmse']:.4f} | "
        f"MAE {metrics['mae']:.4f} | "
        f"Acc {metrics['acc']:.2f}%"
    )

    # NanoSentry single-task
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = NanoSentry(n_sensors=4, d=24, n_classes=3)
    metrics = train_net(model, fold, multi=False)
    params, kb = count_params(model)

    rows.append({
        "fold": test_cell,
        "model": "NanoSentry",
        "variant": "single_task",
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "acc": "",
        "f1": "",
        "params": params,
        "kb": kb
    })

    print(
        f"  NanoSentry single_task | "
        f"RMSE {metrics['rmse']:.4f} | "
        f"MAE {metrics['mae']:.4f}"
    )

    # Ridge
    metrics = run_ridge(fold)

    rows.append({
        "fold": test_cell,
        "model": "Ridge",
        "variant": "single_task",
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "acc": "",
        "f1": "",
        "params": 0,
        "kb": 0
    })

    print(
        f"  Ridge | "
        f"RMSE {metrics['rmse']:.4f} | "
        f"MAE {metrics['mae']:.4f}"
    )

    # CNN-32
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = CNNBaseline(n_channels=4, filters=32, n_classes=3)
    metrics = train_net(model, fold, multi=False)
    params, kb = count_params(model)

    rows.append({
        "fold": test_cell,
        "model": "CNN-32",
        "variant": "single_task",
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "acc": "",
        "f1": "",
        "params": params,
        "kb": kb
    })

    print(
        f"  CNN-32 | "
        f"RMSE {metrics['rmse']:.4f} | "
        f"MAE {metrics['mae']:.4f}"
    )

    # LSTM-64
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = LSTMBaseline(n_channels=4, hidden=64, n_classes=3)
    metrics = train_net(model, fold, multi=False)
    params, kb = count_params(model)

    rows.append({
        "fold": test_cell,
        "model": "LSTM-64",
        "variant": "single_task",
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "acc": "",
        "f1": "",
        "params": params,
        "kb": kb
    })

    print(
        f"  LSTM-64 | "
        f"RMSE {metrics['rmse']:.4f} | "
        f"MAE {metrics['mae']:.4f}"
    )


# ============================================================
# Save and summarize
# ============================================================

with open(RESULTS_FILE, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(rows)

df = pd.DataFrame(rows)

for c in ["rmse", "mae", "acc", "f1"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n" + "=" * 70)
print("BATTERY FINAL RESCUE SUMMARY")
print("=" * 70)

summary = (
    df.groupby("model")
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        rmse_median=("rmse", "median"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std")
    )
    .reset_index()
)

print(summary)

print("\nPer-fold NanoSentry RMSE:")
nano_pivot = df[df["model"] == "NanoSentry"].pivot_table(
    index="fold",
    columns="variant",
    values="rmse"
)

print(nano_pivot)

print(f"\nSaved battery results to: {RESULTS_FILE}")

Battery rescue device: cuda
Using only cells with observed EOL: 12 cells
Total cells found: 13
Cells with EOL: 12
Cells used: ['B0005', 'B0006', 'B0007', 'B0018', 'B0033', 'B0034', 'B0036', 'B0042', 'B0043', 'B0044', 'B0054', 'B0055']

Battery fold: B0005
  NanoSentry multi_task | RMSE 0.0037 | MAE 0.0024 | Acc 100.00%
  NanoSentry single_task | RMSE 0.0044 | MAE 0.0037
  Ridge | RMSE 0.0374 | MAE 0.0307
  CNN-32 | RMSE 0.0232 | MAE 0.0204
  LSTM-64 | RMSE 0.0019 | MAE 0.0016

Battery fold: B0006
  NanoSentry multi_task | RMSE 0.0009 | MAE 0.0008 | Acc 100.00%
  NanoSentry single_task | RMSE 0.0040 | MAE 0.0033
  Ridge | RMSE 0.1141 | MAE 0.1049
  CNN-32 | RMSE 0.0865 | MAE 0.0778
  LSTM-64 | RMSE 0.0039 | MAE 0.0035

Battery fold: B0007
  NanoSentry multi_task | RMSE 0.0064 | MAE 0.0022 | Acc 100.00%
  NanoSentry single_task | RMSE 0.0029 | MAE 0.0026
  Ridge | RMSE 0.0350 | MAE 0.0291
  CNN-32 | RMSE 0.0626 | MAE 0.0480
  LSTM-64 | RMSE 0.0008 | MAE 0.0005

Battery fold: B0018
  Nano

In [13]:
import pandas as pd

df = pd.read_csv("/kaggle/working/polished_results.csv")

for c in ["rmse", "mae", "score", "cz_rmse", "acc", "f1"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("=" * 80)
print("ATTENTION COMPARISON BY SUBSET")
print("=" * 80)

att = df[
    (df["dataset"] == "C-MAPSS") &
    (df["model"].str.startswith("NanoSentry_")) &
    (df["variant"] == "multi_task")
]

att_summary = (
    att.groupby(["subset", "model"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        cz_mean=("cz_rmse", "mean"),
        cz_std=("cz_rmse", "std"),
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std")
    )
    .reset_index()
)

print(att_summary)

print("\n" + "=" * 80)
print("ABLATION BY SUBSET")
print("=" * 80)

abl = df[
    (df["dataset"] == "C-MAPSS") &
    (df["model"] == "NanoSentry_Ablation")
]

abl_summary = (
    abl.groupby(["subset", "variant"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        cz_mean=("cz_rmse", "mean"),
        cz_std=("cz_rmse", "std")
    )
    .reset_index()
)

print(abl_summary)

ATTENTION COMPARISON BY SUBSET
  subset            model  rmse_mean  rmse_std   cz_mean    cz_std   acc_mean  \
0  FD001   NanoSentry_CSG  14.852019  1.159595  4.556325  0.547890  88.333333   
1  FD001   NanoSentry_ECA  14.797121  0.256940  5.805158  1.670121  90.333333   
2  FD001  NanoSentry_None  17.960436  1.260341  6.708638  0.526061  86.666667   
3  FD001    NanoSentry_SE  15.102866  1.134676  4.464993  0.976756  88.000000   
4  FD003   NanoSentry_CSG  14.455626  0.459401  3.483290  0.890774  91.333333   
5  FD003   NanoSentry_ECA  12.896620  0.485249  3.856431  0.378818  92.333333   
6  FD003  NanoSentry_None  16.852677  0.791428  4.744216  0.327681  86.666667   
7  FD003    NanoSentry_SE  14.427813  1.480845  3.733586  0.410689  92.333333   

    acc_std  
0  1.154701  
1  2.886751  
2  2.516611  
3  1.000000  
4  2.081666  
5  2.081666  
6  1.527525  
7  1.527525  

ABLATION BY SUBSET
  subset      variant  rmse_mean  rmse_std   cz_mean    cz_std
0  FD001         full  14.0783

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch

seed = 42
OUT_DIR = "."
os.makedirs(OUT_DIR, exist_ok=True)

# Required objects must already exist in the notebook:
# load_cmapss, NanoSentry, CNNBaseline, LSTMBaseline, train_cmapss, DEVICE
tr, va, te = load_cmapss(1, seed)

def get_preds(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X, yr, _ in loader:
            pr, _ = model(X.to(DEVICE))
            preds.append(pr.detach().cpu().numpy().reshape(-1))
            targets.append(yr.detach().cpu().numpy().reshape(-1))
    return np.concatenate(preds), np.concatenate(targets)

def crit_rmse(pred, target):
    mask = target <= 30.0
    if not mask.any():
        raise RuntimeError("No FD001 test samples fall in the critical zone (RUL <= 30).")
    return np.sqrt(np.mean((pred[mask] - target[mask]) ** 2))

models = {
    "CNN-32": (CNNBaseline(14, 32, 3, False), False, "#69b96a", "o"),
    "LSTM-64": (LSTMBaseline(14, 64, 3, False), False, "#f28e2b", "s"),
    "NanoSentry": (NanoSentry(14, 24, 3, "CSG"), True, "#2764c7", "D"),
}

runs = {}
for name, (model, multi, _, _) in models.items():
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    train_cmapss(model, tr, va, te, multi=multi)
    runs[name] = get_preds(model, te)

records = []
for name, (pred, target) in runs.items():
    for sample_id, (p, t) in enumerate(zip(pred, target)):
        records.append({
            "seed": seed,
            "model": name,
            "sample_id": sample_id,
            "true_rul_cycles": float(t),
            "predicted_rul_cycles": float(p),
            "critical_zone": bool(t <= 30.0),
        })
prediction_csv = os.path.join(OUT_DIR, "F4_updated_critical_zone_predictions_seed42.csv")
pd.DataFrame(records).to_csv(prediction_csv, index=False)

metrics = []
for name, (pred, target) in runs.items():
    mask = target <= 30.0
    metrics.append({
        "model": name,
        "seed": seed,
        "critical_zone_n": int(mask.sum()),
        "critical_rmse_cycles": crit_rmse(pred, target),
        "critical_mean_bias_cycles": float(np.mean(pred[mask] - target[mask])),
    })
metrics_csv = os.path.join(OUT_DIR, "F4_updated_critical_zone_metrics_seed42.csv")
pd.DataFrame(metrics).to_csv(metrics_csv, index=False)
print(pd.DataFrame(metrics).to_string(index=False))

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white",
})

fig, ax = plt.subplots(figsize=(6.6, 5.4), constrained_layout=True)
ax.plot([0, 30], [0, 30], "--", color="#202020", lw=1.25, label="Perfect ($y=x$)", zorder=1)

for name in ["CNN-32", "LSTM-64", "NanoSentry"]:
    pred, target = runs[name]
    _, _, color, marker = models[name]
    mask = target <= 30.0
    score = crit_rmse(pred, target)
    ax.scatter(
        target[mask], pred[mask], s=24, c=color, marker=marker,
        alpha=0.72, linewidths=0.25, edgecolors="white",
        label=f"{name} (Crit. RMSE {score:.2f})", zorder=3,
    )

cnn_pred, cnn_target = runs["CNN-32"]
cnn_mask = cnn_target <= 30.0
cnn_bias = float(np.mean(cnn_pred[cnn_mask] - cnn_target[cnn_mask]))
cnn_ymax = max(35.0, float(max(runs[name][0][runs[name][1] <= 30].max() for name in runs) * 1.08))
annotation_y = min(cnn_ymax * 0.94, 0.90 * ax.get_ylim()[1])
ax.annotate(
    f"CNN-32 mean bias = {cnn_bias:+.1f} cycles",
    xy=(18.0, np.interp(18.0, np.sort(cnn_target[cnn_mask]), np.sort(cnn_pred[cnn_mask]))),
    xytext=(3.0, annotation_y),
    color="#2d7d35",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.28", fc="#eff8ef", ec="#69b96a", lw=0.9),
    arrowprops=dict(arrowstyle="->", color="#69b96a", lw=1.0),
    zorder=4,
)

ax.set_xlim(-0.5, 30.5)
ax.set_ylim(-1.0, cnn_ymax)
ax.set_xticks(np.arange(0, 31, 5))
ax.set_xlabel("True RUL (cycles)")
ax.set_ylabel("Predicted RUL (cycles)")
ax.grid(True, linestyle=":", linewidth=0.7, alpha=0.45)
ax.legend(loc="upper left", frameon=True, framealpha=0.96, edgecolor="#bdbdbd")

png_path = os.path.join(OUT_DIR, "F4_updated_critical_zone.png")
pdf_path = os.path.join(OUT_DIR, "F4_updated_critical_zone.pdf")
fig.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.03)
fig.savefig(pdf_path, format="pdf", bbox_inches="tight", pad_inches=0.03)
plt.show()
print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")
print(f"Saved: {prediction_csv}")
print(f"Saved: {metrics_csv}")